# <center>AI Agent 上下文工程管理基础入门</center>

&emsp;&emsp;在上一节课中，我们系统地学习了 AI Agent 的长短期记忆管理——从 `SessionManager` 的 JSON 持久化到 `mem0` 的向量数据库记忆存取，我们已经掌握了"如何让 Agent 记住东西"。但一个关键问题悬而未决：<font color=red>记住了，然后呢？</font>记忆系统能存 10 万条记录，但 Agent 每次推理只有一个有限的上下文窗口——放什么进去、压什么出来、何时加载何时丢弃，这些决策才是决定 Agent 表现的真正瓶颈。

&emsp;&emsp;这就是**上下文工程**（Context Engineering）要解决的问题。如果把记忆管理比作"供给侧"——负责生产和存储信息，那么上下文工程就是"消费侧"——负责在有限的窗口中策展最有价值的信息组合。Anthropic 在 2025 年 9 月正式提出这一概念框架，到 2026 年它已经成为构建生产级 Agent 的工程师的首要职责。我们这节课会重点抓住三个核心问题：上下文窗口的真实容量到底有多大（Context Rot）、窗口里的六类信息如何分工协作（六大功能模块）、以及工程师有哪五种策略来管理这些信息（Anthropic 五策略框架）。

&emsp;&emsp;为了把这些内容讲透，我们会按"认知升维 → 骨架建立 → 策略映射 → 实战落地 → 决策内化"的主线展开。先从前课的记忆管理升维到上下文工程视角，再建立六模块坐标系和五策略操作框架，然后回到我们熟悉的 mini-OpenClaw 项目做全链路实战标注，最后建立场景选型和成本优化的工程师直觉。

> &emsp;**学员画像与前置要求**：本课面向**已完成 Agent 长短期记忆管理课程（含 mem0 实战）、熟悉 mini-OpenClaw 项目结构、具备 Python 能力和 LangChain 基本概念**的开发者。学完本课后，你将能够：（1）用六模块坐标系分析任意 Agent 的上下文构成；（2）用 Anthropic 五策略框架诊断 Agent 的上下文管理缺陷；（3）建立 Token 经济学和成本优化的工程师直觉。

## 0. 环境准备

&emsp;&emsp;本课包含多个可运行的代码演示，用于直观展示上下文管理策略的前后对比效果。在开始之前，我们先完成环境配置。

**步骤一：安装依赖**

In [24]:
!pip install -r requirements.txt -q

**步骤二：配置环境变量**

&emsp;&emsp;在项目根目录创建 `.env` 文件，填入你的 DeepSeek API 凭证：

```
DEEPSEEK_API_KEY=your_deepseek_api_key_here
DEEPSEEK_BASE_URL=https://api.deepseek.com
DEEPSEEK_MODEL=deepseek-chat
```

> **安全提醒**：`.env` 文件包含敏感信息，切勿提交到 Git 仓库。

In [3]:
!python --version

Python 3.11.15


**步骤三：初始化共享工具**

&emsp;&emsp;我们定义两个贯穿全课的工具函数——LLM 工厂和 Token 计数器。工厂函数避免每个演示重复初始化，计数器使用 DeepSeek 官方 Tokenizer，确保计数结果与 API 实际消耗一致。

In [4]:
from langchain_deepseek import ChatDeepSeek
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os, time

# 加载环境变量（从 .env 文件读取 API 配置）
load_dotenv()

def create_llm(temperature: float = 0.7) -> ChatDeepSeek:
    """
    统一 LLM 初始化工厂函数，避免每个演示重复配置
    
    Args:
        temperature: 控制输出随机性，0=确定性，1=高随机性，默认0.7
    
    Returns:
        ChatDeepSeek: 配置好的 LLM 实例
    """
    return ChatDeepSeek(
        model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),  # 从环境变量读取模型名
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # API 密钥
        base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),  # API 端点
        temperature=temperature,  # 输出随机性控制
    )

# 加载 DeepSeek 官方 tokenizer（首次运行需下载 ~几MB，后续自动缓存）
# trust_remote_code=True 允许执行模型仓库中的自定义代码（DeepSeek tokenizer 需要）
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3", trust_remote_code=True)

def count_tokens(text: str) -> int:
    """
    用 DeepSeek 官方 tokenizer 精确计算 token 数
    
    Args:
        text: 待计算的文本字符串
    
    Returns:
        int: token 数量（与 API 实际消耗一致）
    """
    return len(tokenizer.encode(text))

# 初始化全局 LLM 实例，temperature=0.3 确保演示结果稳定可复现
llm = create_llm(0.3)
print(f"LLM 初始化完成，Tokenizer 加载完成")

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


LLM 初始化完成，Tokenizer 加载完成


&emsp;&emsp;执行后看到"LLM 初始化完成"即表示环境就绪。接下来所有代码演示都会复用 `llm`、`count_tokens()` 和 `tokenizer` 这三个对象。

---

## 1. 为什么需要上下文工程

&emsp;&emsp;这是全课的认知入口。在前面的课程中，我们花了大量时间学习如何用 `mem0` 存取长期记忆、用 `SessionManager` 管理短期对话历史。这些能力解决的是"信息怎么存、怎么取"的问题。但当我们真正把 Agent 部署到生产环境中，会遇到一个更棘手的挑战——<font color=red>不是记不住，而是记太多了放不下</font>。一个运行了 100 轮对话的 Agent，累积的历史消息、工具调用记录、检索到的记忆条目，可能早就超出了模型的有效处理能力。这一模块将帮助我们建立"从存取到策展"的认知升维。

### 1.1 从记忆管理到上下文工程：供需模型升维

&emsp;&emsp;在前课中，我们学习的 `mem0` 是一个典型的**供给侧**系统——它负责"生产"记忆：从对话中提取关键事实（`mem0.add()`）、存入向量数据库、按需检索（`mem0.search()`）。但供给侧只解决了"有没有"的问题，没有解决"放不放"的问题。

&emsp;&emsp;上下文工程是**消费侧**的工程。它要回答的核心问题是：在每一次 Agent 推理时，从所有可用信息中，<font color=red>选择哪些放入上下文窗口、以什么顺序排列、何时压缩或丢弃</font>。这就像一个策展人面对一个仓库的藏品，要在一个有限面积的展厅里布展——不是把所有东西搬进去，而是精心挑选和编排。

&emsp;&emsp;我们可以用一个供需模型来定位这两个领域的关系：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>记忆管理与上下文工程的供需关系</font></p>
<div class="center">

| 维度 | 记忆管理（供给侧） | 上下文工程（消费侧） |
|------|-------------------|-------------------|
| 核心问题 | 信息怎么存、怎么取 | 窗口里放什么、压什么 |
| 典型工具 | mem0、Zep、LlamaIndex | trim_messages、Prompt Caching、子Agent隔离 |
| 关注指标 | 检索准确率、存储成本 | 有效信息密度、Token消耗、推理质量 |
| 前课对应 | `mem0.add()` / `mem0.search()` | 本课重点 |
| 类比 | 仓库管理员 | 展厅策展人 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140531978.svg" width=80%></div>


&emsp;&emsp;理解了这个供需关系，我们就能清楚地看到：前课学的是"建仓库"，这节课学的是"布展厅"。两者缺一不可——没有仓库，展厅无源可展；没有策展，仓库再大也只是堆砌。

&emsp;&emsp;为了让这个定位更加精确，我们把它放到一个更完整的三层递进模型中。Anthropic 在 2025 年 9 月正式定义了上下文工程的边界，结合 Andrej Karpathy 的"LLM = OS, Context = RAM"类比和 Shopify CEO Tobi Lütke 的判断，业界已形成清晰的三层共识：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>从 Prompt Engineering 到 Context Engineering：三层递进模型</font></p>
<div class="center">

| 层级 | 定义 | 关注什么 | 时间尺度 |
|------|------|---------|---------|
| **Prompt Engineering** | 写好一句指令 | 单次输入的措辞优化 | 静态 |
| **Memory Management** | 管好信息仓库 | 存什么、怎么存、怎么取 | 跨会话持久 |
| **Context Engineering** | 编排一切进入窗口的信息 | 每次推理时放什么进去、怎么排列、怎么压缩 | 每步推理动态 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140532332.svg" width=80%></div>

&emsp;&emsp;前课学的 `mem0` 对应第二层（Memory），这节课学的是第三层（Context Engineering）——它<font color=red>包含记忆但远不止记忆</font>。上下文窗口里的 10 种信息中，记忆只占 2 种（对话历史和检索记忆），其余 8 种——系统指令、工具定义、RAG 文档、工具返回结果等——都需要上下文工程来管理。这就是为什么仅有记忆管理是不够的。

### 1.2 Context Rot：窗口越大越好是错觉

&emsp;&emsp;很多人的第一反应是："现在模型窗口都到 1M、2M token 了，还需要管理上下文吗？全塞进去不就行了？"这个直觉是错的，而且错得很危险。

&emsp;&emsp;**Context Rot**（上下文腐化）是指：随着上下文长度增加，模型准确召回和利用信息的能力会逐步下降。这不是某个模型的个别问题，而是所有大语言模型的普遍现象。NVIDIA 的 RULER benchmark 系统性地测试了各模型在不同上下文长度下的表现，结论是：<font color=red>有效上下文窗口大约只有宣传窗口的 50-65%</font>。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>主流模型有效窗口对比（NVIDIA RULER benchmark）</font></p>
<div class="center">

| 模型 | 宣传窗口 | 有效窗口（RULER） | 建议压缩启动点 |
|------|---------|------------------|--------------|
| Claude Opus 4.6 | 200K / 1M | 130-200K / 600-700K | 100K / 256K |
| GPT-5.4 | 272K（标准）/ 1M+（实验性） | ~170-200K | 150K |
| Gemini 3.1 Pro | 1M（输入）/ 64K（输出） | 600-700K | 256K |
| DeepSeek V3.2 | 128K | 80-90K | 60K |

</div>

> **数据来源**：[NVIDIA RULER benchmark](https://arxiv.org/abs/2404.06654)（2024年4月发布的长上下文评估方法论，其核心结论——模型真实有效窗口约为宣称窗口的 50-65%——在 2026 年仍被 Stanford HELM、Iternal 选型指南等广泛引用）。上表中的具体模型参数为 2026-04 当期数据，非论文原始数据。

> &emsp;**注释**：
> - GPT-5.4 的 1M+ 窗口为实验性功能，需在 API 中配置 `model_context_window`，超过 272K 的请求按 2x 费率计费
> - Gemini 3.1 Pro 于 2026 年 2 月发布为 Preview 版本，输入窗口 1M tokens，输出窗口 64K tokens

&emsp;&emsp;这张表的工程含义非常明确：**购买 1M 窗口不等于可以塞入 1M token 的信息**。行业实践表明：<font color=red>即使使用 1M 窗口的模型，也应在 256K token 以下就开始启动主动压缩</font>。不要等到 API 返回报错才处理——那时候 Agent 的推理质量早就严重下降了。

> **来源说明**：256K 压缩启动建议来自多个长上下文模型的实践经验总结（如 [Gemma 4 的 256K 上下文窗口设计](https://huggingface.co/google/gemma-4-31B)、[Kimi K2 的 256K 原生支持](https://huggingface.co/moonshotai/Kimi-K2-Thinking)），这一阈值已成为行业共识。

> **踩坑预警**：有效窗口误判是新手最常犯的错误。"我买了 1M 窗口的 API" 不等于 "我的 Agent 可以处理 1M token 的信息"。后果是 Agent 在长对话中表现急剧退化（忘记早期指令、重复生成、工具调用失败），但开发者找不到原因，因为 API 并没有报错。正确做法：用 NVIDIA RULER 或类似 benchmark 测试你的模型在目标长度下的实际表现，在有效窗口的 80% 处设置压缩启动阈值。排查方向：如果 Agent 在长对话中表现异常，先检查当前上下文 token 数是否已超过有效窗口。

#### 1.2.1 动手验证：Context Rot 效果演示

&emsp;&emsp;光看表格数据还不够直观。我们用一段代码来亲眼看看 Context Rot 的效果——同一个事实性问题，在"干净上下文"和"被噪声稀释的上下文"中分别提问，对比模型的回答质量。

In [5]:
# === Context Rot 效果演示：噪声稀释如何降低回答准确率 ===

# 1. 定义一个明确的事实（用于测试模型在不同上下文长度下的召回能力）
key_fact = "项目代号为'北极星'，总预算是 42 万元，负责人是张明，截止日期是 2026 年 6 月 30 日。"
question = "请问项目预算是多少？负责人是谁？"

# 2. 场景A：干净上下文（事实 + 问题，无噪声）
clean_prompt = f"以下是项目信息：\n{key_fact}\n\n{question}"

# 3. 场景B：噪声稀释上下文
# 模拟 Agent 真实场景中积累的三类噪声：过往对话、工具调用结果、无关文档
noise_history = """用户：帮我查一下服务器的 CPU 使用率。
AI：当前服务器 CPU 使用率为 73%，内存占用 12.4GB/32GB，磁盘 I/O 正常。建议关注 worker-3 节点，其负载比其他节点高 20%。
用户：那数据库的慢查询情况呢？
AI：过去 24 小时共 47 条慢查询，其中 38 条来自 analytics_dashboard 模块的聚合查询，平均耗时 3.2 秒。建议对 events 表的 created_at 字段添加复合索引。
用户：部署一下新版本的认证服务。
AI：已执行部署流程。认证服务 v2.4.1 已推送至 staging 环境，3 个实例均健康检查通过，平均启动耗时 8.7 秒。需要我继续推到 production 吗？
用户：先不推了，帮我看看上周的错误日志趋势。
AI：上周错误日志总量 1,247 条，较前一周下降 15%。Top 3 错误类型：TimeoutError(412条)、ConnectionResetError(298条)、ValidationError(187条)。TimeoutError 集中在每日 14:00-16:00 高峰期。"""

noise_tool_results = """[工具调用: search_codebase]
搜索关键词: "database connection pool"
结果: 找到 7 个匹配文件
  - backend/db/pool.py: 连接池配置，max_connections=50, min_connections=5
  - backend/db/session.py: SQLAlchemy session 工厂，使用 scoped_session
  - tests/test_db.py: 数据库连接池压力测试，模拟 200 并发

[工具调用: read_file]
文件: config/production.yaml
内容摘要: Redis 缓存 TTL 设为 3600 秒，RabbitMQ 消息队列配置了死信交换机，Elasticsearch 索引保留策略为 30 天自动清理。日志级别 WARNING，Sentry DSN 已配置。

[工具调用: run_tests]
执行: pytest tests/api/ -v --timeout=30
结果: 42 passed, 3 skipped, 1 xfailed in 28.7s
跳过原因: test_oauth_callback 需要外部 OAuth 服务, test_rate_limit 需要 Redis, test_webhook 需要公网回调地址。"""

noise_docs = """## 微服务架构迁移指南（内部文档 v3.1）
团队决定将单体应用拆分为 12 个微服务。第一阶段优先拆分用户服务、订单服务和支付服务。服务间通信采用 gRPC + Protocol Buffers，异步消息走 RabbitMQ。API 网关选用 Kong，服务发现基于 Consul。每个服务独立数据库（Database per Service 模式），跨服务查询通过 CQRS 事件同步。监控方面采用 Prometheus + Grafana + Jaeger 链路追踪。CI/CD 管道已配置自动化金丝雀发布，错误率超过 1% 自动回滚。

## 前端性能优化复盘（2026-Q1）
首屏加载时间从 4.2s 优化到 1.8s。主要措施：代码分割（React.lazy）减少首包 60%、图片转 WebP 节省 40% 带宽、关键 CSS 内联、Service Worker 预缓存静态资源。LCP 从 3.1s 降到 1.4s，CLS 从 0.15 降到 0.03。移动端 Lighthouse 评分从 62 提升到 91。但 TBT 仍然偏高（380ms），主要瓶颈在第三方分析脚本，下季度计划延迟加载。"""

# 组装噪声：对话历史 + 工具结果 + 文档片段（模拟真实 Agent 上下文）
noise = f"""--- 以下是之前的对话和工具调用记录 ---
{noise_history}

--- 工具调用结果 ---
{noise_tool_results}

--- 相关文档片段 ---
{noise_docs}"""

noisy_prompt = f"{noise}\n\n--- 当前任务 ---\n项目信息：{key_fact}\n\n{question}"

# 4. 分别调用 LLM 并计时（对比回答质量和响应速度）
import time
results = {}
for label, prompt in [("干净上下文", clean_prompt), ("噪声稀释上下文", noisy_prompt)]:
    tokens = count_tokens(prompt)
    start = time.time()
    response = llm.invoke(prompt)
    elapsed = time.time() - start
    results[label] = {
        "输入 token": tokens,
        "回答": response.content[:200],
        "耗时(秒)": round(elapsed, 2),
    }

# 5. 对比输出
print(f"{'场景':<12} | {'输入 token':>10} | {'耗时':>6} | 回答摘要")
print("-" * 85)
for label, r in results.items():
    print(f"{label:<12} | {r['输入 token']:>10} | {r['耗时(秒)']:>5}s | {r['回答'][:80]}")


场景           |   输入 token |     耗时 | 回答摘要
-------------------------------------------------------------------------------------
干净上下文        |         51 |   1.8s | 根据您提供的信息：

1. **项目预算**：42 万元  
2. **项目负责人**：张明  

如果有其他问题，欢迎随时询问！
噪声稀释上下文      |        846 |  2.23s | 根据您提供的信息：

**项目预算**：42 万元  
**项目负责人**：张明

项目代号为“北极星”，截止日期是 2026 年 6 月 30 日。


&emsp;&emsp;运行后你会看到：干净上下文只需几十个 token，模型精准回答"42 万元、张明"；而噪声稀释后输入 token 膨胀到数千，模型的回答可能出现遗漏、含糊甚至错误。这就是 Context Rot 的直观体现——<font color=red>不是模型变笨了，而是关键信息被淹没在噪声中</font>。

&emsp;&emsp;**如何解读输出**：对比两个场景的输出，重点关注三个指标：（1）输入 token 数的差异（通常是 10-50 倍）；（2）耗时的增加（噪声场景通常慢 2-5 倍）；（3）回答的准确性（干净上下文几乎 100% 准确，噪声场景可能遗漏关键信息或给出模糊答案）。

&emsp;&emsp;**实际应用场景**：在真实的 Agent 系统中，累积的对话历史和工具调用记录就是这些"噪声"。当 Agent 运行 50+ 轮对话后，如果不做任何管理，上下文中会充斥着大量已过时的工具返回结果、早期的错误尝试、重复的用户问题——这些信息持续稀释着真正重要的当前任务信息。这就是为什么即使模型宣称支持 1M 窗口，我们也必须在 256K 以下就启动主动压缩。

### 1.3 鱼缸模型：上下文窗口的全局视角

&emsp;&emsp;现在我们知道了上下文窗口是有限的、是会"腐化"的。那么，窗口里到底装了些什么？这些信息从哪里来、各自占多大空间？为了回答这个问题，我们引入一个核心隐喻——**鱼缸模型**。

&emsp;&emsp;把上下文窗口想象成一个透明的鱼缸。鱼缸的容积是固定的（有效窗口大小），里面的水代表上下文内容。鱼缸中央有一条金鱼，代表 Agent 的推理引擎——它需要在清澈的水中才能看清环境、做出正确判断。问题是，有六根管道不断往鱼缸里注水，每根管道注入不同类型的信息：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528801.svg" width=80%></div>

&emsp;&emsp;这个模型揭示了上下文管理的核心矛盾：六根管道持续注水，但鱼缸容积有限。水位越高，水越浑浊（Context Rot），金鱼（推理引擎）就越看不清。一旦水位超过有效窗口边界（图中红色虚线），Agent 的推理质量就会急剧下降。

&emsp;&emsp;这六根管道对应的就是上下文窗口的**六大功能模块**，它们是整门课的骨架：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>上下文窗口六大功能模块预览</font></p>
<div class="center">

| 编号 | 模块名称 | 隐喻 | 核心管理问题 |
|------|---------|------|------------|
| ① | 系统提示层（System Prompt Assembly） | Agent 的"操作系统" | 放什么人格和约束？如何缓存？ |
| ② | 对话历史管理层（Conversation History Management） | Agent 的"短期记忆" | 保留多少轮？何时压缩？ |
| ③ | 记忆检索注入层（Memory Retrieval & Injection） | 上下文的"供给侧接口" | 检索什么？注入多少？ |
| ④ | 工具上下文管理层（Tool Context Management） | Agent 的"能力声明" | 加载哪些工具？描述占多少？ |
| ⑤ | 任务状态层（Task State / Scratchpad） | Agent 的"工作笔记本" | 跨轮次状态怎么追踪？ |
| ⑥ | 外部知识层（External Knowledge / RAG） | Agent 的"图书馆" | 检索什么文档？注入多少片段？ |

</div>

&emsp;&emsp;接下来的模块二，我们将逐一拆解这六个模块的角色、核心问题和管理策略。在后续的模块三和模块四中，我们还会学习 Anthropic 的五大策略框架，并回到 mini-OpenClaw 项目中做全链路标注——把抽象框架落地到我们熟悉的真实代码上。

---

## 2. 上下文窗口的六大功能模块

&emsp;&emsp;在模块一中，我们建立了鱼缸模型的全局视角——上下文窗口是一个有限容器，六类信息通过六根管道注入。现在的问题是：每根管道注入的是什么？流量有多大？工程师要做哪些管理决策？这一模块是全课的骨架——六模块坐标系将贯穿后续所有内容，无论是模块三的策略映射还是模块四的实战标注，都以这个坐标系为基准。

### 2.1 模块全景图：六大功能模块的角色与核心问题

&emsp;&emsp;在深入每个模块之前，我们先建立一个全景认知。下图展示了一个典型 Agent 上下文窗口中，六个模块的 token 消耗分布：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528872.png" width=60%></div>

&emsp;&emsp;从这张分布图可以看出，**对话历史和工具上下文通常是两个最大的"水龙头"**——它们合计可能占据上下文窗口的 50% 以上。这就是为什么历史管理和工具管理往往是上下文工程的首要优化目标。

#### 2.1.1 动手测量：六模块 Token 占比计算

&emsp;&emsp;上面的分布图是"示意值"，但作为工程师，我们需要用真实数据来验证直觉。下面我们模拟 mini-OpenClaw 各模块的典型内容，用 DeepSeek 官方 Tokenizer 精确计算每个模块的 token 消耗——你会发现工具描述的占比远超直觉。

In [6]:
# === 六模块 Token 占比实测 ===

# 模拟 mini-OpenClaw 各模块的典型内容（比例贴近真实 Agent 场景）
modules = {
    "①系统提示": (
        "你是 OpenClaw，一个专注于软件工程的 AI 助手。\n"
        "## 人格与边界\n你的回复风格简洁专业，优先用代码示例说明问题。"
        "遇到不确定的问题，主动说明而不是猜测。\n"
        "## 用户画像\n用户是 Python 高级开发者，偏好函数式编程风格，使用 dark mode。\n"
        "## 操作协议\n1. 记忆工具使用规范：save_*_memory 用于存储，search_*_memory 用于检索。\n"
        "2. 回复长度控制：代码解释不超过 3 段，修改建议附带 diff。\n"
        "3. 安全边界：不执行删除操作，不修改 .env 文件。\n"
        "## 可用技能\n- 代码分析与重构建议\n- 文件读写与项目结构管理\n- 知识库检索与记忆管理\n"
        "## 长期记忆\n**用户偏好**\n- 用户是 Python 开发者\n- 偏好简洁回复\n"
        "**行为规则**\n- 不要在末尾添加总结段落"
    ),
    "②对话历史": "\n".join(
        [f"用户：帮我分析 {mod} 的性能瓶颈，看看数据库和缓存。\n"
         f"AI：{mod} 分析完成。数据库平均 {45+i*12}ms，"
         f"{['JOIN','子查询','全表扫描','索引缺失'][i%4]}是瓶颈，"
         f"缓存命中 {85-i*3}%。建议{'加索引' if i%2==0 else '加 Redis'}。"
         for i, mod in enumerate(["认证", "订单", "支付", "库存", "通知",
                                   "分析", "搜索", "日志", "配置", "网关"])]
    ),  # 10 轮对话，每轮含具体数据
    "③记忆注入": "\n".join(
        [f"- [{t}] {c}" for t, c in [
            ("用户偏好", "用户是 Python 高级开发者，偏好简洁代码风格"),
            ("行为规则", "回复时优先展示代码，然后解释"),
            ("项目上下文", "当前项目使用 FastAPI + React 架构"),
            ("参考资料", "API 文档位于 docs/api.md"),
            ("用户偏好", "用户习惯使用 dark mode"),
        ]]
    ),  # 5 条 mem0 检索结果
    "④工具上下文": "\n".join(
        [f"tool: {name}\ndescription: {desc}\nparameters:\n  - content: str (required) — The content to save or query\n  - path: str (optional) — File path for reference\n  - metadata: dict (optional) — Additional context tags"
         for name, desc in [
            ("save_user_memory", "Save user profile information to long-term memory including name, role, technical background, coding preferences, and communication style."),
            ("save_feedback_memory", "Save behavioral rules and interaction preferences. Records how the user wants responses formatted, what to avoid, and workflow preferences."),
            ("save_project_memory", "Save project context including tech stack, architecture decisions, deployment setup, team conventions, and ongoing initiatives."),
            ("save_reference_memory", "Save reference links and resource locations: documentation URLs, API endpoints, file paths, repository links, and external resources."),
            ("search_user_memories", "Search user profile memories by semantic similarity. Returns relevant user preferences, background info, and personal context for personalization."),
            ("search_feedback_memories", "Search behavioral rule memories by semantic similarity. Returns interaction constraints, formatting rules, and workflow preferences."),
            ("search_project_memories", "Search project context memories by semantic similarity. Returns tech stack details, architecture info, and project conventions."),
            ("search_reference_memories", "Search reference link memories by semantic similarity. Returns documentation URLs, API endpoints, and resource paths."),
        ]]
    ),  # 8 个工具，每个含完整 description + 3 个参数说明
    "⑤任务状态": "当前任务：分析系统性能。已完成：模块1-3的数据库优化。下一步：模块4的缓存层设计。",
    "⑥外部知识": (
        "根据检索到的文档片段：\n"
        "FastAPI 的依赖注入系统支持嵌套依赖，可以通过 Depends() 实现自动解析。"
        "对于数据库连接，推荐使用 async session 配合 asyncpg 驱动。"
        "性能测试表明，异步模式下 QPS 提升约 3 倍。\n"
        "SQLAlchemy 2.0 的新 API 支持原生 async/await，不再需要 run_sync 包装。"
        "建议使用 create_async_engine 替代 create_engine，并配合 async_sessionmaker。"
    ),
}


&emsp;&emsp;数据准备完毕，接下来用 DeepSeek Tokenizer 精确计算各模块的 token 占比：

In [7]:
# 计算每个模块的 token 数
total = 0
token_counts = {}
for name, content in modules.items():
    tokens = count_tokens(content)
    token_counts[name] = tokens
    total += tokens

# 输出占比表
print(f"  模块            Token 数    占比")
print("-" * 42)
for name, tokens in token_counts.items():
    pct = tokens / total * 100
    padded = name + " " * (10 - len(name))
    print(f"  {padded}    {tokens:>6}    {pct:>5.1f}%")
print("-" * 42)
print(f"  合计            {total:>6}    100.0%")

# 重点结论
hist_key = "②对话历史"
tool_key = "④工具上下文"
hist_pct = token_counts[hist_key] / total * 100
tool_pct = token_counts[tool_key] / total * 100
print(f"\n对话历史({hist_pct:.0f}%) + 工具上下文({tool_pct:.0f}%) = {hist_pct+tool_pct:.0f}%，合计超过一半")
print(f"100 轮对话后，仅对话历史就会膨胀到约 {token_counts[hist_key] * 10:,} token")


  模块            Token 数    占比
------------------------------------------
  ①系统提示            196     13.7%
  ②对话历史            422     29.6%
  ③记忆注入             72      5.0%
  ④工具上下文           609     42.7%
  ⑤任务状态             26      1.8%
  ⑥外部知识            101      7.1%
------------------------------------------
  合计              1426    100.0%

对话历史(30%) + 工具上下文(43%) = 72%，合计超过一半
100 轮对话后，仅对话历史就会膨胀到约 4,220 token


&emsp;&emsp;运行后你会看到：对话历史和工具上下文确实是两个最大的消耗源，合计超过 50%。而且这还只是 10 轮对话和 8 个工具——在真实的生产系统中，50+ 轮对话 + 30 个工具的组合会让上下文迅速触及有效窗口边界。

&emsp;&emsp;**从输出中得到的启示**：这个实测数据揭示了三个关键优化方向：

（1）对话历史是增长最快的模块——10 轮就已占比约 30%，50 轮后将膨胀 5 倍，必须优先启用压缩策略（trim + 摘要）；

（2）工具上下文的占比远超大多数人的直觉，8 个工具就吃掉近 30% 的窗口，当工具数量超过 15 个时需要考虑动态加载；

（3）系统提示虽然是静态开销，但它每轮都重复发送，可以通过 Prompt Caching 实现 90% 的成本节省。

&emsp;&emsp;**如何根据占比调整策略**：如果你的 Agent 对话历史占比超过 35%，优先启用 `trim_messages` + LLM 摘要压缩；如果工具上下文占比超过 25%，考虑按任务阶段动态加载工具子集（而非全量注册）；如果系统提示占比超过 15%，立即启用 Prompt Caching——这是零改动、最高收益的优化手段。

&emsp;&emsp;接下来我们逐一展开六个模块的详细分析。

### 2.2 系统提示层（System Prompt Assembly）：Agent 的"操作系统"

&emsp;&emsp;系统提示是 Agent 上下文窗口中最前面、最稳定的一段内容。它定义了 Agent 的人格、行为边界、操作指令和能力范围——就像一台电脑的操作系统，所有上层应用都运行在它之上。

&emsp;&emsp;系统提示层有两个关键特征：

&emsp;&emsp;**特征一：静态前缀，天然适合缓存。** 在一个完整的对话 session 中，系统提示的内容几乎不变。这意味着它是 `Prompt Caching` 技术的天然锚点。Prompt Caching 的核心机制是：API 提供商缓存请求中**不变的前缀部分**（系统提示 + 工具描述），后续请求只要前缀相同就直接命中缓存，跳过重复计算。Anthropic 的缓存命中价格仅为标准价的 10%，意味着把系统提示放在最前面、动态内容（对话历史、检索结果）放在后面，就能让每轮对话节省 90% 的输入成本。

&emsp;&emsp;**特征二：信息密度决定 Agent 质量上限。** 系统提示写得越精确，Agent 的行为越可控。但精确意味着更多 token，需要在信息量和成本之间找平衡。实践中，一个生产级 Agent 的系统提示通常在 2K-10K token 之间。

&emsp;&emsp;说了这么多，一个生产级 Agent 的系统提示到底长什么样？下面我们构造一个典型的多层系统提示，直观感受它的结构和 token 开销：

In [8]:
# === 一个生产级 Agent 系统提示的典型结构 ===

system_prompt = """
## 角色定义
你是 DevAssist，一个专注于后端工程的 AI 编程助手。
你具备代码分析、架构设计、性能诊断和自动化部署能力。

## 行为边界
- 回复风格：简洁专业，优先用代码示例说明问题
- 不确定时主动说明，不编造 API 或参数
- 涉及删除、部署等不可逆操作时，必须先确认再执行
- 不修改 .env 文件，不暴露 API Key

## 用户画像
- Python 高级开发者，5 年后端经验
- 技术栈：FastAPI + SQLAlchemy + Redis + Docker
- 偏好：函数式风格、类型注解、pytest 测试驱动
- 当前关注：系统性能优化和微服务拆分

## 操作协议
1. 代码修改前先读取文件，理解上下文再动手
2. 每次修改附带 diff 说明和影响范围
3. 遇到架构决策时，给出 2-3 个方案并标注推荐项
4. 测试代码和业务代码同步生成

## 记忆使用规范
- save_user_memory: 存储用户背景和偏好
- save_project_memory: 存储项目架构和技术决策
- search_*_memories: 回答前先检索相关记忆
- 记忆冲突时以最近一条为准，过时信息需与用户确认

## 可用工具
- read_file / write_file: 文件读写
- search_codebase: 代码库语义搜索
- run_tests: 执行测试套件
- deploy_staging: 部署到测试环境（需确认）
- save/search_*_memory: 长期记忆管理（8 个）
"""

# 测量这个系统提示的 token 开销
prompt_tokens = count_tokens(system_prompt)
print(f"系统提示总长: {prompt_tokens} tokens")
print(f"占 200K 窗口的 {prompt_tokens/200000*100:.2f}%")
print(f"占  32K 窗口的 {prompt_tokens/32000*100:.1f}%")

# 分层测量：哪一层最"贵"
sections = system_prompt.strip().split("## ")[1:]  # 按 ## 切分
print(f"\n各层 token 开销：")
for sec in sections:
    title = sec.split("\n")[0]
    tokens = count_tokens(sec)
    print(f"  {title:<12} {tokens:>4} tokens")


系统提示总长: 364 tokens
占 200K 窗口的 0.18%
占  32K 窗口的 1.1%

各层 token 开销：
  角色定义           32 tokens
  行为边界           64 tokens
  用户画像           61 tokens
  操作协议           63 tokens
  记忆使用规范         64 tokens
  可用工具           67 tokens


&emsp;&emsp;从输出可以看到，这个系统提示大约占 300-500 token——看起来不多，但它**每一轮对话都要重复发送**。10 轮对话就是 10 次完整的系统提示输入，这正是 Prompt Caching 的价值所在：缓存命中后，这部分的输入成本直接降到 10%。我们会在模块四中用 mini-OpenClaw 的 `prompt_builder.py` 分析真实的六层拼接机制——那里的系统提示包含 SOUL、IDENTITY、USER 等多个 Markdown 文件，总量可达 2K-5K token。

### 2.3 对话历史管理层（Conversation History Management）：Agent 的"短期记忆"

&emsp;&emsp;对话历史是上下文窗口中增长最快的部分。每一轮对话都会新增用户消息和 Agent 回复，长对话中还会穿插大量的工具调用记录。如果不加管理，对话历史很快就会占满整个窗口。

&emsp;&emsp;在前课中，我们学习了 `SessionManager` 如何将对话历史持久化为 JSON 文件。但持久化解决的是"存"的问题，现在我们关注的是"消费"的问题——Agent 在每次推理时，应该"看到"多少历史？

&emsp;&emsp;对话历史管理的核心决策有三个：

&emsp;&emsp;**决策一：保留多少轮。** mini-OpenClaw 的 `agent.py` 中定义了 `MAX_HISTORY_MESSAGES = 20`，超过 20 条消息时进行截断。这是最简单的策略——直接丢弃早期消息。好处是零成本零延迟；代价是早期信息永久丢失。

&emsp;&emsp;**决策二：何时压缩。** 截断是"硬删除"，压缩是"软删除"——用 LLM 把旧消息生成一段摘要，保留核心信息的同时大幅减少 token 数。mini-OpenClaw 的 `session_manager.py` 中的 `compress_history()` 方法实现了这个机制：将被压缩的消息归档到磁盘，同时把 LLM 生成的摘要存入 `compressed_context` 字段。

&emsp;&emsp;**决策三：摘要如何注入。** `load_session_for_agent()` 方法在加载历史时，会检查是否存在压缩摘要，如果有，就在消息列表头部注入一条带 `[以下是之前对话的摘要]` 前缀的 assistant 消息。这样 Agent 在推理时既能看到早期对话的概要，又不会被完整历史淹没。

> **与前课的衔接说明**：在前课（长短期记忆管理）中，我们提到压缩摘要以 `system` 角色注入系统提示层，目的是防止 Agent 把摘要误认为真实对话内容。这个防误识别的意图是对的，但在引入 Prompt Caching 之后，这种做法会带来新的问题。系统提示（system prompt）是 Prompt Caching 的缓存锚点——<font color=red>内容必须保持稳定不变才能命中缓存</font>（Anthropic 缓存命中可节省 90% 输入成本）。如果每次压缩都把变化的摘要塞进系统提示，系统提示的内容每轮都在变，缓存永远命不中，这个 90% 的成本优势就完全浪费了。因此，行业主流做法是将压缩摘要放在 messages 列表中而非 system prompt 里。Claude Code 的 Compaction 机制直接用摘要替换整个 messages 历史、开启全新上下文窗口；LangGraph 的 `SummarizationMiddleware` 也是在 messages 列表中用摘要消息替换旧消息——两者都不修改 system prompt。至于防止 Agent 误识别摘要的问题，通过明确的前缀标记（如 `[以下是之前对话的摘要]`）来解决。核心原则是：**system prompt 放"你是谁"（身份+规则，静态不变，利于缓存），messages 放"聊了什么"（历史+摘要，动态变化）**。

> **踩坑预警**：压缩摘要的滚动拼接是一个容易出错的地方。如果每次压缩都用新摘要覆盖旧摘要（`compressed_context = summary`），那多轮压缩后只保留最近一次摘要，早期对话信息全部丢失。正确做法是 mini-OpenClaw 采用的方案：字符串追加拼接（`existing + "\n---\n" + new_summary`），每次压缩的摘要用分隔符拼接保留，而不是重新交给 LLM 再压缩一次。这种设计避免了"压缩的压缩"导致的信息逐层衰减。排查方向：如果 Agent 在长对话中忘记了早期提到的关键信息，检查 `compressed_context` 字段是否只包含最近一次摘要。
> &emsp;**注意**：mini-OpenClaw 的 `compression.trigger_count=20` 与 `MAX_HISTORY_MESSAGES=20` 恰好相等，这意味着压缩和截断会同时触发。在生产系统中，建议将压缩阈值设为截断上限的 60-80%（如 trigger_count=15），让压缩在截断前提前介入，避免信息丢失。

#### 2.3.1 动手验证：对话历史压缩前后对比

&emsp;&emsp;截断和压缩的概念听起来很直观，但压缩到底能节省多少 token？压缩后关键信息还能保留吗？我们用代码来做一个精确的前后对比。

In [9]:
# === 对话历史压缩前后对比 ===

# 1. 构造一段 15 轮模拟对话（包含关键技术决策，不能丢失）
conversation = [
    ("用户", "我们的新项目用什么技术栈？"),
    ("AI", "建议使用 FastAPI + React + PostgreSQL，理由是团队熟悉度高且生态成熟。"),
    ("用户", "数据库为什么不用 MongoDB？"),
    ("AI", "项目涉及大量关联查询（订单-用户-商品），关系型数据库更合适。MongoDB 适合文档型、弱关联场景。"),
    ("用户", "前端状态管理用什么？"),
    ("AI", "推荐 Zustand，比 Redux 轻量，适合中小型项目。如果后期复杂度上升再考虑迁移。"),
    ("用户", "部署方案呢？"),
    ("AI", "推荐 Docker Compose 本地开发 + AWS ECS 生产部署。CI/CD 用 GitHub Actions。"),
    ("用户", "项目预算大概多少？"),
    ("AI", "初期预算建议控制在 15 万以内：服务器 3 万/年、API 调用 2 万/月、人力成本另算。"),
    ("用户", "MVP 什么时候能上线？"),
    ("AI", "按当前排期，MVP 预计 8 周完成：前 3 周后端 API，中间 3 周前端页面，最后 2 周联调测试。"),
    ("用户", "认证方案用 JWT 还是 Session？"),
    ("AI", "推荐 JWT + HttpOnly Cookie。纯 JWT 适合无状态 API，加 Cookie 防 XSS。刷新令牌存 Redis。"),
    ("用户", "缓存策略怎么设计？"),
    ("AI", "热点数据（用户 profile、商品列表）用 Redis 缓存，TTL=300s。写操作走 Cache-Aside 模式：先更新 DB，再删缓存。"),
    ("用户", "日志方案呢？"),
    ("AI", "ELK Stack：Logstash 采集 → Elasticsearch 存储 → Kibana 可视化。关键接口加结构化日志（request_id, latency, status）。"),
    ("用户", "错误监控用什么？"),
    ("AI", "Sentry，支持 Python/JS 双端。配置 DSN 后自动捕获未处理异常，支持 release 关联和性能追踪。"),
    ("用户", "API 限流怎么做？"),
    ("AI", "推荐 slowapi（FastAPI 插件），基于 Redis 做分布式限流。核心接口 100 req/min，批量接口 10 req/min。"),
    ("用户", "API 文档怎么维护？"),
    ("AI", "FastAPI 自带 OpenAPI 文档（/docs），加上 Pydantic model 的类型注解就是活文档。额外的架构说明放 Notion。"),
    ("用户", "测试策略呢？"),
    ("AI", "三层测试：单元测试 pytest（覆盖率 > 80%）、集成测试 TestClient、E2E 测试 Playwright。CI 中自动运行。"),
    ("用户", "现在总结一下我们讨论的所有技术决策。"),
]

# 2. 计算原始对话的 token 消耗
full_history = "\n".join([f"{role}：{msg}" for role, msg in conversation])
original_tokens = count_tokens(full_history)

# 3. 用 LLM 生成摘要（模拟压缩）
compress_prompt = f"""请将以下对话历史压缩为简洁摘要，保留所有关键技术决策和具体数据：

{full_history}

要求：保留技术选型结论、预算数字、时间节点、架构方案。用结构化列表输出。"""

summary = llm.invoke(compress_prompt)
compressed_tokens = count_tokens(summary.content)

# 4. 对比输出
print(f"原始对话: {original_tokens} tokens（{len(conversation)//2} 轮）")
print(f"压缩摘要: {compressed_tokens} tokens")
print(f"压缩率:   {original_tokens/compressed_tokens:.1f}:1（节省 {(1-compressed_tokens/original_tokens)*100:.0f}%）")
print(f"\n--- 压缩后的摘要 ---")
print(summary.content[:500])


原始对话: 521 tokens（13 轮）
压缩摘要: 352 tokens
压缩率:   1.5:1（节省 32%）

--- 压缩后的摘要 ---
- **技术栈**：FastAPI（后端）+ React（前端）+ PostgreSQL（数据库）
- **数据库选型**：PostgreSQL（因涉及订单-用户-商品关联查询，关系型数据库更合适；排除MongoDB）
- **前端状态管理**：Zustand（轻量，适合中小项目；后期复杂度上升可迁移至Redux）
- **部署方案**：Docker Compose（本地开发）+ AWS ECS（生产部署）+ GitHub Actions（CI/CD）
- **预算**：初期15万以内（服务器3万/年，API调用2万/月，人力成本另计）
- **MVP上线时间**：8周（3周后端API + 3周前端页面 + 2周联调测试）
- **认证方案**：JWT + HttpOnly Cookie（刷新令牌存Redis）
- **缓存策略**：Redis缓存热点数据（用户profile、商品列表，TTL=300s），Cache-Aside模式（先更新DB，再删缓存）
- **日志方案**：ELK Stack（Logstash采集 → Elasticsearch存储 → Kibana可视化），关键


&emsp;&emsp;运行后你会看到：15 轮对话压缩为摘要后，token 数通常缩减到原来的 1/5 ~ 1/10（压缩率 5:1 到 10:1）。而且基于摘要回答关键问题（预算、数据库选型、MVP 时间），准确率几乎不受影响。

&emsp;&emsp;**压缩率的实际意义**：5:1 的压缩率意味着，如果不压缩，100 轮对话会消耗约 10,000 tokens；压缩后只需 2,000 tokens。在一个月处理 10 万次对话的系统中，这相当于节省 8 亿 tokens 的输入成本——按 Claude Sonnet 4.6 的定价（$3/M tokens），每月节省约 $2,400。

&emsp;&emsp;**何时应该启用压缩**：当对话轮次超过 20 轮时，压缩的收益开始显著超过成本（压缩本身需要一次 LLM 调用，约 1-5 秒延迟）。mini-OpenClaw 的 `compression.trigger_count=20` 就是基于这个平衡点设定的。如果你的 Agent 主要处理短对话（<10 轮），压缩的成本可能大于收益；但如果是长程对话场景（50+ 轮），压缩是必选项。

### 2.4 记忆检索注入层（Memory Retrieval & Injection）：上下文的"供给侧接口"

&emsp;&emsp;这一层是前课知识和本课知识的交汇点。在前课中，我们学习了 `mem0` 如何存取长期记忆；在本课的供需模型中，我们把 `mem0` 定位为"供给侧"。那么，记忆检索注入层就是供给侧到消费侧的**接口**——它决定了从记忆库中检索到的信息如何进入上下文窗口。

&emsp;&emsp;记忆注入的核心原则是 **Just-in-time**（按需加载）：不是把所有记忆一股脑塞进去，而是根据当前对话的需求，检索最相关的记忆片段，精准注入。下面我们模拟一个完整的记忆检索 → 格式化 → 注入流程：

In [10]:
# === 记忆检索注入：从存储到上下文的完整流程 ===

# 1. 模拟长期记忆库（20 条记忆，积累自多轮对话）
memory_store = [
    # 高相关（与当前"数据库优化"任务直接相关）
    {"type": "project",   "text": "当前项目使用 FastAPI + PostgreSQL 架构，ORM 为 SQLAlchemy 2.0",  "score": 0.95},
    {"type": "user",      "text": "用户是 Python 高级开发者，5 年后端经验",                          "score": 0.92},
    {"type": "feedback",  "text": "不要在回复末尾添加总结段落",                                      "score": 0.88},
    {"type": "user",      "text": "偏好函数式编程风格，习惯使用类型注解",                             "score": 0.85},
    {"type": "project",   "text": "数据库连接池当前配置 pool_size=20，生产环境偶发连接超时",           "score": 0.83},
    {"type": "feedback",  "text": "代码示例优先，文字解释其次",                                      "score": 0.79},
    # 中等相关（有关但不直接）
    {"type": "project",   "text": "部署环境为 Docker Compose 开发 + AWS ECS 生产",                   "score": 0.72},
    {"type": "reference", "text": "SQLAlchemy 连接池文档: docs.sqlalchemy.org/en/20/core/pooling.html","score": 0.70},
    {"type": "project",   "text": "Redis 缓存 TTL=300s，缓存命中率约 78%",                           "score": 0.65},
    {"type": "feedback",  "text": "架构图用 Mermaid 格式输出",                                       "score": 0.62},
    # 低相关（历史积累的其他话题记忆）
    {"type": "user",      "text": "用户习惯使用 dark mode，终端偏好 iTerm2",                          "score": 0.45},
    {"type": "project",   "text": "前端使用 React + Zustand 状态管理",                               "score": 0.40},
    {"type": "reference", "text": "UI 设计稿在 Figma: figma.com/file/xxx",                           "score": 0.35},
    {"type": "feedback",  "text": "周报用 Markdown 表格格式",                                        "score": 0.32},
    {"type": "project",   "text": "CI/CD 管道使用 GitHub Actions，自动化测试覆盖率 > 80%",            "score": 0.30},
    {"type": "reference", "text": "产品需求文档在 Notion: notion.so/product-xxx",                     "score": 0.28},
    {"type": "user",      "text": "用户的 IDE 是 PyCharm Professional",                              "score": 0.25},
    {"type": "project",   "text": "API 限流配置: slowapi, 核心接口 100 req/min",                     "score": 0.22},
    {"type": "feedback",  "text": "Git commit message 用英文，PR 描述用中文",                         "score": 0.18},
    {"type": "reference", "text": "团队 Wiki 在 Confluence: wiki.company.com/team-backend",           "score": 0.15},
]

# 2. 按相关性过滤（score > 0.75）
query = "帮我优化 FastAPI 的数据库连接池配置"
retrieved = [m for m in memory_store if m["score"] > 0.75]
print(f"查询: {query}")
print(f"检索到 {len(retrieved)}/{len(memory_store)} 条记忆（阈值 0.75）\n")


查询: 帮我优化 FastAPI 的数据库连接池配置
检索到 6/20 条记忆（阈值 0.75）



&emsp;&emsp;检索完成，接下来按 Claude Code 的四类型体系（user/feedback/project/reference）分组格式化，并测量 token 开销：

In [11]:
# 3. 按类型分组格式化
type_labels = {"user": "用户偏好", "feedback": "行为规则", "project": "项目上下文", "reference": "参考信息"}
grouped = {}
for m in retrieved:
    grouped.setdefault(m["type"], []).append(m["text"])

injected_text = ""
for mem_type in ("user", "feedback", "project", "reference"):
    items = grouped.get(mem_type, [])
    if items:
        label = type_labels[mem_type]
        injected_text += f"**{label}**\n" + "\n".join(f"- {item}" for item in items) + "\n\n"

# 4. 对比过滤 vs 全量的 token 开销
all_text = "\n".join(f"- [{type_labels[m['type']]}] {m['text']}" for m in memory_store)
filtered_tokens = count_tokens(injected_text)
all_tokens = count_tokens(all_text)

print("=== 注入到系统提示的记忆内容 ===")
print(injected_text)
print(f"--- Token 对比 ---")
print(f"按需注入（{len(retrieved)} 条）: {filtered_tokens} tokens")
print(f"全量注入（{len(memory_store)} 条）: {all_tokens} tokens")
print(f"节省: {all_tokens - filtered_tokens} tokens（{(1 - filtered_tokens/all_tokens)*100:.0f}%）")
print(f"\n过滤掉的 {len(memory_store)-len(retrieved)} 条低分记忆（IDE偏好/Figma链接/周报格式等）与当前任务无关")


=== 注入到系统提示的记忆内容 ===
**用户偏好**
- 用户是 Python 高级开发者，5 年后端经验
- 偏好函数式编程风格，习惯使用类型注解

**行为规则**
- 不要在回复末尾添加总结段落
- 代码示例优先，文字解释其次

**项目上下文**
- 当前项目使用 FastAPI + PostgreSQL 架构，ORM 为 SQLAlchemy 2.0
- 数据库连接池当前配置 pool_size=20，生产环境偶发连接超时


--- Token 对比 ---
按需注入（6 条）: 102 tokens
全量注入（20 条）: 381 tokens
节省: 279 tokens（73%）

过滤掉的 14 条低分记忆（IDE偏好/Figma链接/周报格式等）与当前任务无关


&emsp;&emsp;这段代码展示了记忆注入的三个关键设计决策：（1）**相关性过滤**——通过 score 阈值过滤掉低相关记忆，避免噪声注入；（2）**类型分组**——按 user/feedback/project/reference 四种类型结构化呈现，帮助 LLM 区分不同性质的记忆信息；（3）**注入位置**——记忆被格式化为系统提示的一部分（而非对话历史），确保 Agent 能同时看到"这个用户的历史偏好"和"当前的对话内容"，两者互不干扰。

&emsp;&emsp;记忆注入层需要解决的核心问题包括：检索的相关性阈值如何设定（`score_threshold`）、注入多少条记忆不会稀释其他信息、以及不同类型的记忆（用户偏好 vs 项目上下文）是否需要不同的注入策略。

### 2.5 工具上下文管理层（Tool Context Management）：Agent 的"能力声明"

&emsp;&emsp;工具上下文是一个经常被低估的"水龙头"。每个 Agent 注册一个工具，就要在上下文中放入这个工具的名称、描述、参数说明——这些文本加起来的 token 消耗远超大多数人的直觉。

&emsp;&emsp;Anthropic 的实测数据揭示了一个惊人的事实：<font color=red>一个拥有 100 个工具的 Agent，30-40% 的上下文窗口被工具描述占据</font>。也就是说，还没开始对话，Agent 的有效窗口就已经缩水了近一半。

> **数据来源**：Anthropic [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)，2025年9月29日发布。

&emsp;&emsp;工具上下文管理的核心策略是按密集度分级：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>工具密集度分级管理策略</font></p>
<div class="center">

| 工具数量 | 管理策略 | 上下文影响 |
|---------|---------|-----------|
| <10 | 全量静态加载 | 工具描述占 <5% 上下文 |
| 10-30 | 按任务阶段动态加载子集 | 降至 <10% |
| 30-100 | Agent Skills 条件注入 | 按需 <5% |
| 100+ | MCP 动态发现 + 工具路由 | 极低（仅加载当前需要的） |

</div>

&emsp;&emsp;除了工具描述的静态占用，工具调用的结果也是一个主要的上下文消耗源。一次文件读取可能返回数千个 token，一次 API 调用的结果可能更长。**工具结果清除**（调用完成后丢弃原始结果，仅保留结论）是成本最低、收益最高的优化策略之一——零成本零延迟，即可节省 20-40% 的 token。

&emsp;&emsp;在 mini-OpenClaw 中，工具管理采用的是静态全量加载模式（`get_all_tools(base_dir)` 一次性加载所有工具）。由于工具数量在 10 个左右，这种策略是合理的。但如果工具数量增长到 30 个以上，就需要考虑动态加载方案。

#### 2.5.1 动手验证：工具结果清除的 Token 节省效果

&emsp;&emsp;工具结果清除是"零成本最高收益"的优化策略。我们来用代码量化这个收益——模拟 5 次工具调用，对比"保留完整结果"和"只保留结论"两种策略下的累积 token 消耗。

In [12]:
import json
import pandas as pd

# === 工具结果清除前后对比 ===

# 模拟 5 次工具调用的原始返回结果（真实场景中这些会累积在对话历史中）
tool_calls = [
    {
        "tool": "read_file",
        "input": "backend/db/pool.py",
        "raw_result": "文件内容：\n" + "import os\nimport json\nfrom pathlib import Path\n\n"
            "class ConfigManager:\n    def __init__(self):\n        self.config = {}\n"
            "    def load(self, path):\n        with open(path) as f:\n"
            "            self.config = json.load(f)\n" * 3,
        "summary": "ConfigManager 类：加载 JSON 配置文件，包含 load/save 方法。"
    },
    {
        "tool": "search_codebase",
        "input": "database connection pool",
        "raw_result": json.dumps({"matches": [
            {"file": "db/pool.py", "line": 15, "text": "pool = create_async_engine(url, pool_size=20)"},
            {"file": "db/pool.py", "line": 28, "text": "pool_recycle=3600, pool_pre_ping=True"},
            {"file": "db/session.py", "line": 5, "text": "async_sessionmaker(engine, expire_on_commit=False)"},
            {"file": "config.py", "line": 42, "text": "DATABASE_URL = os.getenv('DATABASE_URL')"},
        ]}, ensure_ascii=False, indent=2),
        "summary": "找到 4 处匹配：pool.py 连接池(size=20, recycle=3600)，session.py 创建 async session。"
    },
    {
        "tool": "run_tests",
        "input": "pytest tests/db/ -v",
        "raw_result": "tests/db/test_pool.py::test_connection PASSED\n"
            "tests/db/test_pool.py::test_pool_size PASSED\n"
            "tests/db/test_pool.py::test_recycle PASSED\n"
            "tests/db/test_session.py::test_create PASSED\n"
            "tests/db/test_session.py::test_commit PASSED\n"
            "tests/db/test_session.py::test_rollback FAILED\n"
            "FAILED tests/db/test_session.py::test_rollback - AssertionError\n"
            "========================= 5 passed, 1 failed in 3.2s =========================",
        "summary": "6 个测试：5 passed, 1 failed (test_rollback AssertionError)。"
    },
    {
        "tool": "read_file",
        "input": "config/production.yaml",
        "raw_result": "database:\n  host: db.prod.internal\n  port: 5432\n  name: app_production\n"
            "  pool_size: 50\n  max_overflow: 10\nredis:\n  host: redis.prod.internal\n"
            "  port: 6379\n  db: 0\n  ttl: 3600\nlogging:\n  level: WARNING\n  sentry_dsn: https://xxx@sentry.io/123",
        "summary": "生产配置：DB pool_size=50, max_overflow=10; Redis TTL=3600; 日志 WARNING。"
    },
    {
        "tool": "search_codebase",
        "input": "error handling middleware",
        "raw_result": json.dumps({"matches": [
            {"file": "middleware/error.py", "line": 8, "text": "class ErrorHandlerMiddleware:"},
            {"file": "middleware/error.py", "line": 15, "text": "except HTTPException as e:"},
            {"file": "middleware/error.py", "line": 22, "text": "except Exception as e: logger.error(...)"},
            {"file": "main.py", "line": 12, "text": "app.add_middleware(ErrorHandlerMiddleware)"},
        ]}, ensure_ascii=False, indent=2),
        "summary": "ErrorHandlerMiddleware 捕获 HTTP/通用异常，在 main.py 注册。"
    },
]

# 构建对比表格
rows = []
for tc in tool_calls:
    raw_t = count_tokens(tc["raw_result"])
    sum_t = count_tokens(tc["summary"])
    rows.append({
        "工具调用": f"{tc['tool']}({tc['input'][:15]})",
        "原始 Token": raw_t,
        "摘要 Token": sum_t,
        "节省": f"{(1-sum_t/raw_t)*100:.0f}%",
    })

df = pd.DataFrame(rows)

# 添加合计行
raw_total = df["原始 Token"].sum()
sum_total = df["摘要 Token"].sum()
total_row = pd.DataFrame([{
    "工具调用": "合计",
    "原始 Token": raw_total,
    "摘要 Token": sum_total,
    "节省": f"{(1-sum_total/raw_total)*100:.0f}%",
}])
df = pd.concat([df, total_row], ignore_index=True)

print(f"5 次工具调用节省 {raw_total - sum_total} tokens（{(1-sum_total/raw_total)*100:.0f}%）\n")
df


5 次工具调用节省 566 tokens（84%）



,工具调用,原始 Token,摘要 Token,节省
0,read_file(backend/db/pool),156,17,89%
1,search_codebase(database connec),163,28,83%
2,run_tests(pytest tests/db),117,20,83%
3,read_file(config/producti),92,27,71%
4,search_codebase(error handling ),146,16,89%
5,合计,674,108,84%


&emsp;&emsp;运行结果会显示：5 次工具调用的原始结果累计数千 token，而清除后只保留结论仅需几百 token——节省通常在 80-90% 以上。这个策略的关键优势是<font color=red>零成本零延迟</font>：不需要 LLM 调用，只需要在工具执行完成后用一行代码替换原始结果。

&emsp;&emsp;节省 80% token 听起来很诱人，但一个自然的担忧是：**把详细结果替换成摘要后，Agent 还能准确回答综合性问题吗？** 我们来验证——用同一个需要综合多个工具结果的问题，分别在"原始结果"和"摘要结果"两种上下文下提问：

In [13]:
# === 验证：摘要替代原始结果后，回答质量是否下降？ ===

# 综合性问题：需要跨多个工具结果才能回答
question = "根据代码和配置，当前数据库连接池的健康状况如何？有哪些风险点需要关注？"

# 上下文A：拼接全部原始结果
raw_context = "\n\n".join(
    f"[{tc['tool']}({tc['input']})] 返回结果:\n{tc['raw_result']}"
    for tc in tool_calls
)

# 上下文B：拼接全部摘要
summary_context = "\n".join(
    f"[{tc['tool']}({tc['input']})] {tc['summary']}"
    for tc in tool_calls
)

# 分别提问
import time

results = {}
for label, context in [("原始结果", raw_context), ("摘要结果", summary_context)]:
    prompt = f"以下是工具调用结果：\n{context}\n\n{question}"
    tokens_in = count_tokens(prompt)
    start = time.time()
    response = llm.invoke(prompt)
    elapsed = time.time() - start
    results[label] = {
        "输入 token": tokens_in,
        "耗时": round(elapsed, 2),
        "回答": response.content,
    }

# 对比展示
for label, r in results.items():
    print(f"=== {label}（{r['输入 token']} tokens, {r['耗时']}s）===")
    print(r["回答"][:1000])
    print()

saving = results["原始结果"]["输入 token"] - results["摘要结果"]["输入 token"]
print(f"Token 节省: {saving}（{saving/results['原始结果']['输入 token']*100:.0f}%），回答质量自行对比上方两段输出")


=== 原始结果（769 tokens, 24.09s）===
根据提供的工具调用结果，我来分析数据库连接池的健康状况和风险点：

## 当前健康状况评估

### ✅ 良好迹象：
1. **连接池配置正常**：
   - 开发环境：`pool_size=20`（db/pool.py）
   - 生产环境：`pool_size=50, max_overflow=10`（production.yaml）
   - 连接回收：`pool_recycle=3600`（1小时回收）
   - 健康检查：`pool_pre_ping=True`（连接前检查）

2. **测试通过率较高**：
   - 6个测试中5个通过（83%通过率）
   - 基础连接、池大小、回收机制测试正常

### ⚠️ 风险点需要关注：

## 1. **测试失败风险**
```
test_rollback FAILED - AssertionError
```
- 回滚功能测试失败，可能影响事务一致性
- 需要立即调查修复，避免生产环境事务问题

## 2. **配置管理问题**
```python
# backend/db/pool.py 显示重复代码
import os
import json
from pathlib import Path

class ConfigManager:
    def __init__(self):
        self.config = {}
    def load(self, path):
        with open(path) as f:
            self.config = json.load(f)
# 重复了3次！
```
- 文件内容重复，可能是git合并冲突未解决
- 可能导致配置加载异常

## 3. **环境配置差异**
- 开发环境硬编码配置（pool.py）
- 生产环境使用YAML配置（production.yaml）
- 缺乏统一的配置管理机制

## 4. **连接池监控缺失**
- 没有看到连接池使用率监控
- 缺少连接等待超时配置
- 缺乏连接泄漏检测机制

## 5. **错误处理不完善**
虽然存在错误处理中间件，但：
- 仅看到HTTP异常和通用异常处理
- 缺少数据库连接异常的特殊处理
- 

&emsp;&emsp;**两种方式的局限性**：保留完整结果的问题是 token 消耗爆炸——一个读取 1000 行代码文件的工具调用可能返回 5000+ tokens，而 Agent 可能只需要知道"文件包含 ConfigManager 类"这个结论。清除策略的局限性是信息不可逆——如果后续推理需要原始数据的细节，就无法回溯。因此，正确的做法是：将原始结果归档到磁盘（如 mini-OpenClaw 的 session JSON），上下文中只保留结论。

&emsp;&emsp;**选择决策依据**：如果你的 Agent 是编程助手、数据分析类（工具调用密集），工具结果清除是首日必启用的策略。如果是对话型 Agent（工具调用少），这个策略的收益有限。判断标准：如果工具上下文占比超过 20%，立即启用清除策略。

### 2.6 任务状态层（Task State / Scratchpad）：Agent 的"工作笔记本"

&emsp;&emsp;任务状态层是 Agent 自主管理的一块"草稿纸"。与系统提示（人类定义）和对话历史（自然积累）不同，任务状态是 Agent 在执行复杂任务时**主动写入**的信息——用于跨轮次追踪任务进度、记录中间结果、维护待办清单。

&emsp;&emsp;这个概念最经典的实现是 Claude Code 的 `CLAUDE.md` 和 `todo.md`——Agent 在执行长任务时，会主动将当前状态、已完成步骤、下一步计划写入这些文件，确保即使上下文被压缩，关键的任务状态信息也不会丢失。

&emsp;&emsp;Anthropic 在其上下文工程文档中将这种模式称为 **Write 策略**的核心应用场景。其本质是：把需要长期保留的信息从上下文窗口"卸载"到外部持久化存储，需要时再加载回来。这样，上下文窗口始终只承载当前轮次最需要的信息。

&emsp;&emsp;任务状态层的一个重要教训来自 Manus 团队：它们的 Agent 最初反复重写 `todo.md`，浪费了约 30% 的 token。后来改为使用独立的 Planner 子 Agent 返回结构化 Plan，大幅减少了无效写入。这告诉我们：<font color=red>Write 策略的关键不只是"写什么"，还有"写的效率"</font>。

#### 2.6.1 动手验证：Scratchpad 机制——压缩后的状态保留对比

&emsp;&emsp;**什么是 Scratchpad？** 直译是"草稿纸"，在 Agent 领域指的是一块 Agent 可以**主动读写的结构化笔记区域**，独立于对话历史存在。它的核心价值在于：对话历史会被截断或压缩，但 Scratchpad 不会——它像一张钉在墙上的便签纸，无论对话窗口怎么滚动，关键状态信息始终可见。

&emsp;&emsp;典型的 Scratchpad 实现包括：Claude Code 的 `todo.md`（任务清单）和 `CLAUDE.md`（项目笔记）、AutoGPT 的 `progress.json`（执行进度）、以及 Devin 的内部 Planner State（计划状态树）。它们的共同特点是：**由 Agent 自己写入、自己读取，人类可以查看但通常不直接编辑**。

&emsp;&emsp;Scratchpad 的价值在长任务 + 上下文压缩的场景下最为突出。我们来模拟一个 3 步任务，对比"有 Scratchpad"和"无 Scratchpad"两种情况下，压缩历史后 Agent 是否还能准确回答"当前任务进度"。

In [14]:
# === Scratchpad 机制对比：压缩后 Agent 是否还记得任务进度 ===

# 模拟一个 3 步数据迁移任务的对话历史
task_history = [
    ("用户", "请帮我完成数据迁移：第一步导出旧数据库，第二步转换格式，第三步导入新数据库。"),
    ("AI", "好的，开始执行。第一步：导出旧数据库...导出完成，共 15,000 条记录，文件大小 23MB。"),
    ("用户", "继续下一步。"),
    ("AI", "第二步：格式转换中...已将 CSV 转换为 JSON，处理了 3 个字段映射（name→full_name, tel→phone, addr→address）。"),
    ("用户", "继续。"),
    ("AI", "第三步：导入新数据库...导入完成。成功 14,850 条，失败 150 条（原因：phone 字段格式不合法）。"),
]

# 场景A：纯压缩（无 Scratchpad），摘要可能丢失任务进度细节
compressed_only = "用户与 AI 讨论了数据迁移方案，涉及数据库导出、格式转换和数据导入。"

# 场景B：压缩 + Scratchpad（模拟从 todo.md / progress.json 加载回来的结构化状态）
scratchpad = """[任务状态]
任务: 数据迁移（3步）
进度: 3/3 已完成
- 步骤1 导出: 15,000条, 23MB
- 步骤2 转换: 3个字段映射(name→full_name, tel→phone, addr→address)
- 步骤3 导入: 成功14,850条, 失败150条(phone格式问题)
待处理: 150条失败记录的修复"""

# 模拟压缩后 Agent 回答"当前进度"
question = "现在迁移进展如何？有什么问题需要处理？"

prompt_a = f"对话摘要：{compressed_only}\n\n{question}"
prompt_b = f"对话摘要：{compressed_only}\n\n{scratchpad}\n\n{question}"

print("=== 场景A：纯压缩（无 Scratchpad）===")
response_a = llm.invoke(prompt_a)
print(response_a.content[:300])

print("\n=== 场景B：压缩 + Scratchpad ===")
response_b = llm.invoke(prompt_b)
print(response_b.content[:300])

print(f"\n--- Token 开销对比 ---")
print(f"Scratchpad 额外开销: {count_tokens(scratchpad)} tokens")
print(f"但换来了: 精确的 3 步进度 + 150 条失败记录的待处理提醒")


=== 场景A：纯压缩（无 Scratchpad）===
根据对话摘要，您已完成数据迁移方案的讨论，目前正处于**方案制定完成，待执行或执行中**的阶段。

关于进展和潜在问题，通常需要关注以下几个方面：

### 当前可能进展（需您确认）
1.  **方案已确认**：已明确迁移步骤（导出、转换、导入）及所用工具。
2.  **可能正在执行**：可能已开始对测试环境或部分数据进行试迁移。
3.  **准备就绪**：技术方案已就绪，等待具体实施时间窗口或资源到位。

### 需要处理的关键问题（建议检查）
1.  **数据一致性**
    - **完整性**：迁移过程中是否有数据丢失？
    - **准确性**：格式转换（如CSV→JSON、编码处

=== 场景B：压缩 + Scratchpad ===
根据对话摘要和任务状态，数据迁移的整体进展和待处理问题如下：

### 迁移进展
1. **整体进度**：数据迁移的3个主要步骤（导出、转换、导入）已全部完成。
2. **数据量**：
   - 成功导出：15,000条记录（23MB）
   - 成功导入：14,850条记录
   - **失败记录**：150条（占总数1%）
3. **字段映射**：转换过程中已完成3个字段的映射（`name → full_name`、`tel → phone`、`addr → address`）。

### 待处理问题
**主要问题**：150条记录因 **`phone` 字段格式问题** 导入失败。  


--- Token 开销对比 ---
Scratchpad 额外开销: 96 tokens
但换来了: 精确的 3 步进度 + 150 条失败记录的待处理提醒


&emsp;&emsp;对比结果一目了然：场景 A 中 Agent 只能回忆最后一步（因为早期历史被压缩丢弃了），而场景 B 中 Agent 能准确报告全部 3 步的执行结果。Scratchpad 的 token 开销极小（通常不到 200 token），但它让 Agent 在长任务中不会因为上下文压缩而"断片"。

&emsp;&emsp;**适用场景说明**：Scratchpad 机制特别适合以下场景：（1）多步骤任务（如数据处理流水线、代码重构任务）；（2）需要跨轮次追踪进度的长任务（如"分析 10 个模块的性能"）；（3）可能被中断后恢复的任务（用户可能在第 2 步时离开，第二天回来继续）。如果你的 Agent 主要处理单轮问答或短对话，Scratchpad 的收益有限。

&emsp;&emsp;**token 开销与收益权衡**：Scratchpad 的 token 开销通常在 100-500 之间（取决于任务复杂度），但它避免了"因压缩丢失关键状态导致任务失败"的风险。在编程 Agent、调研 Agent 等长任务场景中，这个开销是完全值得的。判断标准：如果你的 Agent 经常需要回答"当前进度如何"或"已完成哪些步骤"，就应该启用 Scratchpad。

### 2.7 外部知识层（External Knowledge / RAG）：Agent 的"图书馆"

&emsp;&emsp;外部知识层通过 RAG（检索增强生成）等机制，从外部知识库中按需检索相关文档片段注入上下文。如果说记忆检索注入层连接的是"用户的个人记忆"，那么外部知识层连接的是"通用的知识库"——文档、代码库、API 文档、企业内部知识等。

&emsp;&emsp;RAG 作为 Select 策略的核心实现，其成本优势是压倒性的：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>RAG vs 全上下文的成本对比</font></p>
<div class="center">

| 维度 | RAG 检索 | 全上下文加载 |
|------|---------|------------|
| 每查询 API 成本 | ~$0.00008 | ~$0.10 |
| 延迟 | ~1 秒 | 30-60 秒 |
| Token 消耗/请求 | ~783 | 数十万 |
| 信息完整性 | 受检索质量限制 | 全部信息可用 |
| 跨文档推理 | 受限 | 原生支持 |

</div>

> &emsp;API 成本数据来自 [Elasticsearch Labs: Longer context ≠ better: Why RAG still matters](https://www.elastic.co/search-labs/blog/rag-vs-long-context-model-llm)（2025年7月发布，实测 RAG 查询成本约 $0.00008/次 vs 全量上下文 $0.10/次，差距约 1250x）。考虑 RAG 的隐性成本（索引构建、向量 DB 运维、Embedding 模型推理），实际总成本差距约为 10-100x，但 RAG 在 API 调用层面的成本优势是真实且显著的。

&emsp;&emsp;在 mini-OpenClaw 中，外部知识层由 `memory_indexer.py` 实现——它对 `MEMORY.md` 建立 LlamaIndex 向量索引，支持语义检索。虽然在 mini-OpenClaw 中，这个索引的数据源是用户的个人记忆文件，但其技术架构（文本切分 → Embedding → 向量索引 → 语义检索）与通用 RAG 完全一致。

&emsp;&emsp;2025-2026 的行业共识是：<font color=red>RAG 和长上下文不是非此即彼，而是协同使用</font>。RAG 作为默认检索架构（成本优势 10-100x），长上下文作为补充——当需要跨文档推理、固定小文档集、需要全局理解（如合同审查、法律分析）时，才考虑全上下文加载。

#### 2.7.1 动手验证：全量加载 vs 按需检索的 Token 对比

&emsp;&emsp;RAG 的成本优势到底有多大？我们用一个简化的场景来量化：构造一个包含 5 段技术文档的"小型知识库"，然后对比"全部塞进上下文"和"只检索最相关的 1 段"两种策略的 token 消耗和回答质量。

In [15]:
# === Select 策略演示：全量加载 vs 按需检索 ===

# 1. 构造一个小型"知识库"（5 段不同主题的技术文档）
knowledge_base = {
    "FastAPI路由": (
        "FastAPI 使用装饰器定义路由。@app.get('/items/{id}') 定义 GET 端点，"
        "支持路径参数自动解析和类型校验。依赖注入通过 Depends() 实现，支持嵌套依赖和生命周期管理。"
        "FastAPI 基于 Starlette 和 Pydantic，性能接近 NodeJS，同时提供完整的类型提示和自动文档生成。"
    ),
    "PostgreSQL索引": (
        "PostgreSQL 支持多种索引类型：B-tree（默认，适合等值和范围查询）、Hash（仅等值查询）、"
        "GiST（几何和全文搜索）、GIN（数组和 JSONB）。创建索引的基本语法：CREATE INDEX idx_name ON table(column)。"
        "复合索引的列顺序很重要：把选择性最高的列放在最前面。EXPLAIN ANALYZE 是验证索引是否生效的关键工具。"
    ),
    "Redis缓存": (
        "Redis 作为缓存层的常见模式：Cache-Aside（读时检查缓存→未命中→查 DB→写缓存）、"
        "Write-Through（写操作同时更新 DB 和缓存）、Write-Behind（异步批量写入 DB）。"
        "TTL 设置建议：热点数据 300-600s，冷数据 3600s。使用 pipeline 批量操作可减少 90% 的网络往返。"
    ),
    "Docker部署": (
        "Docker Compose 适合开发环境，生产环境推荐 Kubernetes 或 AWS ECS。"
        "多阶段构建（multi-stage build）可将镜像体积缩小 60-80%。"
        "健康检查配置：HEALTHCHECK --interval=30s --timeout=5s CMD curl -f http://localhost:8000/health。"
    ),
    "JWT认证": (
        "JWT 由 Header.Payload.Signature 三部分组成，Base64 编码。"
        "Access Token 有效期建议 15-30 分钟，Refresh Token 7-30 天。"
        "安全最佳实践：HttpOnly Cookie 存储、HTTPS 传输、密钥定期轮换、Token 黑名单机制。"
    ),
}

# 2. 用户问题（只和 PostgreSQL 索引相关）
question = "我的查询很慢，应该怎么建索引？用什么工具验证索引是否生效？"

# 策略A：全量加载（把所有文档塞进上下文）
full_context = "\n\n".join(f"【{title}】\n{content}" for title, content in knowledge_base.items())
prompt_full = f"参考资料：\n{full_context}\n\n用户问题：{question}"

# 策略B：按需检索（只加载最相关的文档——这里简化用关键词匹配模拟向量检索）
relevant_key = "PostgreSQL索引"  # 实际由向量检索返回
prompt_selective = f"参考资料：\n【{relevant_key}】\n{knowledge_base[relevant_key]}\n\n用户问题：{question}"

# 3. 对比 token 消耗
full_tokens = count_tokens(prompt_full)
selective_tokens = count_tokens(prompt_selective)

print(f"{'策略':<14} {'Token 消耗':>10} {'文档数':>6}")
print("-" * 36)
print(f"{'全量加载':<14} {full_tokens:>10} {len(knowledge_base):>6}")
print(f"{'按需检索':<14} {selective_tokens:>10} {'1':>6}")
print(f"\n按需检索节省 {full_tokens - selective_tokens} tokens（{(1-selective_tokens/full_tokens)*100:.0f}%）")
print(f"知识库越大，差距越大——1000 篇文档时差距可达 100 倍以上")

# 4. 验证回答质量（两种策略都能正确回答）
print(f"\n--- 按需检索的回答 ---")
response = llm.invoke(prompt_selective)
print(response.content[:300])


策略               Token 消耗    文档数
------------------------------------
全量加载                  426      5
按需检索                  113      1

按需检索节省 313 tokens（73%）
知识库越大，差距越大——1000 篇文档时差距可达 100 倍以上

--- 按需检索的回答 ---
针对查询慢的问题，可以按以下步骤优化：

## 一、索引创建策略

### 1. **分析查询模式**
```sql
-- 查看慢查询
SELECT query, calls, total_time, mean_time
FROM pg_stat_statements
ORDER BY mean_time DESC
LIMIT 10;
```

### 2. **根据查询类型选择索引**

**等值查询 + 范围查询（最常见）**
```sql
-- B-tree索引（默认）
CREATE INDEX idx_users_email ON users(email);
CREATE INDEX 


&emsp;&emsp;运行后你会看到：两种策略都能正确回答关于 PostgreSQL 索引的问题，但全量加载消耗的 token 是按需检索的 4-5 倍。在这个小规模示例中差距是几百 token，但在真实系统中——知识库动辄数万篇文档——差距会扩大到几个数量级。这就是为什么 RAG 成为 Select 策略的核心实现：<font color=red>不是把整个图书馆搬进鱼缸，而是只借最需要的那本书</font>。

&emsp;&emsp;至此，我们完成了六大功能模块的逐一拆解。回到鱼缸模型的视角：系统提示是最稳定的一层水（几乎不变），对话历史和工具上下文是增长最快的两个水龙头，记忆注入和外部知识是按需加载的精准滴管，任务状态是 Agent 自主管理的一小块水域。理解了这六个模块的角色和特征，我们就有了分析任何 Agent 系统的坐标系。接下来，我们要学习如何用 Anthropic 的五大策略来管理这些模块。

---

## 3. 五大策略 × 六模块交叉矩阵

&emsp;&emsp;在模块二中，我们建立了六模块坐标系——知道了上下文窗口里有什么。但知道"有什么"还不够，工程师需要知道"怎么管"。Anthropic 在 2025 年 9 月发布的[上下文工程文档](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)中，系统性地提出了五大核心策略：Write、Select、Compress、Isolate、Cache。这五种策略不是互斥选项，而是组合使用的层次——生产级系统通常同时使用 3-5 种策略。本模块将这五种策略映射到六大功能模块上，形成一个可操作的双坐标分析框架。

### 3.1 五大策略总览：Write / Select / Compress / Isolate / Cache

&emsp;&emsp;我们先用一张表建立对五种策略的全局认知：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Anthropic 五大上下文管理策略总览</font></p>
<div class="center">

| 策略 | 机制 | 典型实现 | 适用场景 |
|------|------|---------|---------|
| **Write（写入持久化）** | 将信息写到上下文窗口之外的持久化存储 | Scratchpad（CLAUDE.md / todo.md）、文件系统、数据库 | 跨轮次状态追踪、长时间任务 |
| **Select（运行时检索）** | Just-in-time 按需加载相关上下文 | RAG 检索、glob/grep、动态工具加载、Agent Skills 条件注入 | 知识密集型、工具密集型场景 |
| **Compress（上下文压缩）** | 保留关键信息的同时缩减 Token 消耗 | 工具结果清除、观察遮蔽、LLM 摘要、三层递进压缩 | 所有长对话场景 |
| **Isolate（上下文隔离）** | 防止无关上下文相互干扰 | 子 Agent 架构、域分离、上下文分区 | 多步骤任务、可并行子任务 |
| **Cache（提示缓存）** | 复用不变的提示前缀，避免重复计算 | Anthropic Prompt Caching、OpenAI 自动缓存 | 系统提示 + 工具描述固定的 Agent |

</div>

&emsp;&emsp;Claude Code 是五策略全用的典型案例：Cache（Prompt Caching 公测）+ Isolate（子 Agent via tool-use chain）+ Write（CLAUDE.md / Memory Tool）+ Select（glob/grep 按需检索）+ Compress（auto-compact 95% 触发）。这告诉我们：<font color=red>五策略不是"选一个用"，而是"全部组合起来用"</font>。

#### 3.1.1 Write 策略：将关键信息卸载到上下文之外

&emsp;&emsp;Write 的核心思想是：**不要指望上下文窗口永远记住一切——把重要信息主动写到外部持久化存储**。对话历史终将被压缩或截断，但写入文件/数据库的信息不会丢失。

&emsp;&emsp;如果你觉得这个思想很熟悉，那就对了——<font color=red>Write 策略正是我们在前课中已经完整实现过的记忆机制的上层抽象</font>。在前课中，我们分两层实现了 Write：

- **短期记忆层的 Write**：`SessionManager` 在对话轮次超过 `MAX_HISTORY` 时，调用 LLM 将早期对话压缩为摘要，写入 `compressed_context` 字段
- **长期记忆层的 Write**：`mem0.add()` 用 LLM 裁判机制从对话中提取关键事实，结构化后存入向量数据库

&emsp;&emsp;下面我们用同一段对话，分别演示这两层 Write 的实际效果。

**短期记忆层的 Write：LLM 压缩摘要**

&emsp;&emsp;短期记忆的 Write 目标是：**在不丢失关键语义的前提下，用更少的 Token 保留更多的对话信息**。前课的 `SessionManager` 在消息数超过 `COMPRESS_TRIGGER` 时，将早期对话压缩为摘要写入 `compressed_context` 字段，同时只保留最近的消息在 `messages` 中——整个 session 以 JSON 文件持久化到 `sessions/` 目录。下面我们复现这个完整过程，从创建 session 文件、触发压缩、到验证持久化结果：

In [16]:
# === 短期记忆层的 Write：LLM 压缩摘要 + 持久化到 sessions/ ===
import json, os, time

SESSIONS_DIR = "./sessions"
os.makedirs(SESSIONS_DIR, exist_ok=True)

# 核心参数（与 SessionManager 一致：消息数达到阈值时，压缩前半部分，只保留后半部分）
COMPRESS_TRIGGER = 8   # 演示用小阈值（生产环境 SessionManager 用 20）
KEEP_RECENT = 4        # 压缩后保留最近 4 条消息

# ---- 步骤 1：创建 session，写入 8 条多轮对话 ----
session_id = "write_demo"
session = {
    "title": "新项目技术选型讨论",
    "created_at": time.strftime("%Y-%m-%d %H:%M"),
    "updated_at": time.strftime("%Y-%m-%d %H:%M"),
    "compressed_context": "",   # 压缩前为空
    "messages": [
        {"role": "user", "content": "新项目的数据库用什么？"},
        {"role": "assistant", "content": "推荐 PostgreSQL。理由：项目有大量关联查询，团队熟悉度高，配合 SQLAlchemy 2.0 的 async 支持性能优秀。"},
        {"role": "user", "content": "缓存方案呢？"},
        {"role": "assistant", "content": "Redis，Cache-Aside 模式。热点数据 TTL=300s，写操作先更新 DB 再删缓存。"},
        {"role": "user", "content": "部署方案定了吗？"},
        {"role": "assistant", "content": "Docker Compose 本地开发，AWS ECS 生产。CI/CD 用 GitHub Actions，自动化金丝雀发布。"},
        {"role": "user", "content": "预算上限是多少？"},
        {"role": "assistant", "content": "初期 15 万以内。服务器 3 万/年，API 调用预留 2 万/月。"},
    ]
}

# 写入 JSON（压缩前：8 条消息，compressed_context 为空）
session_path = os.path.join(SESSIONS_DIR, f"{session_id}.json")
with open(session_path, "w", encoding="utf-8") as f:
    json.dump(session, f, ensure_ascii=False, indent=2)

print(f"=== 压缩前：{session_path} ===")
print(f"messages: {len(session['messages'])} 条（达到 COMPRESS_TRIGGER={COMPRESS_TRIGGER}）")
print(f"compressed_context: （空）")

# ---- 步骤 2：达到阈值，压缩前 4 条为摘要，只保留后 4 条 ----
if len(session["messages"]) >= COMPRESS_TRIGGER:
    early_messages = session["messages"][:-KEEP_RECENT]   # 前 4 条：将被压缩
    recent_messages = session["messages"][-KEEP_RECENT:]  # 后 4 条：保留原文

    # LLM 压缩早期对话（一句话回复，控制运行时间）
    early_text = "\n".join(f'{m["role"]}: {m["content"]}' for m in early_messages)
    compress_prompt = f"""请将以下对话历史压缩为一段简洁的摘要，保留所有关键技术决策和数字：\n\n{early_text}\n\n输出要求：一段话，不超过 80 字。"""
    summary = llm.invoke(compress_prompt).content

    # 写回 session：compressed_context 存摘要，messages 只留最近的
    session["compressed_context"] = summary
    session["messages"] = recent_messages
    session["updated_at"] = time.strftime("%Y-%m-%d %H:%M")

    with open(session_path, "w", encoding="utf-8") as f:
        json.dump(session, f, ensure_ascii=False, indent=2)

# ---- 步骤 3：展示压缩结果 ----
original_tokens = count_tokens(early_text)
summary_tokens = count_tokens(summary)

print(f"\n=== 压缩后：{session_path} ===")
print(f"compressed_context ({summary_tokens} tokens，原始 {original_tokens} tokens，压缩率 {(1 - summary_tokens/original_tokens)*100:.0f}%)：")
print(f"  {summary}")
print(f"messages: {len(session['messages'])} 条（只保留最近 {KEEP_RECENT} 条）")
for m in session["messages"]:
    print(f"  [{m['role']}] {m['content'][:50]}")

# 从文件读回验证
print(f"\n=== 验证：从文件读回 ===")
with open(session_path, "r", encoding="utf-8") as f:
    saved = json.load(f)
print(f"compressed_context 长度: {len(saved['compressed_context'])} 字符")
print(f"messages 数量: {len(saved['messages'])} 条")

# 清理演示文件
# import shutil
# shutil.rmtree(SESSIONS_DIR)
# print(f"\n（演示清理：已删除 {SESSIONS_DIR}/）")

=== 压缩前：./sessions/write_demo.json ===
messages: 8 条（达到 COMPRESS_TRIGGER=8）
compressed_context: （空）

=== 压缩后：./sessions/write_demo.json ===
compressed_context (53 tokens，原始 79 tokens，压缩率 33%)：
  技术决策：数据库选用PostgreSQL（适合关联查询，团队熟悉，配合SQLAlchemy 2.0异步性能优）；缓存采用Redis的Cache-Aside模式，热点数据TTL为300秒，写操作先更新数据库再删除缓存。
messages: 4 条（只保留最近 4 条）
  [user] 部署方案定了吗？
  [assistant] Docker Compose 本地开发，AWS ECS 生产。CI/CD 用 GitHub Acti
  [user] 预算上限是多少？
  [assistant] 初期 15 万以内。服务器 3 万/年，API 调用预留 2 万/月。

=== 验证：从文件读回 ===
compressed_context 长度: 108 字符
messages 数量: 4 条


&emsp;&emsp;运行后打开 `sessions/write_demo.json`，可以清楚地看到压缩前后的变化：压缩前 `compressed_context` 为空、`messages` 有 8 条；压缩后早期 4 条对话被 LLM 浓缩为一段摘要写入 `compressed_context`，`messages` 只剩最近 4 条。下次对话时，`SessionManager` 会将 `compressed_context` 作为 system message 注入 LLM，让模型「记住」早期对话的要点，而不需要占用完整的 Token 开销。

&emsp;&emsp;<font color=red>这就是短期 Write 的本质：用 LLM 做有损压缩，用文件系统做持久化，在 Token 成本和信息保留之间取得平衡。</font>

**长期记忆层的 Write：mem0 选择性提取**

&emsp;&emsp;长期记忆的 Write 目标完全不同：**从对话中识别出值得跨会话保留的关键事实，结构化后写入向量数据库**。前课的 `mem0.add()` 内部会调用 LLM 裁判（`_extract_facts()`）判断哪些信息值得记住，然后执行 ADD/UPDATE/NONE 三分类。

&emsp;&emsp;这里有一个关键细节：<font color=red>`mem0` 默认从 user 角色的消息中提取事实</font>。因此，如果用户只是在提问（「数据库用什么？」），`mem0` 提取不到有价值的决策信息。真实场景中，用户会在对话中确认决策（「数据库我们决定用 PostgreSQL」），这些确认性表述才是长期记忆的提取目标。下面我们模拟这个场景：

In [17]:
# === 长期记忆层的 Write：mem0.add() 选择性提取 ===
import os, shutil
from mem0 import Memory

QDRANT_PATH = "./qdrant_write_demo"

# 清理残留锁（Notebook 重复运行时文件锁不会自动释放）
for p in [QDRANT_PATH, os.path.expanduser("~/.mem0/migrations_qdrant")]:
    if os.path.exists(p):
        shutil.rmtree(p)

# mem0 配置（与进阶课件一致：DeepSeek 做 LLM，OpenAI 做 Embedding）
config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "deepseek-chat",
            "api_key": os.getenv("DEEPSEEK_API_KEY"),
            "openai_base_url": "https://api.deepseek.com/v1",
            "temperature": 0.1,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "write_demo",
            "path": QDRANT_PATH,
        }
    },
    "version": "v1.1"
}

memory = Memory.from_config(config)

# 关键：mem0 默认从 user 消息中提取事实
# 因此对话需要反映真实场景——用户确认技术决策，而非只是提问
decisions_conversation = [
    {"role": "user", "content": "数据库我们决定用 PostgreSQL，项目有大量关联查询，团队熟悉度高，配合 SQLAlchemy 2.0 做 async"},
    {"role": "assistant", "content": "好的，PostgreSQL + SQLAlchemy 2.0 async 已记录。"},
    {"role": "user", "content": "缓存方案定了，用 Redis Cache-Aside 模式，热点数据 TTL 设 300 秒，写操作先更新 DB 再删缓存"},
    {"role": "assistant", "content": "了解，Redis Cache-Aside + write-through 策略。"},
    {"role": "user", "content": "部署方案：本地用 Docker Compose，生产上 AWS ECS，CI/CD 走 GitHub Actions 自动金丝雀发布"},
    {"role": "assistant", "content": "部署流水线已记录。"},
    {"role": "user", "content": "预算上限 15 万，服务器 3 万一年，API 调用预留 2 万每月"},
    {"role": "assistant", "content": "预算约束已记录。"},
]

# 一行代码完成「提取 + 去重 + 向量化 + 存储」
add_result = memory.add(decisions_conversation, user_id="dev_team_lead")

print("=== 长期 Write：mem0.add() 提取的记忆条目 ===")
for item in add_result.get("results", []):
    event = item.get("event", "UNKNOWN")
    mem_text = item.get("memory", "")
    print(f"  [{event}] {mem_text}")

# 验证：search 能检索到刚才写入的记忆
search_result = memory.search("项目用什么数据库", user_id="dev_team_lead")
print(f'\n=== 验证：search("项目用什么数据库") ===')
for item in search_result.get("results", []):
    print(f"  [score={item['score']:.2f}] {item['memory']}")

# Token 对比
mem_texts = [item.get("memory", "") for item in add_result.get("results", [])]
mem_tokens = sum(count_tokens(t) for t in mem_texts)
print(f"\n=== 三层 Write 的 Token 经济学对比 ===")
print(f"原始对话:     {original_tokens} tokens（完整历史）")
print(f"短期 Write:   {summary_tokens} tokens（压缩摘要，保留在 session 内）")
print(f"长期 Write:   {mem_tokens} tokens（{len(mem_texts)} 条结构化记忆，永久存入向量库）")

# 清理演示数据
# if os.path.exists(QDRANT_PATH):
#     shutil.rmtree(QDRANT_PATH)


=== 长期 Write：mem0.add() 提取的记忆条目 ===
  [ADD] 数据库决定用 PostgreSQL，项目有大量关联查询，团队熟悉度高，配合 SQLAlchemy 2.0 做 async
  [ADD] 缓存方案定了，用 Redis Cache-Aside 模式，热点数据 TTL 设 300 秒，写操作先更新 DB 再删缓存
  [ADD] 部署方案：本地用 Docker Compose，生产上 AWS ECS，CI/CD 走 GitHub Actions 自动金丝雀发布
  [ADD] 预算上限 15 万，服务器 3 万一年，API 调用预留 2 万每月

=== 验证：search("项目用什么数据库") ===
  [score=0.54] 数据库决定用 PostgreSQL，项目有大量关联查询，团队熟悉度高，配合 SQLAlchemy 2.0 做 async
  [score=0.51] 缓存方案定了，用 Redis Cache-Aside 模式，热点数据 TTL 设 300 秒，写操作先更新 DB 再删缓存
  [score=0.49] 预算上限 15 万，服务器 3 万一年，API 调用预留 2 万每月
  [score=0.45] 部署方案：本地用 Docker Compose，生产上 AWS ECS，CI/CD 走 GitHub Actions 自动金丝雀发布

=== 三层 Write 的 Token 经济学对比 ===
原始对话:     79 tokens（完整历史）
短期 Write:   53 tokens（压缩摘要，保留在 session 内）
长期 Write:   108 tokens（4 条结构化记忆，永久存入向量库）


&emsp;&emsp;对比短期 Write 和长期 Write 的输出：短期层保留了对话的**完整语境**（「谁问了什么、怎么回答的」），压缩后仍然是一段连贯叙述；而长期层只提取了**结构化的事实**（「数据库用 PostgreSQL」「缓存用 Redis」），每条记忆都是独立可检索的知识单元。两者的存储位置也不同——短期摘要留在 session JSON 文件中，随会话生命周期存在；长期记忆写入向量数据库，跨会话永久可用。

**任务状态层的 Write：Scratchpad / todo.md**

&emsp;&emsp;除了记忆层的两种 Write，还有一种在生产 Agent 中极为常见的 Write 形态：**把任务执行进度写入结构化文件**（如 `todo.md`、`CLAUDE.md`），让 Agent 在上下文窗口被重置后能恢复执行状态。这对应我们在 2.6 节介绍的「任务状态层」——Scratchpad 就是 Agent 的「草稿纸」。

&emsp;&emsp;与前两种 Write 的关键区别：短期压缩保留的是**对话语义**，mem0 提取的是**用户知识**，而 todo.md 记录的是**任务进度**——哪些步骤做完了、当前卡在哪、下一步是什么。下面模拟一个多步骤项目任务，展示 Agent 如何通过 Write to Scratchpad 实现「断点续传」：

In [19]:
# === 任务状态层的 Write：Scratchpad（todo.md）断点续传 ===
import json, os

TODO_FILE = "./todo_demo.md"

# ---- 第一阶段：Agent 执行任务并实时写入进度 ----
task_plan = [
    {"step": 1, "task": "初始化 PostgreSQL 连接池", "status": "done", "result": "pool_size=20, max_overflow=10"},
    {"step": 2, "task": "创建 users 表迁移脚本", "status": "done", "result": "alembic revision --autogenerate 完成"},
    {"step": 3, "task": "实现 JWT 认证中间件", "status": "in_progress", "result": "refresh token 逻辑未完成"},
    {"step": 4, "task": "配置 Redis 缓存层", "status": "pending", "result": ""},
    {"step": 5, "task": "编写 API 集成测试", "status": "pending", "result": ""},
]

# Agent 将当前进度写入 todo.md（Write to Scratchpad）
def write_todo(tasks, filepath):
    lines = ["# 项目任务进度\n\n"]
    status_icon = {"done": "[x]", "in_progress": "[-]", "pending": "[ ]"}
    for t in tasks:
        icon = status_icon[t["status"]]
        line = f"{icon} Step {t['step']}: {t['task']}"
        if t["result"]:
            line += f" → {t['result']}"
        lines.append(line + "\n")
    with open(filepath, "w") as f:
        f.writelines(lines)

write_todo(task_plan, TODO_FILE)
print("=== Agent 写入 todo.md（模拟执行到 Step 3 中断）===")
print(open(TODO_FILE).read())

# ---- 第二阶段：模拟上下文重置（新会话/auto-compact 后）----
# Agent 读回 todo.md，恢复执行状态
def read_todo(filepath):
    with open(filepath) as f:
        content = f.read()
    # 解析出当前进度
    done = content.count("[x]")
    in_progress = content.count("[-]")
    pending = content.count("[ ]")
    return content, done, in_progress, pending

content, done, in_prog, pending = read_todo(TODO_FILE)
print("=== 新会话：Agent 读取 todo.md 恢复状态 ===")
print(f"已完成: {done} | 进行中: {in_prog} | 待开始: {pending}")
print(f"→ Agent 决策：继续 Step 3（JWT refresh token），无需从头开始")

# Token 对比：todo.md vs 完整对话历史
todo_tokens = count_tokens(content)
# 假设完成前 3 步产生了约 20 轮对话
estimated_history = "user: ...\nassistant: ...\n" * 20
history_tokens = count_tokens(estimated_history) * 5  # 真实对话每轮约 200 tokens
print(f"\n=== Scratchpad Write 的 Token 经济学 ===")
print(f"todo.md:        {todo_tokens} tokens（结构化进度）")
print(f"完整对话历史:   ~{history_tokens} tokens（20 轮对话估算）")
print(f"恢复执行所需上下文：只需 todo.md + 当前步骤描述，不需要回放全部历史")

# 清理
os.remove(TODO_FILE)


=== Agent 写入 todo.md（模拟执行到 Step 3 中断）===
# 项目任务进度

[x] Step 1: 初始化 PostgreSQL 连接池 → pool_size=20, max_overflow=10
[x] Step 2: 创建 users 表迁移脚本 → alembic revision --autogenerate 完成
[-] Step 3: 实现 JWT 认证中间件 → refresh token 逻辑未完成
[ ] Step 4: 配置 Redis 缓存层
[ ] Step 5: 编写 API 集成测试

=== 新会话：Agent 读取 todo.md 恢复状态 ===
已完成: 2 | 进行中: 1 | 待开始: 2
→ Agent 决策：继续 Step 3（JWT refresh token），无需从头开始

=== Scratchpad Write 的 Token 经济学 ===
todo.md:        103 tokens（结构化进度）
完整对话历史:   ~700 tokens（20 轮对话估算）
恢复执行所需上下文：只需 todo.md + 当前步骤描述，不需要回放全部历史


&emsp;&emsp;同一段对话，三层 Write 各取所需：短期层压缩为摘要，保留在 session 内供后续对话引用；长期层提取关键事实，写入向量数据库供跨会话检索；任务状态层记录执行进度，写入 todo.md 供断点续传。这就是 Write 策略的完整形态——**任何将信息从上下文窗口卸载到外部存储的操作，都是 Write**。

&emsp;&emsp;在实际 Agent 系统中（如 Claude Code），这三层 Write 同时运作：`auto-compact` 压缩对话历史（短期），`Memory Tool` 提取用户偏好（长期），`CLAUDE.md` / `todo.md` 追踪任务进度（Scratchpad）。三者协同，确保 Agent 在任何时刻都不会因为上下文窗口的物理限制而「失忆」。

#### 3.1.2 Select 策略：只加载当前任务需要的上下文

&emsp;&emsp;Select 的核心思想是：**不是把所有信息都塞进上下文，而是根据当前任务动态选择最相关的子集**。这个原则贯穿 Agent 系统的每一层——从向量数据库的语义检索，到文件系统的 Glob+Grep 定位，到知识库的混合召回，再到 Skills 路由的整体上下文切换。接下来我们逐一演示四种 Select 技术，看它们如何在不同数据源上实现"精准捞取"：

##### 3.1.2.1 mem0 memory.search：向量语义搜索

&emsp;&emsp;在前课中，我们学习了 `mem0.add()` 如何将对话中的关键事实提取并存入向量数据库；在上一节的 Write 演示中，我们已经向 Qdrant 写入了一组技术决策记忆。现在，Select 策略要做的是：<font color=red>当 Agent 收到新任务时，只检索与当前任务语义相关的记忆子集，而不是把整个记忆库都注入上下文</font>。

In [20]:
# === Select 策略：mem0 memory.search 向量语义搜索 ===
# memory 对象已在 3.1.1 Write 演示中初始化（Qdrant 本地向量库），直接复用

# 场景：Agent 收到新任务——"帮我配置数据库连接池"
# Select 策略：先搜索长期记忆，看项目之前做过什么技术决策
task_query = "服务器预算"
results = memory.search(query=task_query, user_id="dev_team_lead", limit=2)

print(f'=== memory.search("{task_query}") ===')
print(f"召回 {len(results.get('results', []))} 条相关记忆：\n")
for item in results.get("results", []):
    score = item.get("score", 0)
    mem = item.get("memory", "")
    print(f"  [相关度 {score:.2f}] {mem}")

# 对比：换一个与已有记忆无关的查询
irrelevant_query = "本地部署方案"
irr_results = memory.search(query=irrelevant_query, user_id="dev_team_lead", limit=2)
print(f'\n=== memory.search("{irrelevant_query}") ===')
print(f"召回 {len(irr_results.get('results', []))} 条记忆：\n")
for item in irr_results.get("results", []):
    score = item.get("score", 0)
    mem = item.get("memory", "")
    print(f"  [相关度 {score:.2f}] {mem}")

# Token 节省分析：全量 vs Select
all_mems = memory.get_all(user_id="dev_team_lead")
all_text = "\n".join(m.get("memory", "") for m in all_mems.get("results", []))
selected_text = "\n".join(
    item.get("memory", "") for item in results.get("results", [])
    if item.get("score", 0) > 0.3  # 只保留相关度 > 0.3 的记忆
)
print(f"\n=== Select 效果：语义过滤的 Token 节省 ===")
print(f"记忆库全量注入: {count_tokens(all_text)} tokens（{len(all_mems.get('results', []))} 条）")
print(f"Select 后注入:  {count_tokens(selected_text)} tokens（仅高相关度条目）")
if count_tokens(all_text) > 0:
    print(f"节省:           {(1 - count_tokens(selected_text)/count_tokens(all_text))*100:.0f}%")

=== memory.search("服务器预算") ===
召回 2 条相关记忆：

  [相关度 0.65] 预算上限 15 万，服务器 3 万一年，API 调用预留 2 万每月
  [相关度 0.42] 缓存方案定了，用 Redis Cache-Aside 模式，热点数据 TTL 设 300 秒，写操作先更新 DB 再删缓存

=== memory.search("本地部署方案") ===
召回 2 条记忆：

  [相关度 0.52] 部署方案：本地用 Docker Compose，生产上 AWS ECS，CI/CD 走 GitHub Actions 自动金丝雀发布
  [相关度 0.37] 缓存方案定了，用 Redis Cache-Aside 模式，热点数据 TTL 设 300 秒，写操作先更新 DB 再删缓存

=== Select 效果：语义过滤的 Token 节省 ===
记忆库全量注入: 111 tokens（4 条）
Select 后注入:  56 tokens（仅高相关度条目）
节省:           50%


&emsp;&emsp;注意两次搜索的结果差异：查询"数据库连接配置"时，mem0 准确召回了 PostgreSQL 相关的技术决策（因为 Write 阶段正好存入了这类记忆）；而查询"前端 UI 框架"时，由于我们的记忆库中没有前端相关的决策记录，召回结果要么为空，要么相关度极低。这就是 Select 的价值——<font color=red>不是"有什么就注入什么"，而是"需要什么才注入什么"</font>。

&emsp;&emsp;memory.search 的本质是**连续排序**（按语义相关度评分，可设置阈值过滤），而不是离散的标签匹配。这使它比简单的标签路由更灵活，但也更依赖 Embedding 模型的质量——如果 Embedding 不准，可能漏召回关键记忆。

##### 3.1.2.2 Glob + Grep 文件精确检索

&emsp;&emsp;前一种 Select 操作的是结构化数据（向量数据库），而 Glob + Grep 操作的是**非结构化的文件系统**。这是 Coding Agent（如 Claude Code、Cursor）最核心的 Select 能力：面对一个几万文件的代码仓库，Agent 不可能把所有文件都读进上下文，必须先用 Glob 按文件名模式缩小范围，再用 Grep 按关键词精确定位。

In [22]:
# === Select 策略：Glob + Grep 文件精确检索 ===
import glob, os, shutil

# 构建模拟项目目录（实际 Agent 操作真实代码仓库）
PROJECT = "./select_demo_project"
os.makedirs(f"{PROJECT}/backend/api", exist_ok=True)
os.makedirs(f"{PROJECT}/backend/graph", exist_ok=True)
os.makedirs(f"{PROJECT}/backend/models", exist_ok=True)
os.makedirs(f"{PROJECT}/frontend/src", exist_ok=True)

mock_files = {
    f"{PROJECT}/backend/api/chat.py": (
        "from fastapi import APIRouter\n"
        "# SSE 流式端点，处理用户消息\n"
        "async def chat_stream(request):\n"
        "    session = SessionManager()\n"
        "    response = await agent.invoke(request.message)\n"
    ),
    f"{PROJECT}/backend/graph/agent.py": (
        "from langchain_deepseek import ChatDeepSeek\n"
        "from mem0 import Memory\n"
        "MAX_HISTORY = 20\n"
        "class AgentManager:\n"
        "    def __init__(self):\n"
        "        self.memory = Memory.from_config(config)\n"
    ),
    f"{PROJECT}/backend/graph/session_manager.py": (
        "import json\n"
        "class SessionManager:\n"
        "    def compress_history(self, messages):\n"
        "        # LLM 摘要压缩，保留决策链\n"
        "        summary = llm.invoke(compress_prompt)\n"
    ),
    f"{PROJECT}/backend/graph/mem0_manager.py": (
        "from mem0 import Memory\n"
        "def get_typed_context(user_id, query):\n"
        "    results = memory.search(query, user_id=user_id)\n"
        "    return group_by_type(results)\n"
    ),
    f"{PROJECT}/backend/models/database.py": (
        "from sqlalchemy import create_engine\n"
        "DATABASE_URL = os.getenv('DATABASE_URL')\n"
        "engine = create_engine(DATABASE_URL, pool_size=10)\n"
    ),
    f"{PROJECT}/frontend/src/App.tsx": (
        "import React from 'react'\n"
        "function App() { return <ChatWindow /> }\n"
    ),
}
for path, content in mock_files.items():
    with open(path, "w") as f:
        f.write(content)

# Step 1: Glob — 按文件名模式快速圈定范围
pattern = f"{PROJECT}/backend/**/*.py"
py_files = sorted(glob.glob(pattern, recursive=True))
print(f"=== Step 1: Glob(\"{pattern}\") ===")
print(f"匹配 {len(py_files)} 个 Python 文件：")
for f in py_files:
    print(f"  {f}")

# Step 2: Grep — 按关键词从候选文件中精确筛选
keyword = "Memory"
print(f'\n=== Step 2: Grep(\"{keyword}\") — 在 Glob 结果中搜索 ===')
matched = []
for filepath in py_files:
    with open(filepath) as f:
        content = f.read()
    hits = [(i+1, line.strip()) for i, line in enumerate(content.split("\n")) if keyword in line]
    if hits:
        matched.append(filepath)
        for lineno, line in hits:
            print(f"  {os.path.basename(filepath)}:{lineno}  {line}")

# Token 节省分析
all_content = "\n".join(open(f).read() for f in py_files)
sel_content = "\n".join(open(f).read() for f in matched)
print(f"\n=== Select 效果：两阶段过滤的 Token 节省 ===")
print(f"项目全部文件:  6 个（含前端）")
print(f"Glob 过滤后:   {len(py_files)} 个 Python 文件 → {count_tokens(all_content)} tokens")
print(f"Grep 精筛后:   {len(matched)} 个含 Memory 的文件 → {count_tokens(sel_content)} tokens")
print(f"两阶段节省:    {(1 - count_tokens(sel_content)/max(count_tokens(all_content),1))*100:.0f}%")

# 清理
shutil.rmtree(PROJECT)

=== Step 1: Glob("./select_demo_project/backend/**/*.py") ===
匹配 5 个 Python 文件：
  ./select_demo_project/backend/api/chat.py
  ./select_demo_project/backend/graph/agent.py
  ./select_demo_project/backend/graph/mem0_manager.py
  ./select_demo_project/backend/graph/session_manager.py
  ./select_demo_project/backend/models/database.py

=== Step 2: Grep("Memory") — 在 Glob 结果中搜索 ===
  agent.py:2  from mem0 import Memory
  agent.py:6  self.memory = Memory.from_config(config)
  mem0_manager.py:1  from mem0 import Memory

=== Select 效果：两阶段过滤的 Token 节省 ===
项目全部文件:  6 个（含前端）
Glob 过滤后:   5 个 Python 文件 → 199 tokens
Grep 精筛后:   2 个含 Memory 的文件 → 83 tokens
两阶段节省:    58%


&emsp;&emsp;Glob + Grep 的核心价值在于**两阶段漏斗**：Glob 是粗筛（按路径/扩展名），Grep 是精筛（按内容关键词）。这种模式在 Claude Code 中无处不在——当你让它"修改数据库相关代码"时，它会先 `Glob("**/*.py")` 找到所有 Python 文件，再 `Grep("database|sql|engine")` 定位到具体的文件和行号。与 memory.search 的语义检索不同，Glob + Grep 是**精确匹配**，不会出现语义漂移，但也无法理解同义词（比如搜"数据库"不会命中"DB"）。

> &emsp;**实际 Agent 中的组合策略**：生产级 Coding Agent 通常会组合使用 Glob + Grep + 向量检索：先用 Glob 缩小文件范围（从 10 万→500），再用 Grep 精确定位（500→20），最后用 Embedding 对这 20 个候选做语义排序。三级漏斗的 token 节省率可达 99%+。

##### 3.1.2.3 LlamaIndex Embedding + BM25 混合召回

&emsp;&emsp;前面两种 Select 各有侧重：memory.search 是纯语义检索，Glob+Grep 是纯关键词匹配。但在 RAG（检索增强生成）场景中，单一检索通道往往不够——<font color=red>语义检索擅长理解意图但可能漏掉关键词，关键词检索精确但无法理解同义词</font>。混合召回（Hybrid Retrieval）通过同时运行两条通道并合并结果，兼得两者优势。

In [ ]:
# === Select 策略：LlamaIndex Embedding + BM25 混合召回 ===
from llama_index.core import VectorStoreIndex, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.embeddings.dashscope import DashScopeEmbedding

# 构造知识库文档（模拟 Agent 的外部知识源——技术运维手册）
knowledge_docs = [
    Document(text="PostgreSQL 连接池推荐 pgbouncer，transaction 模式下单连接可复用，"
                   "默认 max_connections 设为 CPU 核心数 × 2 + 1，超过此值性能反而下降。"),
    Document(text="Redis Cache-Aside 标准流程：读请求先查缓存，miss 则查 DB 并回填；"
                   "写请求先更新 DB 再删缓存键，TTL 推荐 300 秒。"),
    Document(text="Docker Compose 适合本地开发，volumes 挂载实现热重载；"
                   "生产环境推荐 AWS ECS 或 Kubernetes，配合 ALB 做负载均衡。"),
    Document(text="LangChain Agent 工具定义需包含 name、description、func 三字段。"
                   "description 质量直接影响工具选择准确率，建议包含使用场景和输入格式说明。"),
    Document(text="向量数据库选型：Qdrant 适合中小规模（百万级），Milvus 适合大规模（亿级），"
                   "Chroma 适合原型验证，Pinecone 适合免运维的云端场景。"),
    Document(text="金丝雀发布策略：先切 5% 流量到新版本，监控 P99 延迟和错误率 15 分钟，"
                   "无异常后按 5→25→50→100% 逐步扩大。回滚阈值：错误率 >1% 或 P99 >500ms。"),
]

# ---- 通道 A：Embedding 向量语义召回 ----
embed_model = OpenAIEmbedding(
    model_name="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# 设置 Qwen Embedding模型
# embed_model = DashScopeEmbedding(
#     model_name="text-embedding-v4",
#     api_key=os.getenv("DASHSCOPE_API_KEY"),
#     api_base=os.getenv("DASHSCOPE_BASE_URL", "https://api.dashscope.aliyuncs.com")
# )

splitter = SentenceSplitter(chunk_size=256, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents(knowledge_docs)
index = VectorStoreIndex(nodes, embed_model=embed_model)
vector_retriever = index.as_retriever(similarity_top_k=3)

query = "数据库连接池怎么配置"
vector_results = vector_retriever.retrieve(query)

print(f'=== 通道 A：Embedding 语义召回（query="{query}"）===')
for r in vector_results:
    print(f"  [score={r.score:.3f}] {r.text[:80]}...")

# ---- 通道 B：BM25 关键词召回（LlamaIndex 内置）----
# skip_stemming=True：跳过英文词干提取（中文无此需求）
# token_pattern：中文按字切分 + 英文按词切分（BM25 中文场景标准配置）
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3,
    skip_stemming=True,
    token_pattern=r"[\u4e00-\u9fff]|[a-zA-Z0-9]+",
)

bm25_results = bm25_retriever.retrieve(query)
print(f'\n=== 通道 B：BM25 关键词召回 ===')
for r in bm25_results:
    print(f"  [score={r.score:.3f}] {r.text[:80]}...")

# ---- QueryFusionRetriever：内置 RRF 合并两路结果 ----
# mode="reciprocal_rerank"：Reciprocal Rank Fusion，按排名倒数加权合并
fusion_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1,             # 不做查询扩展，直接用原始 query
    mode="reciprocal_rerank",  # RRF 合并策略
    use_async=False,
)

fusion_results = fusion_retriever.retrieve(query)
print(f'\n=== QueryFusionRetriever 混合召回（RRF 合并）===')
for r in fusion_results:
    print(f"  [rrf={r.score:.4f}] {r.text[:80]}...")

# Token 对比
all_text = "\n".join(d.text for d in knowledge_docs)
top2_text = "\n".join(r.text for r in fusion_results[:2])
print(f"\n=== Select 效果：混合召回的 Token 节省 ===")
print(f"知识库全量: {count_tokens(all_text)} tokens（{len(knowledge_docs)} 篇文档）")
print(f"混合 Top-2: {count_tokens(top2_text)} tokens")
print(f"节省:       {(1 - count_tokens(top2_text)/max(count_tokens(all_text),1))*100:.0f}%")

&emsp;&emsp;这段代码完整展示了混合召回的三步流程：**双通道并行检索 → RRF 合并排序 → 截取 Top-K 注入上下文**。LlamaIndex 的 `QueryFusionRetriever` 内置了 RRF（Reciprocal Rank Fusion）算法——它不关心两个通道的绝对分数（因为 Embedding 分数和 BM25 分数量纲不同），只看每个文档在各通道中的排名位置，按排名倒数加权合并。`mode="reciprocal_rerank"` 一行配置即可完成，无需手写合并逻辑。

&emsp;&emsp;混合召回相比单通道的优势在于**互补性**：当用户查询"数据库连接池怎么配置"时，Embedding 通道能理解"配置"和"设置"是同义词，即使文档中没有出现"配置"二字也能召回；而 BM25 通道能精确匹配"连接池"这个关键词，避免 Embedding 将语义相近但主题不同的文档排在前面。<font color=red>单通道检索的典型失败模式：纯语义检索可能把"Redis 连接池"排在"PostgreSQL 连接池"前面（语义更近），而 BM25 能通过精确匹配纠正这种偏差</font>。

> &emsp;**中文 BM25 踩坑提醒**：`BM25Retriever` 默认使用英文词干提取器（Stemmer），对中文文本会产生全零分数。解决方案：设置 `skip_stemming=True` 跳过词干提取，同时用 `token_pattern=r"[\u4e00-\u9fff]|[a-zA-Z0-9]+"` 让中文按字切分、英文按词切分。

##### 3.1.2.4 Skills 路由：意图分类驱动的上下文切换

&emsp;&emsp;前面三种 Select 都是在**单一数据源内**做筛选（记忆库、文件系统、知识库）。Skills 路由则是更高层级的 Select——它根据用户意图，决定**激活哪个完整的能力模块**，每个模块自带独立的工具集、提示词模板和知识源。这对应 Claude Code 的 Skill 机制：输入 `/commit` 激活 Git 提交流程，输入 `/review-pr` 激活代码审查流程，两者使用完全不同的上下文配置。

&emsp;&emsp;为什么需要 Skills 路由？一个全功能 Agent 可能注册了 20+ 个工具，但每个任务只需要其中 3-5 个。如果把全部工具的 description 都塞进系统提示，不仅浪费 token，还会降低工具选择准确率——<font color=red>LLM 在 20 个候选中挑选正确工具的准确率，远低于在 5 个候选中挑选</font>。Skills 路由通过两个阶段解决这个问题：**LLM 意图分类**（根据 Skill 描述判断用户意图属于哪个 Skill）→ **上下文动态装配**（只加载选中 Skill 的工具集和系统提示）。下面我们用 LangChain 的 `@tool` 装饰器定义工具，用 `with_structured_output` 实现 LLM 分类器，完整演示这个流程。

&emsp;&emsp;首先定义三组 Skill 各自的专属工具。注意每个工具的 **docstring 就是工具描述**——LLM 在运行时通过阅读这些描述来决定调用哪个工具，描述质量直接影响工具选择准确率：

In [96]:
# === Select 策略：Skills 路由——步骤 1：定义各 Skill 的专属工具 ===
from langchain_core.tools import tool

# ---- Skill A：代码调试（3 个工具）----
@tool
def read_file(path: str) -> str:
    """读取指定路径的源代码文件，返回文件内容和行号"""
    return f"[模拟] 文件内容: {path}"

@tool
def search_codebase(query: str) -> str:
    """在代码仓库中搜索关键词，返回匹配的文件路径和行号"""
    return f"[模拟] 搜索结果: {query}"

@tool
def run_tests(test_path: str) -> str:
    """运行指定路径的测试文件，返回通过/失败结果"""
    return f"[模拟] 测试通过: {test_path}"

# ---- Skill B：部署运维（3 个工具）----
@tool
def deploy_staging(version: str) -> str:
    """将指定版本部署到 staging 环境，返回部署状态"""
    return f"[模拟] staging 部署完成: {version}"

@tool
def check_health(env: str) -> str:
    """检查指定环境的服务健康状态，返回 HTTP 状态码"""
    return f"[模拟] {env} 健康: 200 OK"

@tool
def rollback_deploy(env: str) -> str:
    """回滚指定环境到上一个稳定版本"""
    return f"[模拟] {env} 已回滚"

# ---- Skill C：记忆管理（2 个工具）----
@tool
def save_memory(content: str) -> str:
    """将用户偏好或项目决策保存到长期记忆库"""
    return f"[模拟] 已保存记忆: {content}"

@tool
def search_memories(query: str) -> str:
    """在长期记忆库中搜索相关记忆条目"""
    return f"[模拟] 记忆检索: {query}"

print(f"已定义 8 个工具，分属 3 个 Skill")
for t in [read_file, search_codebase, run_tests, deploy_staging, check_health, rollback_deploy, save_memory, search_memories]:
    print(f"  {t.name}: {t.description[:50]}")

已定义 8 个工具，分属 3 个 Skill
  read_file: 读取指定路径的源代码文件，返回文件内容和行号
  search_codebase: 在代码仓库中搜索关键词，返回匹配的文件路径和行号
  run_tests: 运行指定路径的测试文件，返回通过/失败结果
  deploy_staging: 将指定版本部署到 staging 环境，返回部署状态
  check_health: 检查指定环境的服务健康状态，返回 HTTP 状态码
  rollback_deploy: 回滚指定环境到上一个稳定版本
  save_memory: 将用户偏好或项目决策保存到长期记忆库
  search_memories: 在长期记忆库中搜索相关记忆条目


&emsp;&emsp;接下来构建 Skill 注册表和 LLM 意图分类器。注册表中每个 Skill 的 `description` 字段是分类器的唯一判断依据——<font color=red>description 写得越精确，分类准确率越高</font>，这与工具描述影响工具选择的原理完全一致：

In [97]:
# === Skills 路由——步骤 2：Skill 注册表 + LLM 意图分类器 ===
from pydantic import BaseModel, Field

# Skill 注册表：description 供 LLM 分类器阅读，tools 和 system_prompt 在分类后加载
skill_registry = {
    "code_debug": {
        "description": "代码调试与错误排查：处理报错信息、定位 bug 根因、运行测试验证修复",
        "tools": [read_file, search_codebase, run_tests],
        "system_prompt": "你是代码调试专家。先复现问题，再定位根因，最后验证修复。",
    },
    "deploy_ops": {
        "description": "部署与运维操作：发布新版本到各环境、健康检查、回滚、金丝雀发布流程",
        "tools": [deploy_staging, check_health, rollback_deploy],
        "system_prompt": "你是部署运维专家。严格执行金丝雀发布流程，每步都要健康检查。",
    },
    "memory_manage": {
        "description": "记忆管理：保存用户偏好和项目决策到长期记忆、检索历史记忆",
        "tools": [save_memory, search_memories],
        "system_prompt": "你是记忆管理助手。帮用户管理长期偏好和项目上下文。",
    },
}

# LLM 意图分类器：用 structured output 让 LLM 返回结构化的分类结果
class SkillIntent(BaseModel):
    """用户意图分类结果"""
    skill_id: str = Field(description="匹配的 Skill ID，必须是 code_debug / deploy_ops / memory_manage 之一")
    confidence: float = Field(description="分类置信度 0.0-1.0")

# 分类 prompt：列出所有 Skill 的 description，让 LLM 选择
skill_descriptions = "\n".join(
    f"- {sid}: {s['description']}" for sid, s in skill_registry.items()
)
classify_prompt = f"""根据用户消息，从以下 Skills 中选择最匹配的一个：

{skill_descriptions}

返回 skill_id 和 confidence。"""

classifier = llm.with_structured_output(SkillIntent)
print("Skill 注册表和 LLM 分类器已构建")
print(f"分类器输入的 Skill 描述：\n{skill_descriptions}")

Skill 注册表和 LLM 分类器已构建
分类器输入的 Skill 描述：
- code_debug: 代码调试与错误排查：处理报错信息、定位 bug 根因、运行测试验证修复
- deploy_ops: 部署与运维操作：发布新版本到各环境、健康检查、回滚、金丝雀发布流程
- memory_manage: 记忆管理：保存用户偏好和项目决策到长期记忆、检索历史记忆


&emsp;&emsp;一切就绪，下面用三条不同意图的用户消息测试完整的 Skills 路由流程：LLM 分类 → 装配选中 Skill 的工具集 → 对比全量加载 vs Skill 子集的 Token 开销：

In [98]:
# === Skills 路由——步骤 3：端到端演示（分类 → 装配 → Token 对比）===

test_messages = [
    "代码报错了，AgentManager 初始化失败",
    "把新版本部署到 staging 环境",
    "帮我记住：以后 code review 要检查 token 消耗",
]

# 基线：如果不做 Skills 路由，Agent 需要加载全部 8 个工具的描述
all_tools = [t for s in skill_registry.values() for t in s["tools"]]
all_tools_desc = "\n".join(f"{t.name}: {t.description}" for t in all_tools)
all_tokens = count_tokens(all_tools_desc)

for msg in test_messages:
    # Stage 1：LLM 意图分类
    intent = classifier.invoke(f"{classify_prompt}\n\n用户消息: {msg}")
    skill = skill_registry[intent.skill_id]

    # Stage 2：只装配选中 Skill 的工具（Select 的核心价值）
    skill_tools_desc = "\n".join(f"{t.name}: {t.description}" for t in skill["tools"])
    skill_tokens = count_tokens(skill_tools_desc)

    print(f'用户: "{msg}"')
    print(f'  → LLM 分类: {intent.skill_id} (confidence={intent.confidence})')
    print(f'    装配工具: {[t.name for t in skill["tools"]]}')
    print(f'    系统提示: "{skill["system_prompt"][:40]}..."')
    print(f'    Token: 全量 {all_tokens} → Skill {skill_tokens}（节省 {(1-skill_tokens/all_tokens)*100:.0f}%）')
    print()

# ---- 真正的 Agent 构造：用最后一条消息演示上下文装配 ----
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

last_msg = test_messages[-1]  # "帮我记住：以后 code review 要检查 token 消耗"
intent = classifier.invoke(f"{classify_prompt}\n\n用户消息: {last_msg}")
selected_skill = skill_registry[intent.skill_id]

# 构造 Agent：只传入选中 Skill 的工具和系统提示
prompt = ChatPromptTemplate.from_messages([
    ("system", selected_skill["system_prompt"]),  # ← system_prompt 真正传入
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_agent(
    llm,
    tools=selected_skill["tools"],  # ← tools 真正传入上下文
    prompt=prompt,
)

print(f"\n=== 真正的 Agent 调用（用户: \"{last_msg}\"）===")
print(f"选中 Skill: {intent.skill_id}")
print(f"Agent 可见工具: {[t.name for t in selected_skill['tools']]}")
print(f"Agent 系统提示: {selected_skill['system_prompt']}")

# 调用 Agent
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=selected_skill["tools"], verbose=False)
result = agent_executor.invoke({"input": last_msg})

print(f"\nAgent 响应: {result['output']}")
print(f"\n✓ 验证：Agent 只能调用 {intent.skill_id} Skill 的 {len(selected_skill['tools'])} 个工具")
print(f"  （而非全部 {len(all_tools)} 个工具）")

用户: "代码报错了，AgentManager 初始化失败"
  → LLM 分类: code_debug (confidence=0.95)
    装配工具: ['read_file', 'search_codebase', 'run_tests']
    系统提示: "你是代码调试专家。先复现问题，再定位根因，最后验证修复。..."
    Token: 全量 139 → Skill 54（节省 61%）

用户: "把新版本部署到 staging 环境"
  → LLM 分类: deploy_ops (confidence=0.95)
    装配工具: ['deploy_staging', 'check_health', 'rollback_deploy']
    系统提示: "你是部署运维专家。严格执行金丝雀发布流程，每步都要健康检查。..."
    Token: 全量 139 → Skill 51（节省 63%）

用户: "帮我记住：以后 code review 要检查 token 消耗"
  → LLM 分类: memory_manage (confidence=0.95)
    装配工具: ['save_memory', 'search_memories']
    系统提示: "你是记忆管理助手。帮用户管理长期偏好和项目上下文。..."
    Token: 全量 139 → Skill 32（节省 77%）

=== 分类后的 Agent 构造（伪代码）===
# 只加载 2 个工具，而非全部 8 个


&emsp;&emsp;三条测试消息被准确路由到各自的 Skill，Token 节省 60%-76%。Skills 路由是 Select 策略的最高抽象层级：它不是在一个数据源内做筛选，而是**切换整个上下文配置**——不同的 Skill 加载不同的工具集、使用不同的系统提示。这意味着一次 Skill 路由决策，实际上同时触发了前面三种 Select 的组合：查哪些记忆（3.1.2.1）、搜哪些文件（3.1.2.2）、检索哪些知识（3.1.2.3），并在此基础上决定加载哪些工具、使用哪套系统提示。

&emsp;&emsp;值得注意的是，`description` 字段的质量直接决定分类准确率。无论是 Skill 级别的描述（决定意图路由准确率），还是工具级别的 docstring（决定工具选择准确率），原理完全一致。<font color=red>在 LLM 驱动的系统中，自然语言描述就是接口定义</font>。

#### 3.1.3 Compress 策略：保留决策链，丢弃执行细节

&emsp;&emsp;Compress 是最丰富的策略家族，包含四种主要技术：**Compaction**（整体压缩重启）、**硬截断与 LLM 摘要压缩**（第2章已演示）、**工具结果清除**（Tool Result Clearing）、**观察遮蔽**（Observation Masking）。它们的共同原理是：<font color=red>Agent 的推理质量主要取决于"做了什么决策"和"为什么这样做"，而不是"工具返回了什么原始数据"</font>。

> &emsp;**提示**：Compress 包含四种子技术，本节内容较其他策略更详细，这是因为压缩策略在实际工程中使用频率最高、变体最多。最后我们会用同一场景做三种技术的并排对比，帮你建立清晰的选型直觉。

##### 3.1.3.1 Compaction：对话压缩重启

&emsp;&emsp;在具体的压缩技术中，有一个概念需要特别区分——**Compaction**（对话压缩重启）。
它和普通的 LLM 摘要不同：Compaction 是当上下文接近窗口极限时，将**整个对话历史**发送给 LLM 进行高保真压缩，
然后用压缩摘要**替换**原始历史，开启新的上下文窗口继续工作。
Claude Code 默认在 token 使用超过窗口的 **95%** 时自动触发 Compaction
（标准模型 200K 窗口约 190K tokens，1M 扩展上下文模型约 950K tokens），
保留架构决策、未解决 bug、关键实现细节，丢弃冗余工具输出和已解决的问题。
这个 95% 的触发阈值可以通过环境变量 `CLAUDE_AUTOCOMPACT_PCT_OVERRIDE` 调整（取值范围 1-100），
例如设置为 50 则在窗口使用过半时就提前触发压缩。


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409123042488.png" width=60%></div>

> **注意**：以下为 Anthropic SDK Compaction 功能的示意性 API 设计，实际 API 可能有所不同，请以官方文档为准。

In [ ]:
# Anthropic SDK 的 Compaction 配置示例（伪代码，展示 API 设计思路）
runner = client.beta.messages.tool_runner(
    model="claude-sonnet-4-6",
    tools=tools,
    messages=messages,
    compaction_control={
        "enabled": True,  # 启用自动压缩
        "context_token_threshold": 5000,  # 触发阈值：当上下文超过5000 token时启动压缩
    },
)
# 效果：150K tokens 的对话 → Compaction 后降至 ~79K tokens，质量不降
# 压缩机制：保留架构决策、未解决bug、关键实现细节，丢弃冗余工具输出和已解决问题

&emsp;&emsp;理解了 Compaction 的概念设计后，我们用 LangChain 的消息类型来实现一个**可运行的 Compaction**。
注意它和前面短期记忆压缩的关键区别：短期记忆是「压缩早期 + 保留最近」，
而 Compaction 是<font color=red>全量压缩</font>——将**整个对话历史**（包括工具调用、代码修改、测试结果）
压缩为一条结构化的 `SystemMessage`，然后用这条摘要**替换**全部原始消息，开启新的上下文窗口继续工作：

In [45]:
# === Compaction 实战：用 LangChain 消息类型实现对话压缩重启 ===
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ---- 步骤 1：构造一段工具密集型的 Agent 对话（模拟 Claude Code 场景）----
# 注意：Compaction 场景包含 user / assistant / tool 三种角色，
# 而短期记忆场景只有 user / assistant 两种
messages = [
    HumanMessage(content="帮我重构 auth 模块，支持 OAuth2 认证流程"),
    AIMessage(content="好的，先读取当前代码结构。\n[调用 read_file('auth.py')]"),
    AIMessage(content="[tool result] auth.py: 340行，含 BasicAuth 类，使用 session-based 认证"),
    AIMessage(content="分析完成。当前使用 BasicAuth，需新增 OAuth2Provider。\n"
             "决策：采用 refresh_token 流程，token 存储于 Redis，增加速率限制。\n"
             "[调用 edit_file('auth.py')]"),
    AIMessage(content="[tool result] auth.py 已修改：+OAuth2Provider 类，+token 刷新逻辑，共新增 180 行"),
    HumanMessage(content="再加上单元测试"),
    AIMessage(content="编写 12 个测试用例覆盖 OAuth2 核心流程。\n[调用 write_file('test_auth.py')]"),
    AIMessage(content="[tool result] test_auth.py 创建成功，含 12 个测试用例"),
    HumanMessage(content="运行测试看看结果"),
    AIMessage(content="[调用 run_tests('test_auth.py')]"),
    AIMessage(content="[tool result] 11/12 通过，1 个 token 过期边界用例不稳定（偶发 timing issue）"),
    AIMessage(content="测试结果：11/12 通过。不稳定用例是 test_token_expiry_boundary，"
             "原因是 timing 精度问题，下一步需要用 mock time 修复。"),
]

total_tokens = sum(count_tokens(m.content) for m in messages)
print(f"=== Compaction 前 ===")
print(f"消息数: {len(messages)} 条（含 user/assistant/tool 三种角色）")
print(f"总 tokens: {total_tokens}")

# ---- 步骤 2：检测是否达到阈值，触发全量 Compaction ----
MAX_CONTEXT_TOKENS = 150   # 演示用小阈值（生产环境 Claude Code 用 200K * 95%）

if total_tokens > MAX_CONTEXT_TOKENS * 0.95:
    # 关键区别：Compaction 压缩 **全部** 消息，不保留最近消息原文
    full_text = "\n".join(f"[{m.type}] {m.content}" for m in messages)

    # 结构化压缩提示词：要求 LLM 输出 Claude Code 风格的 Compaction Summary
    compress_prompt = f"""请将以下 Agent 对话历史压缩为结构化摘要。

    对话内容：
    {full_text}

    输出要求（严格按以下五个字段输出，每个字段一行）：
    Task: 一句话描述用户的核心任务
    Files modified: 列出所有被修改或创建的文件及关键变更
    Decisions: 列出做出的关键技术决策
    Status: 当前进度和测试结果
    Next: 下一步待做的事项"""

    summary = llm.invoke(compress_prompt).content

    # 关键操作：用 SystemMessage(结构化摘要) 替换 **整个** 消息列表
    compacted = [SystemMessage(content=f"[Compaction Summary]\n{summary}")]
else:
    compacted = messages
    summary = ""
    print("未达阈值，无需 Compaction")

# ---- 步骤 3：展示压缩结果 ----
compacted_tokens = sum(count_tokens(m.content) for m in compacted)
print(f"\n=== Compaction 后 ===")
print(f"消息数: {len(messages)} → {len(compacted)}"
      f"（全部 {len(messages)} 条被压缩为 1 条 SystemMessage）")
print(f"Tokens: {total_tokens} → {compacted_tokens}"
      f"（压缩率 {(1 - compacted_tokens/total_tokens)*100:.0f}%）")
print(f"\n--- Compaction Summary ---")
print(compacted[0].content)

# ---- 步骤 4：验证——压缩后 LLM 仍能回答早期问题 ----
verify_msgs = compacted + [HumanMessage(content="我们之前定的认证方案是什么？一句话回复")]
answer = llm.invoke(verify_msgs).content
print(f"\n=== 验证：压缩后仍能回答早期决策 ===")
print(f"  Q: 我们之前定的认证方案是什么？")
print(f"  A: {answer}")


=== Compaction 前 ===
消息数: 12 条（含 user/assistant/tool 三种角色）
总 tokens: 246

=== Compaction 后 ===
消息数: 12 → 1（全部 12 条被压缩为 1 条 SystemMessage）
Tokens: 246 → 130（压缩率 47%）

--- Compaction Summary ---
[Compaction Summary]
Task: 重构 auth 模块以支持 OAuth2 认证流程并添加单元测试。
Files modified: auth.py (新增 OAuth2Provider 类和 token 刷新逻辑，共180行)，test_auth.py (创建，包含12个测试用例)。
Decisions: 采用 refresh_token 流程，将 token 存储于 Redis，并增加速率限制。
Status: 代码修改和测试用例编写已完成，测试运行结果11/12通过，一个边界用例因 timing 精度问题不稳定。
Next: 使用 mock time 修复不稳定的测试用例 test_token_expiry_boundary。

=== 验证：压缩后仍能回答早期决策 ===
  Q: 我们之前定的认证方案是什么？
  A: 我们之前定的认证方案是：**基于 OAuth2 的授权码流程，使用 JWT 作为访问令牌，配合刷新令牌和 Redis 存储来实现安全的用户认证和会话管理。**


&emsp;&emsp;全部 12 条消息被压缩为 1 条结构化的 `SystemMessage`
（注意不是短期记忆的「1 条摘要 + N 条最近原文」，而是<font color=red>全量替换</font>）。
压缩后的 Compaction Summary 包含五个结构化字段——
**Task**（任务）、**Files modified**（文件变更）、**Decisions**（技术决策）、
**Status**（当前状态）、**Next**（下一步），
完整保留了 Agent 继续工作所需的全部上下文。验证环节证明：
<font color=red>即使原始对话全部丢弃，LLM 仍能从结构化摘要中提取关键决策</font>。

&emsp;&emsp;对比 3.1.1 节的短期 Write（SessionManager 压缩），两者有三个本质区别：

- **压缩范围不同**：短期记忆只压缩早期消息、保留最近原文；Compaction 压缩全部消息、不保留原文
- **输出格式不同**：短期记忆输出一段非结构化摘要；Compaction 输出结构化的五字段 Summary
- **存储目标不同**：短期记忆将摘要写入 `sessions/*.json` 文件持久保存（「记住」）；
Compaction 将摘要注入当前消息列表的 `SystemMessage`，仅在当前会话内生效，不落盘（「续航」）

&emsp;&emsp;Compaction 的价值在于让 Agent 能处理**理论上无限长**的任务——没有它，任何 Agent 在 20-30 轮工具密集的对话后就会因窗口溢出而失败。

&emsp;&emsp;Anthropic SDK 的 Compaction 是概念层面的设计。在实际工程中，LangChain 提供了两种可直接使用的压缩实现——**硬截断**（`trim_messages`，零成本）和 **LLM 摘要压缩**（`SummarizationMiddleware`，有损但高保真）。我们分别演示这两种机制。

##### 3.1.3.2 硬截断：trim_messages

&emsp;&emsp;`trim_messages` 是最直接的压缩方式——不调用 LLM，直接按 token 上限从后往前保留消息。优点是零成本零延迟，缺点是早期信息直接丢弃：

In [28]:
# === LangChain trim_messages：硬截断（零 LLM 调用）===
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

# 1. 构造 12 轮对话历史（包含关键技术决策）
messages = []
decisions = ["选 PostgreSQL", "用 Redis 缓存", "JWT 认证", "Docker 部署",
             "pool_size=50", "GitHub Actions CI", "ELK 日志", "Sentry 监控",
             "slowapi 限流", "React 前端", "Zustand 状态管理", "pytest 测试"]
for i, decision in enumerate(decisions):
    messages.append(HumanMessage(content=f"第{i+1}个技术决策是什么？"))
    messages.append(AIMessage(content=f"决策{i+1}：{decision}。理由是团队熟悉度高、社区生态成熟、与现有架构兼容。"))

original_tokens = sum(count_tokens(m.content) for m in messages)
print(f"原始消息: {len(messages)} 条（{len(messages)//2} 轮），约 {original_tokens} tokens")

# 2. 硬截断：只保留最近 N 条消息
# token_counter=len 让 max_tokens 以「消息条数」为单位（LangChain 设计）
trimmed = trim_messages(
    messages,
    strategy="last",       # 从后往前保留
    max_tokens=8,          # 保留最近 8 条消息（4 轮）
    token_counter=len,     # len = 按消息条数计（非 token 数）
    start_on="human",      # 确保从 human 消息开始
)

trimmed_tokens = sum(count_tokens(m.content) for m in trimmed)
kept_rounds = len(trimmed) // 2
lost_rounds = len(decisions) - kept_rounds

lost_list = ", ".join(decisions[:lost_rounds])                                                                               
kept_list = ", ".join(decisions[lost_rounds:])                                                                                             
print(f"\n丢失的早期决策({lost_rounds}个): {lost_list}")                                                                                   
print(f"保留的近期决策({kept_rounds}个): {kept_list}")                                                                                     
print(f"\n 硬截断的代价：前 {lost_rounds} 个关键决策（PostgreSQL、Redis 等）被直接丢弃，不可恢复") 


原始消息: 24 条（12 轮），约 348 tokens

丢失的早期决策(8个): 选 PostgreSQL, 用 Redis 缓存, JWT 认证, Docker 部署, pool_size=50, GitHub Actions CI, ELK 日志, Sentry 监控
保留的近期决策(4个): slowapi 限流, React 前端, Zustand 状态管理, pytest 测试

 硬截断的代价：前 8 个关键决策（PostgreSQL、Redis 等）被直接丢弃，不可恢复


##### 3.1.3.3 LLM 摘要压缩：SummarizationMiddleware

&emsp;&emsp;`SummarizationMiddleware` 是 LangChain 版的 Compaction——超过 token 阈值时，自动调用 LLM 将早期消息压缩为摘要，保留最近 N 条原文：

In [29]:
# === LangChain SummarizationMiddleware：LLM 摘要压缩 ===                                                                                  
# SummarizationMiddleware 是 LangChain Agent 内置中间件，                                                                                  
# 当对话 token 超过阈值时自动调用 LLM 压缩早期消息为摘要。                                                                                 
# 参考：https://python.langchain.com/docs/concepts/short-term-memory/                                                                      
                                                                                                                                            
from langchain.agents import create_agent                                                                                                  
from langchain.agents.middleware import SummarizationMiddleware                                                                            
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver                                                                                      
                                                                                                                                        
checkpointer = InMemorySaver()

# 创建带摘要中间件的 Agent
agent = create_agent(
    model=llm,               # 复用前面初始化的 DeepSeek 模型
    tools=[],                # 纯对话演示，不需要工具
    middleware=[                                                                                                             
        SummarizationMiddleware(                                                                                                           
            model=llm,                   # 同模型做摘要（生产环境可用更便宜的小模型）                                                      
            trigger=("tokens", 300),     # 触发阈值（演示用小值，生产通常 4000-8000）                                                      
            keep=("messages", 6),        # 保留最近 6 条原文不压缩                                                                         
        )                                                                                                                                  
    ],                                                                                                                                     
    checkpointer=checkpointer,          # 状态持久化（摘要需要跨轮次保存）                                                                 
)                                                                                                                                          
                                                                                                                                            
# 用相同的 12 轮技术决策对话测试                                                                                                       
config = {"configurable": {"thread_id": "compress-demo"}}
decisions = ["选 PostgreSQL", "用 Redis 缓存", "JWT 认证", "Docker 部署",                                                    
            "pool_size=50", "GitHub Actions CI", "ELK 日志", "Sentry 监控",                                                               
            "slowapi 限流", "React 前端", "Zustand 状态管理", "pytest 测试"]                                                              
                                                                                                                                            
print("=== 逐轮发送技术决策，观察 SummarizationMiddleware 压缩过程 ===\n")                                                                 
for i, decision in enumerate(decisions):                                                                                                   
    response = agent.invoke(                                                                                                               
        {"messages": [HumanMessage(content=f"记住第{i+1}个技术决策：{decision}。理由：团队熟悉度高、社区成熟。")]},                                                
        config                                                                                                                             
    )                                                                                                                                      
    msgs = response["messages"]                                                                                                            
    msg_count = len(msgs)                                                                                                                  
    expected = (i + 1) * 2  # 无压缩时的预期消息数                                                                                         
                                                                                                                                            
    if msg_count < expected:                                                                                                           
        # 压缩已触发，第一条消息是摘要                                                                                                     
        summary_preview = msgs[0].content[:200].replace('\n', ' ')                                                                         
        latest_reply = msgs[-1].content[:100].replace('\n', ' ')                                                                           
        print(f"轮次 {i+1:2d} | 消息数: {msg_count:3d}（无压缩应为 {expected:2d}）← 压缩中")                                               
        print(f"         [摘要] {summary_preview}...")                                                                                   
        print(f"         [最新] {latest_reply}...")                                                                                      
        print()                                                                                                                            
    else:                                                                                                                                  
        print(f"轮次 {i+1:2d} | 消息数: {msg_count:3d}（无压缩应为 {expected:2d}）")                                                       
                                                                                                                                            
# 最终验证                                                                                                                                 
print("=== 验证：早期决策是否被摘要保留 ===")                                                                                              
final = agent.invoke({"messages": [HumanMessage(content="请列出之前所有的技术决策")]}, config)                                                                     
print(final["messages"][-1].content[:500]) 


=== 逐轮发送技术决策，观察 SummarizationMiddleware 压缩过程 ===

轮次  1 | 消息数:   2（无压缩应为  2）
轮次  2 | 消息数:   4（无压缩应为  4）
轮次  3 | 消息数:   6（无压缩应为  6）
轮次  4 | 消息数:   8（无压缩应为  8）
轮次  5 | 消息数:   8（无压缩应为 10）← 压缩中
         [摘要] Here is a summary of the conversation to date:  ## SESSION INTENT Establish and document a series of technical decisions for a project, starting with database and caching selections.  ## SUMMARY Two f...
         [最新] 好的，已记录第五个技术决策。  **决策记录 #5**  *   **决策内容**：将数据库连接池的默认大小设置为 **`pool_size=50`**。 *   **主要理由**：     1.  ...

轮次  6 | 消息数:   8（无压缩应为 12）← 压缩中
         [摘要] Here is a summary of the conversation to date:  ## SESSION INTENT Establish and document a series of technical decisions for a project, starting with database and caching selections.  ## SUMMARY Three...
         [最新] 好的，已记录第六个技术决策。  **决策记录 #6**  *   **决策内容**：采用 **GitHub Actions** 作为持续集成工具。 *   **主要理由**：     1.  **团队...

轮次  7 | 消息数:   8（无压缩应为 14）← 压缩中
         [摘要] Here is a summary of the conversation to date:  ## SESSION

&emsp;&emsp;两种方式的核心权衡一目了然：`trim_messages` **快但丢信息**（适合对早期历史不敏感的场景），`SummarizationMiddleware` **慢但保信息**（适合长任务中每个决策都可能被回溯的场景）。生产环境通常组合使用——先用 Summarization 压缩到安全线，再用 trim 做兜底截断。

##### 3.1.3.4 工具结果清除：Tool Result Clearing

&emsp;&emsp;在引入更复杂的压缩技术之前，有一种最轻量的做法值得首先考虑——**工具结果清除**（Tool Result Clearing）。它的机制是：工具调用完成后，用 LLM 将 `ToolMessage` 的冗长原始输出压缩为一行结论性摘要，保留关键发现、丢弃原始细节。

&emsp;&emsp;<font color=red>工具结果清除与后面要讲的观察遮蔽有本质区别：清除是"摘要替换"（保留工具输出的核心结论），遮蔽是"完全丢弃"（只留占位符，靠 Reasoning 链携带信息）。</font>清除的成立前提是：一行高质量摘要足以替代原始输出中 90% 的信息价值，就像你看完一份 100 页的报告后写了一句话总结——后续讨论围绕总结进行，100 页原文就可以安全丢弃。

&emsp;&emsp;我们用一个具体场景来感受：Agent 分析项目依赖时，工具返回了完整的 `requirements.txt` 和 `docker-compose.yml`（大量原始文本），我们用 LLM 将每条工具输出压缩为一行摘要，然后对比压缩前后的 token 消耗和回答质量：

In [30]:
# === 工具结果清除（Tool Result Clearing）===
# 核心机制：工具调用完成后，将 ToolMessage 的原始返回值替换为占位符，
# 仅保留"调用了什么工具"的事实，依赖 Agent 的 AIMessage 携带关键结论。
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import time

# 模拟 Agent 分析项目依赖的完整消息链（2 次工具调用）
original_messages = [
    HumanMessage(content="这个项目用了哪些核心依赖？各自什么版本？有没有已知的兼容性问题？"),

    # Step 1: Agent 决定查看 requirements.txt
    AIMessage(
        content="我先查看项目的依赖配置文件，确认核心依赖和版本信息。",
        tool_calls=[{"name": "read_file", "args": {"path": "requirements.txt"}, "id": "call_1"}]
    ),
    # 工具返回完整文件内容（大量 token）
    ToolMessage(content="""# Core dependencies
    fastapi==0.104.1
    uvicorn[standard]==0.24.0
    pydantic==2.5.0
    sqlalchemy==2.0.23
    alembic==1.13.0
    asyncpg==0.29.0

    # Cache & Message Queue
    redis==5.0.1
    celery==5.3.6
    kombu==5.3.4

    # AI/ML
    langchain==0.1.0
    langchain-openai==0.0.2
    openai==1.6.1
    tiktoken==0.5.2
    chromadb==0.4.22
    sentence-transformers==2.2.2

    # Monitoring
    prometheus-client==0.19.0
    sentry-sdk[fastapi]==1.39.1
    structlog==23.2.0

    # Testing
    pytest==7.4.3
    pytest-asyncio==0.23.2
    httpx==0.25.2
    factory-boy==3.3.0""", tool_call_id="call_1"),

    # Step 2: Agent 进一步检查 docker-compose
    AIMessage(
        content="requirements.txt 显示核心依赖为 FastAPI 0.104 + SQLAlchemy 2.0 + Redis 5.0 + LangChain 0.1.0。"
                "接下来检查 docker-compose 确认基础设施版本。",
        tool_calls=[{"name": "read_file", "args": {"path": "docker-compose.yml"}, "id": "call_2"}]
    ),
    ToolMessage(content="""version: '3.8'
    services:
      app:
        build: .
        ports: ["8000:8000"]
        depends_on: [db, redis, chromadb]
        environment:
          - DATABASE_URL=postgresql+asyncpg://user:pass@db:5432/app
          - REDIS_URL=redis://redis:6379/0
          - CHROMADB_HOST=chromadb
      db:
        image: postgres:16.1
        volumes: [pgdata:/var/lib/postgresql/data]
        environment:
          POSTGRES_DB: app
          POSTGRES_USER: user
          POSTGRES_PASSWORD: pass
      redis:
        image: redis:7.2-alpine
        command: redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru
      chromadb:
        image: chromadb/chroma:0.4.22
        volumes: [chromadata:/chroma/chroma]
    volumes:
      pgdata:
      chromadata:""", tool_call_id="call_2"),

    # Agent 最终结论（已浓缩所有关键信息）
    AIMessage(content=(
        "项目核心依赖分析完成：\n"
        "1. **Web 框架**：FastAPI 0.104.1 + Uvicorn 0.24.0 + Pydantic 2.5.0\n"
        "2. **数据库**：PostgreSQL 16.1 + SQLAlchemy 2.0.23 + asyncpg 0.29.0\n"
        "3. **缓存/队列**：Redis 7.2 + Celery 5.3.6\n"
        "4. **AI/ML**：LangChain 0.1.0 + OpenAI 1.6.1 + ChromaDB 0.4.22\n\n"
        "兼容性风险：LangChain 0.1.0 是早期版本，与 langchain-openai 0.0.2 "
        "的 API 可能在后续升级中出现 breaking change。建议锁定版本或关注 changelog。"
    )),
]

print(f"消息链构造完成：{len(original_messages)} 条消息（{len(original_messages)//2} 轮工具调用 + 最终结论）")

消息链构造完成：6 条消息（3 轮工具调用 + 最终结论）


&emsp;&emsp;消息链构造完成后，我们先看一下各类型消息的 token 占比——这是理解工具结果清除价值的关键：`ToolMessage`（工具返回值）在总 token 中占多大比重？

In [31]:
# --- 展示原始消息链中各类型的 token 占比 ---
print("=== 原始消息链：各消息类型 token 占比 ===\n")
type_tokens = {"HumanMessage": 0, "AIMessage": 0, "ToolMessage": 0}
for msg in original_messages:
    t = type(msg).__name__
    tokens = count_tokens(str(msg.content))
    type_tokens[t] += tokens

total = sum(type_tokens.values())
for t, tok in type_tokens.items():
    bar = "█" * int(tok / total * 30)
    print(f"  {t:15s} {tok:4d} tokens ({tok/total*100:4.1f}%) {bar}")
print(f"  {'总计':15s} {total:4d} tokens")

=== 原始消息链：各消息类型 token 占比 ===

  HumanMessage      17 tokens ( 2.3%) 
  AIMessage        241 tokens (32.5%) █████████
  ToolMessage      483 tokens (65.2%) ███████████████████
  总计               741 tokens


&emsp;&emsp;可以看到 `ToolMessage` 占了总 token 的大头。现在我们定义清除函数——与后面的观察遮蔽不同，工具结果清除不是简单地用占位符替换，而是<font color=red>用 LLM 将每条工具输出压缩为一行结论性摘要</font>，保留关键发现、丢弃原始细节：

In [32]:
# --- 定义工具结果清除函数并执行 ---
def clear_tool_results(messages):
    """工具调用完成后，将冗长的原始返回值压缩为一行结论性摘要"""
    cleared = []
    for msg in messages:
        if isinstance(msg, ToolMessage):
            # 用 LLM 将工具输出压缩为一行摘要（只压缩单条输出，比整段对话压缩便宜得多）
            summary_resp = llm.invoke(
                f"用一句话（不超过80字）总结以下工具输出的关键发现：\n\n{msg.content}"
            )
            cleared.append(ToolMessage(
                content=f"[摘要] {summary_resp.content.strip()}",
                tool_call_id=msg.tool_call_id
            ))
        else:
            cleared.append(msg)
    return cleared

cleared_messages = clear_tool_results(original_messages)

# 展示摘要效果
print("=== 工具结果清除：原始输出 → 一行摘要 ===\n")
tool_idx = 0
for orig, clr in zip(original_messages, cleared_messages):
    if isinstance(orig, ToolMessage):
        tool_idx += 1
        orig_tokens = count_tokens(orig.content)
        clr_tokens = count_tokens(clr.content)
        print(f"ToolMessage #{tool_idx}:")
        print(f"  原始: {orig_tokens} tokens | {orig.content[:80]}...")
        print(f"  摘要: {clr_tokens} tokens | {clr.content}")
        print(f"  压缩率: {(1 - clr_tokens/orig_tokens)*100:.0f}%\n")

# Token 对比
cleared_tokens = sum(count_tokens(str(m.content)) for m in cleared_messages)
saved_pct = (1 - cleared_tokens / total) * 100
print(f"=== 整体 Token 对比 ===")
print(f"清除前: {total} tokens")
print(f"清除后: {cleared_tokens} tokens（节省 {saved_pct:.0f}%）")

=== 工具结果清除：原始输出 → 一行摘要 ===

ToolMessage #1:
  原始: 260 tokens | # Core dependencies
    fastapi==0.104.1
    uvicorn[standard]==0.24.0
    pydan...
  摘要: 32 tokens | [摘要] 这是一个基于FastAPI的AI应用后端，集成了异步数据库、缓存队列、LangChain智能处理及全链路监控测试能力。
  压缩率: 88%

ToolMessage #2:
  原始: 223 tokens | version: '3.8'
    services:
      app:
        build: .
        ports: ["8000:8...
  摘要: 44 tokens | [摘要] 该Docker Compose配置定义了一个包含应用、PostgreSQL数据库、Redis缓存和ChromaDB向量数据库的四服务微服务架构，通过容器网络互联并配置了数据持久化。
  压缩率: 80%

=== 整体 Token 对比 ===
清除前: 741 tokens
清除后: 334 tokens（节省 55%）


&emsp;&emsp;每条工具输出从几十上百 tokens 压缩为一行摘要，整体节省可观。但关键问题是：只保留摘要后，LLM 还能正确回答关于项目依赖的追问吗？我们对比完整上下文和摘要替换后的回答质量：

In [33]:
# --- LLM 回答质量对比：完整保留 vs 工具结果清除 ---
question = "根据以上信息，这个项目的核心依赖有哪些？有什么兼容性风险？"
print("=== LLM 回答质量对比 ===")

for label, msgs in [("完整保留", original_messages), ("工具结果清除", cleared_messages)]:
    ctx = "\n".join(f"[{type(m).__name__}] {str(m.content)[:500]}" for m in msgs)
    start = time.time()
    resp = llm.invoke(f"以下是 Agent 分析项目依赖的过程：\n{ctx}\n\n{question}")
    elapsed = time.time() - start
    print(f"\n--- {label}（{elapsed:.1f}s）---")
    print(resp.content[:400])

print(f"\n关键发现：ToolMessage 占总 token 的 {type_tokens['ToolMessage']/total*100:.0f}%，")
print(f"   清除后节省 {saved_pct:.0f}%。由于 Agent 最后的 AIMessage 已浓缩了")
print(f"   所有关键信息（依赖名称、版本、兼容性风险），LLM 仍能完整回答。")
print(f"   但如果追问「requirements.txt 里还有哪些测试依赖？」，清除后就答不出了。")

=== LLM 回答质量对比 ===

--- 完整保留（17.3s）---
根据分析结果，这个项目的核心依赖及版本如下：

## **核心依赖分类**

### **1. Web框架层**
- **FastAPI 0.104.1** - 现代异步Web框架
- **Uvicorn 0.24.0** - ASGI服务器
- **Pydantic 2.5.0** - 数据验证和序列化

### **2. 数据库层**
- **PostgreSQL 16.1** - 主数据库（通过Docker镜像）
- **SQLAlchemy 2.0.23** - ORM
- **asyncpg 0.29.0** - PostgreSQL异步驱动
- **Alembic 1.13.0** - 数据库迁移工具

### **3. 缓存与消息队列**
- **Redis 7.2** - 缓存和消息代理（通过Docker）
- **Celery 5.3.6** - 分布式任务队列
- **

--- 工具结果清除（10.8s）---
根据 Agent 的分析，该项目核心依赖及版本如下：

---

### **核心依赖清单**
1. **Web 框架层**
   - FastAPI 0.104.1
   - Uvicorn 0.24.0
   - Pydantic 2.5.0

2. **数据库层**
   - PostgreSQL 16.1（基础设施）
   - SQLAlchemy 2.0.23（ORM）
   - asyncpg 0.29.0（异步驱动）

3. **缓存与队列**
   - Redis 7.2（基础设施）
   - Celery 5.3.6（异步任务队列）

4. **AI/ML 与向量处理**
   - LangChain 0.1.0（AI 应用框架）
   - OpenAI 1.6.1（大模型接口）
   - ChromaDB 0.4.22（向量数据库）

---

### **兼容性风险*

关键发现：ToolMessage 占总 token 的 65%，
   清除后节省 55%。由于 Agent 最后的 AIMessage 已浓缩了
   所有关键信息（依赖名称、版本、兼容性风险），LLM 仍能完整回答。
   但如果追问「requirements.txt

&emsp;&emsp;从对比可以看出，工具结果清除的核心价值在于：用极少的 LLM 调用成本（只压缩单条工具输出，远比压缩整段对话便宜），换取了大幅的 token 节省，且摘要保留了关键结论，LLM 仍能正确回答。

&emsp;&emsp;但工具结果清除有一个明确的局限：<font color=red>摘要是有损的，如果后续追问需要原始细节（比如"requirements.txt 里还有哪些测试依赖？"），摘要中没有保留的信息就无法回答了</font>。这就是它被定位为"首选但有条件"策略的原因——适合工具输出量大、但后续只需要结论的场景。对于需要反复回溯工具原始输出的长链任务，我们需要更精细的策略——接下来要讲的观察遮蔽（完全丢弃工具输出，靠 Reasoning 链保留决策信息）。

##### 3.1.3.5 观察遮蔽：Observation Masking

&emsp;&emsp;观察遮蔽（Observation Masking）比工具结果清除更精细。它的机制来自一个关键洞察：Agent 每一步交互的上下文可以分解为三个部分——**Action**（Agent 的动作指令）、**Reasoning**（推理过程）、**Observation**（环境返回的观察结果）。Observation Masking 的做法是保留 Action 和 Reasoning，将旧的 Observation 替换为占位符。JetBrains Research 在 [PIPer: Pipelined Prompting for LLM-based SE Agents](https://github.com/JetBrains-Research/PIPer)（NeurIPS 2025 DL4C workshop）中证明：这个看似粗暴的操作在 SE Agent 场景下，成本降低 52%、准确率反而提升 2.6%——因为去除了过时的环境噪声，模型的注意力更加集中。

&emsp;&emsp;我们用代码来直观感受 Observation Masking 的原理——将一次工具调用拆解为 Action（做了什么）、Reasoning（为什么这样做）、Observation（工具返回了什么），然后对比遮蔽前后的 token 和回答质量：

In [34]:
# === Observation Masking：保留 Action + Reasoning，遮蔽 Observation ===
# 核心原理（JetBrains, NeurIPS 2025）：
# Agent 每步的 Reasoning 已自然浓缩了上一步 Observation 的关键发现，
# 因此遮蔽冗长的原始 Observation 不会丢失决策信息。

# 模拟 3 步工具调用链（Agent 调试数据库连接超时问题）
tool_chain = [
    {
        "action": "search_codebase('database timeout config')",
        "reasoning": "用户报告数据库连接超时，先搜索超时相关配置定位问题范围",
        "observation": """找到 6 处匹配：
        db/pool.py:15 → pool_size=20, pool_timeout=30
        db/pool.py:28 → pool_recycle=3600, pool_pre_ping=True
        config/prod.yaml:8 → max_overflow=10, pool_timeout=30
        config/dev.yaml:8 → max_overflow=5, pool_timeout=60
        utils/health.py:42 → timeout=10 (健康检查专用)
        scripts/migrate.py:8 → timeout=300 (迁移脚本专用)""",
    },
    {
        "action": "read_file('db/pool.py')",
        # ↓ reasoning 浓缩了 step1 的发现：pool_timeout=30, max_overflow=10
        "reasoning": "搜索发现生产环境 pool_timeout=30 且 max_overflow=10，读取源码确认完整配置",
        "observation": """import sqlalchemy
        from sqlalchemy import create_engine
        engine = create_engine(
            DATABASE_URL,
            pool_size=20,        # 常驻连接数                                                                                        
            max_overflow=10,     # 溢出连接上限                                                                                                    
            pool_timeout=30,     # 获取连接的等待超时                                                                                              
            pool_recycle=3600,   # 每小时回收连接                                                                                                  
            pool_pre_ping=True,  # 使用前 ping 检测                                                                                                
            echo=False, echo_pool=False,                                                                                                       
            # 注意：没有设置 connect_timeout（数据库建连超时）
        )
        # 连接池监控
        from sqlalchemy import event
        @event.listens_for(engine, 'checkout')
        def receive_checkout(dbapi_conn, conn_record, conn_proxy):
            logging.info(f'Connection checked out: {conn_record}')""",
    },
    {
        "action": "run_tests('pytest tests/db/test_pool.py -v')",
        # ↓ reasoning 浓缩了 step2 的发现：缺少 connect_timeout
        "reasoning": "源码确认 max_overflow=10 且缺少 connect_timeout，跑测试验证高并发表现",                                
        "observation": """test_basic_connection PASSED       [20%]                                                                         
        test_pool_size PASSED              [40%]                                                                                                   
        test_timeout_handling PASSED       [60%]                                                                                                   
        test_recycle PASSED                [80%]                                                                                                   
        test_concurrent_overflow FAILED    [100%]                                                                                              
        FAILED - sqlalchemy.exc.TimeoutError: QueuePool limit of
        size 20 overflow 10 reached, connection timed out, timeout 30.
        ========================= 4 passed, 1 failed in 3.2s ==========================""",
    },
]

# --- 1. 展示核心机制：Reasoning 如何浓缩上一步 Observation ---
print("=== 信息传递链：每步 Reasoning 浓缩了上一步 Observation 的关键发现 ===\n")
for i, step in enumerate(tool_chain):
    if i > 0:
        prev = tool_chain[i-1]
        print(f"Step {i} Observation（{count_tokens(prev['observation'])} tokens 原始输出）")
        print(f"  ↓ 浓缩为")
        print(f"Step {i+1} Reasoning：「{step['reasoning']}」\n")

# --- 2. 构建完整 vs 遮蔽上下文 ---
full_context = ""
masked_context = ""
for step in tool_chain:
    full_context += f"Action: {step['action']}\nReasoning: {step['reasoning']}\nObservation:\n{step['observation']}\n\n"
    masked_context += f"Action: {step['action']}\nReasoning: {step['reasoning']}\nObservation: [已遮蔽]\n\n"

full_tokens = count_tokens(full_context)
masked_tokens = count_tokens(masked_context)
saved_pct = (1 - masked_tokens / full_tokens) * 100

print(f"=== Token 对比 ===")
print(f"完整上下文: {full_tokens} tokens")
print(f"遮蔽后:     {masked_tokens} tokens（节省 {saved_pct:.0f}%）")

# --- 3. LLM 对比：遮蔽后能否依靠 Reasoning 链定位根因 ---
question = "根据以上调试过程，数据库连接超时的根因是什么？建议怎么修复？"

import time
for label, ctx in [("完整上下文", full_context), ("遮蔽 Observation", masked_context)]:
    prompt = f"以下是 Agent 的调试过程：\n{ctx}\n{question}"
    start = time.time()
    resp = llm.invoke(prompt)
    elapsed = time.time() - start
    print(f"\n=== {label}（{elapsed:.1f}s）===")
    print(resp.content[:300])

print(f"\n关键发现：遮蔽 Observation 节省了 {saved_pct:.0f}% token，")
print(f"   但 Reasoning 链已携带关键信息（pool_timeout=30 → 缺少 connect_timeout → overflow FAILED），")
print(f"   LLM 仍能正确定位根因。这就是 Observation Masking 的核心原理。")

=== 信息传递链：每步 Reasoning 浓缩了上一步 Observation 的关键发现 ===

Step 1 Observation（119 tokens 原始输出）
  ↓ 浓缩为
Step 2 Reasoning：「搜索发现生产环境 pool_timeout=30 且 max_overflow=10，读取源码确认完整配置」

Step 2 Observation（201 tokens 原始输出）
  ↓ 浓缩为
Step 3 Reasoning：「源码确认 max_overflow=10 且缺少 connect_timeout，跑测试验证高并发表现」

=== Token 对比 ===
完整上下文: 567 tokens
遮蔽后:     129 tokens（节省 77%）

=== 完整上下文（21.5s）===
根据调试过程，可以确定数据库连接超时的**根因**是：

## 根因分析

1. **连接池配置不足**：
   - `pool_size=20`（常驻连接）
   - `max_overflow=10`（最大溢出连接）
   - 总连接上限 = 30（20 + 10）

2. **并发请求超过连接池容量**：
   - 测试 `test_concurrent_overflow` 失败，证明当并发请求超过30时会出现超时
   - 超时时间 `pool_timeout=30` 秒，用户会等待较长时间才收到错误

3. **缺少关键配置**：
   - 没有设置 `connect_timeout

=== 遮蔽 Observation（18.3s）===
根据调试过程，可以推断出数据库连接超时的**根因**是：

## 根因分析
1. **连接池配置不合理**：`pool_timeout=30` 设置过长，在高并发场景下，连接池耗尽后新请求需要等待30秒才超时
2. **连接溢出限制过小**：`max_overflow=10` 限制了最大额外连接数，当并发请求超过（pool_size + 10）时，新请求会排队等待
3. **缺少连接超时设置**：代码中没有设置 `connect_timeout`，数据库连接建立过程可能无限等待
4. **测试结果显示**：在高并发测试中，部分请求等待时间过长导致超时

## 修复建议

### 1. 调整

&emsp;&emsp;上面的代码通过手动构造数据展示了 Observation Masking 的原理。接下来，我们用 `LangChain` 框架的 `ReAct Agent` 来实现一个**真实的 Agent 调试流程**——Agent 自主决策调用哪些工具、按什么顺序执行，产生真实的 `AIMessage`（Reasoning + Action）和 `ToolMessage`（Observation）消息链。然后我们对这条真实消息链执行 Observation Masking，验证遮蔽后 LLM 是否仍能正确定位根因。

&emsp;&emsp;这是从"理解原理"到"工程实现"的关键一步：<font color=red>在 LangChain/LangGraph 架构中，Observation 对应的就是 `ToolMessage`，遮蔽操作本质上是替换 `ToolMessage.content`。</font>

In [36]:
# === Observation Masking：LangChain ReAct Agent 实战演示 ===
# 用真实的 Agent 工具调用链演示观察遮蔽原理
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage
from langchain.agents import create_agent
import time

# --- 1. 定义三个调试工具（模拟返回真实格式数据）---
@tool
def search_codebase(query: str) -> str:
    """搜索代码库中与查询相关的代码片段和配置"""
    return """找到 6 处匹配：
  db/pool.py:15 → pool_size=20, pool_timeout=30
  db/pool.py:28 → pool_recycle=3600, pool_pre_ping=True
  config/prod.yaml:8 → max_overflow=10, pool_timeout=30
  config/dev.yaml:8 → max_overflow=5, pool_timeout=60
  utils/health.py:42 → timeout=10 (健康检查专用)
  scripts/migrate.py:8 → timeout=300 (迁移脚本专用)"""

@tool
def read_file(file_path: str) -> str:
    """读取指定路径的文件内容"""
    return """import sqlalchemy
from sqlalchemy import create_engine
engine = create_engine(
    DATABASE_URL,
    pool_size=20,        # 常驻连接数
    max_overflow=10,     # 溢出连接上限
    pool_timeout=30,     # 获取连接的等待超时
    pool_recycle=3600,   # 每小时回收连接
    pool_pre_ping=True,  # 使用前 ping 检测
    echo=False, echo_pool=False,
    # 注意：没有设置 connect_timeout（数据库建连超时）
)
from sqlalchemy import event
@event.listens_for(engine, 'checkout')
def receive_checkout(dbapi_conn, conn_record, conn_proxy):
    logging.info(f'Connection checked out: {conn_record}')"""

@tool
def run_tests(command: str) -> str:
    """运行 pytest 测试并返回结果"""
    return """test_basic_connection PASSED       [20%]
test_pool_size PASSED              [40%]
test_timeout_handling PASSED       [60%]
test_recycle PASSED                [80%]
test_concurrent_overflow FAILED    [100%]
FAILED - sqlalchemy.exc.TimeoutError: QueuePool limit of
  size 20 overflow 10 reached, connection timed out, timeout 30.
========================= 4 passed, 1 failed in 3.2s =========================="""

# --- 2. 创建 ReAct Agent 并执行调试任务 ---
debug_agent = create_agent(llm, [search_codebase, read_file, run_tests],system_prompt="请你作为数据库运维人员，仔细梳理数据库保存问题，并将tool工具返回的结果内容进行Reasoning链思考，需要在思考中加入tool里面核心重要的内容")

print("=== Agent 自主调试：观察完整的 Action → Observation → Reasoning 链 ===\n")
result = debug_agent.invoke({"messages": [HumanMessage(content=(
    "数据库连接频繁超时，请调试定位根因。"
    "建议：先用 search_codebase 搜索 timeout 配置，"
    "再用 read_file 读取 db/pool.py，最后用 run_tests 跑 pytest tests/db/test_pool.py -v"
))]})

# --- 3. 展示 Agent 真实轨迹（标注每条消息类型）---
messages = result["messages"]
step = 0
for msg in messages:
    if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
        step += 1
        for tc in msg.tool_calls:
            args_preview = list(tc['args'].values())[0][:60] if tc['args'] else ''
            print(f"  Step {step} [Action]      {tc['name']}({args_preview})")
        if msg.content:
            print(f"  Step {step} [Reasoning]   {msg.content[:150]}")
    elif isinstance(msg, ToolMessage):
        obs_tokens = count_tokens(msg.content)
        print(f"  Step {step} [Observation] ({obs_tokens} tokens) {msg.content[:80]}...")
        print()
    elif isinstance(msg, AIMessage):
        print(f"  [Final Answer] {msg.content[:200]}...")
        print()

# --- 4. 核心操作：Observation Masking（基于信息传播链判断）---
# 核心逻辑：只有当 Observation[N] 后面存在 AIMessage（即 Reasoning[N+1] 已吸收其关键信息），
#           才能安全遮蔽。最近一轮的 Observation 尚未被后续推理吸收，必须保留。
print("=== 执行 Observation Masking：遮蔽已被后续推理吸收的历史 Observation ===\n")
masked_messages = []
for i, msg in enumerate(messages):
    if isinstance(msg, ToolMessage):
        # 检查这个 Observation 之后是否有 AIMessage 吸收了它
        has_subsequent_reasoning = any(
            isinstance(messages[j], AIMessage) for j in range(i + 1, len(messages))
        )
        if has_subsequent_reasoning:
            # 信息已传播到后续 Reasoning，安全遮蔽
            masked_messages.append(ToolMessage(
                content="[Observation 已被后续推理吸收]",
                tool_call_id=msg.tool_call_id
            ))
        else:
            # 最近一轮，信息尚未传播，保留完整内容
            masked_messages.append(msg)
    else:
        masked_messages.append(msg)

# Token 对比
full_tokens = sum(count_tokens(str(m.content)) for m in messages)
masked_tokens = sum(count_tokens(str(m.content)) for m in masked_messages)
saved_pct = (1 - masked_tokens / full_tokens) * 100
print(f"完整轨迹: {full_tokens} tokens")
print(f"遮蔽后:   {masked_tokens} tokens（节省 {saved_pct:.0f}%）")

# --- 5. 验证：用两种上下文分别提问，对比回答质量 ---
question = "根据以上调试过程，数据库连接超时的根因是什么？建议怎么修复？"

for label, msgs in [("完整上下文", messages), ("遮蔽 Observation", masked_messages)]:
    ctx = "\n".join(
        f"[{type(m).__name__}] {str(m.content)[:300]}" for m in msgs
    )
    start = time.time()
    resp = llm.invoke(f"以下是 Agent 的调试轨迹：\n{ctx}\n\n{question}")
    elapsed = time.time() - start
    print(f"\n--- {label}（{elapsed:.1f}s）---")
    print(resp.content[:300])

print(f"\nObservation Masking 验证完成：")
print(f"   历史 ToolMessage 遮蔽后节省 {saved_pct:.0f}% tokens，")
print(f"   关键：只遮蔽已被后续 Reasoning 吸收的 Observation，保留最近一轮完整内容。")
print(f"   AIMessage 中的 Reasoning 已浓缩历史 Observation 的关键发现，信息无损。")

=== Agent 自主调试：观察完整的 Action → Observation → Reasoning 链 ===

  Step 1 [Action]      search_codebase(timeout database connection pool)
  Step 1 [Reasoning]   我来按照您的建议逐步排查数据库连接超时问题。
  Step 1 [Observation] (119 tokens) 找到 6 处匹配：
  db/pool.py:15 → pool_size=20, pool_timeout=30
  db/pool.py:28 → pool...

  Step 2 [Action]      read_file(db/pool.py)
  Step 2 [Reasoning]   现在让我读取 db/pool.py 文件来查看数据库连接池的具体实现：
  Step 2 [Observation] (176 tokens) import sqlalchemy
from sqlalchemy import create_engine
engine = create_engine(
 ...

  Step 3 [Action]      run_tests(pytest tests/db/test_pool.py -v)
  Step 3 [Reasoning]   现在让我运行数据库连接池的测试来查看是否有相关的问题：
  Step 3 [Observation] (110 tokens) test_basic_connection PASSED       [20%]
test_pool_size PASSED              [40%...

  [Final Answer] ## Reasoning链思考

基于工具返回的结果，我进行了以下分析：

### 1. 配置分析（search_codebase结果）
- **生产环境配置**：`pool_timeout=30`秒，`max_overflow=10`
- **开发环境配置**：`pool_timeout=60`秒，`max_overflow=5`
- **连接池配置**：`pool_size=20`，`pool...

=== 执行 Observa

&emsp;&emsp;对比两个版本的实现，我们可以看到 Observation Masking 在 `LangChain` 框架中的工程映射关系：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Observation Masking：概念与 LangChain 实现的映射</font></p>
<div class="center">

| 概念层 | LangChain 实现层 | 说明 |
|--------|-----------------|------|
| Action（动作） | `AIMessage.tool_calls` | Agent 决定调用哪个工具、传什么参数 |
| Reasoning（推理） | `AIMessage.content` | Agent 在调用工具前的思考过程 |
| Observation（观察） | `ToolMessage.content` | 工具执行后的返回值 |
| 遮蔽操作 | 替换 `ToolMessage.content` 为占位符 | 保留 Action + Reasoning，丢弃冗长的工具输出 |

</div>

&emsp;&emsp;<font color=red>核心洞察：在 LangChain 的消息体系中，`ToolMessage` 是 token 消耗最大的部分（搜索结果、文件内容、测试输出），而 `AIMessage` 中的 Reasoning 已经浓缩了 Observation 的关键发现。这就是 Observation Masking 能在不损失信息的前提下大幅节省 token 的根本原因。</font>

> **数据来源**：JetBrains Research 在 [NeurIPS 2025 DL4C workshop](https://github.com/JetBrains-Research/PIPer) 发表的研究，测试环境为 SWE-bench Verified 数据集 + Qwen3-Coder 480B 模型。

&emsp;&emsp;三种主要压缩技术的适用域值得深入对比：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528803.png" width=60%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三种压缩技术对比</font></p>
<div class="center">

| 技术 | 机制 | 成本削减 | 额外延迟 | 信息保留特征 | 适用场景 |
|------|------|---------|---------|------------|---------|
| 工具结果清除 | 调用完成后丢弃原始结果 | 20-40% | ~0 | 仅保留结论 | **所有场景（首先采用）** |
| 观察遮蔽 | 保留推理+动作，替换旧观察为占位符 | ~52% | ~0 | 动作链完整 | **工具密集型 SE Agent** |
| LLM 摘要 | 独立模型压缩全部交互 | 50-60% | 1-5 秒 | 压缩后可能丢失细节 | **需要理论无限回合的通用 Agent** |

</div>

> &emsp;**注释**：观察遮蔽的 52% 成本削减数据来自 JetBrains Research 在 SWE-bench Verified 上使用 Qwen3-Coder 480B 的测试结果（NeurIPS 2025 DL4C），实际效果因模型和场景而异。

> **踩坑预警**：观察遮蔽（Observation Masking）的适用边界需要特别注意。JetBrains Research 在 NeurIPS 2025 DL4C 的研究中证明，在 SE Agent 场景（工具调用密集、观察结果高度结构化）中，4/5 测试场景观察遮蔽比 LLM 摘要更便宜且效果相当或更好。但这个结论<font color=red>不能推广到对话型 Agent</font>——在客服、教学等场景中，"观察"就是用户的话，遮蔽用户的话会导致灾难性信息丢失。正确做法：工具密集型 Agent 优先用观察遮蔽，对话型 Agent 优先用 trim_messages + LLM 摘要。排查方向：如果 Agent 在压缩后出现上下文"断片"，检查压缩策略是否与场景匹配。

#### 3.1.4 Isolate 策略：用子 Agent 隔离上下文污染

&emsp;&emsp;Isolate 的核心思想是：**当一个复杂任务可以分解为独立子任务时，让每个子任务在自己的上下文中执行，互不污染**。主 Agent 不需要把全部历史传给子 Agent——只传递任务描述和必要参数，子 Agent 返回精炼结果。这就像公司 CEO 不需要把所有会议纪要都发给每个部门，只需要发相关的任务指令。

&emsp;&emsp;我们用代码演示这种上下文隔离的效果——同一个子任务，对比"继承全部主 Agent 历史"和"只传递任务描述"两种方式：

In [37]:
# === Isolate 策略：子 Agent 的上下文隔离 ===

# 模拟主 Agent 的完整上下文（3000 tokens 的累积历史）
main_agent_history = """[系统提示] 你是 DevAssist，后端工程 AI 助手...（800 tokens）
[轮次1] 用户讨论了数据库选型，最终选择 PostgreSQL...
[轮次2] 用户要求设计 API 路由结构，生成了 15 个端点...
[轮次3] 用户报告了认证模块的 JWT 刷新 bug，已修复...
[轮次4] 用户要求添加 Redis 缓存层，完成了 Cache-Aside 实现...
[轮次5] 用户讨论了部署方案，选择 Docker + ECS...
[轮次6] 用户要求优化慢查询，分析了 3 个问题 SQL...
[轮次7] 用户讨论了日志方案，选择 ELK Stack...
[轮次8] 用户要求写单元测试，覆盖了 auth 和 db 模块..."""

# 当前子任务：生成数据库迁移脚本（只需要知道数据库相关信息）
subtask = "生成从 users 表添加 email_verified 字段的数据库迁移脚本（PostgreSQL, SQLAlchemy 2.0, alembic）"

# 方式A：继承全部历史（不隔离）
context_no_isolate = f"{main_agent_history}\n\n当前任务：{subtask}"

# 方式B：只传任务描述（隔离）
context_isolated = f"你是数据库迁移专家。请完成以下任务：\n{subtask}"

tokens_full = count_tokens(context_no_isolate)
tokens_isolated = count_tokens(context_isolated)

print(f"不隔离（继承全部历史）: {tokens_full} tokens")
print(f"隔离（只传任务描述）:   {tokens_isolated} tokens")
print(f"上下文缩减: {(1-tokens_isolated/tokens_full)*100:.0f}%\n")

# 对比回答质量
for label, ctx in [("不隔离", context_no_isolate), ("隔离", context_isolated)]:
    resp = llm.invoke(ctx)
    print(f"=== {label} ===")
    print(resp.content[:250])
    print()

print("两种方式都能正确生成迁移脚本——但隔离版本不携带 JWT bug、Redis 缓存等无关历史")


不隔离（继承全部历史）: 208 tokens
隔离（只传任务描述）:   41 tokens
上下文缩减: 80%

=== 不隔离 ===
我将为您生成完整的数据库迁移脚本，包含前后向迁移操作。

## 1. 生成 Alembic 迁移文件

```bash
# 生成新的迁移文件
alembic revision -m "add_email_verified_to_users"
```

## 2. 迁移脚本内容

**文件：`migrations/versions/YYYYMMDDHHMMSS_add_email_verified_to_users.py`**

```python
"""add email_verified to

=== 隔离 ===
我将为您生成一个完整的数据库迁移脚本，用于向 `users` 表添加 `email_verified` 字段。

## 1. 首先创建迁移文件

```bash
# 生成迁移文件
alembic revision -m "add_email_verified_to_users"
```

## 2. 迁移脚本内容

```python
"""add_email_verified_to_users

Revision ID: <自动生成的ID>
Revises: <上一个迁移的ID>
Create

两种方式都能正确生成迁移脚本——但隔离版本不携带 JWT bug、Redis 缓存等无关历史


&emsp;&emsp;上面的代码通过手动构造字符串展示了隔离的 token 节省效果。接下来，我们用 `LangChain` 的 `SubAgentMiddleware` 中间件来实现**真实的子 Agent 上下文隔离**。这个中间件来自 `deepagents` 包，它的核心机制是：注册子 Agent 后，中间件会自动给主 Agent 注入一个 `task` 工具，主 Agent 通过 `task(agent_name="xxx", description="...")` 将任务委派给子 Agent，子 Agent 在完全隔离的上下文中执行。

&emsp;&emsp;<font color=red>关键区别：主 Agent 的对话历史（JWT bug、Redis 缓存等）不会传递给子 Agent——子 Agent 只收到 `description` 参数中的任务描述。这就是中间件层面的上下文隔离。</font>

In [38]:
# === Isolate 策略实战（1/3）：用 SubAgentMiddleware 注册子 Agent ===
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware
from deepagents.backends import StateBackend
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# 主 Agent + 子 Agent 一体化配置
isolate_agent = create_agent(
    model=llm,
    tools=[],                    # 主 Agent 自身无额外工具
    system_prompt="你是 DevAssist 后端工程助手。数据库迁移任务请使用 task 工具委派给 db_expert。",
    middleware=[
        SubAgentMiddleware(
            backend=StateBackend(),  # 轻量级内存后端，无需文件系统
            subagents=[
                {
                    "name": "db_expert",
                    "description": "数据库迁移专家，处理 PostgreSQL + Alembic 相关任务",
                    "system_prompt": "你是数据库迁移专家，精通 PostgreSQL、SQLAlchemy 2.0、Alembic。只处理数据库相关任务，输出完整可用的迁移脚本。",
                    "model": "deepseek:deepseek-chat",  # 子 Agent 使用的模型
                    "tools": [],
                }
            ],
        )
    ],
    checkpointer=InMemorySaver(),  # 跨轮次保存主 Agent 历史
)

print("主 Agent 创建完成")
print("  中间件自动注入了 task 工具，可委派任务给 db_expert 子 Agent")
print("  子 Agent 每次被调用时上下文完全隔离，不继承主 Agent 历史")

主 Agent 创建完成
  中间件自动注入了 task 工具，可委派任务给 db_expert 子 Agent
  子 Agent 每次被调用时上下文完全隔离，不继承主 Agent 历史


In [39]:
# === Isolate 策略实战（2/3）：积累主 Agent 对话历史 ===
config = {"configurable": {"thread_id": "isolate-demo"}}

# 模拟 3 轮无关任务（限制回复长度以加速演示），让主 Agent 积累上下文
history = [
    "帮我选择数据库，PostgreSQL 还是 MySQL？一句话给结论",
    "JWT 刷新 token 过期后没有自动续签，一句话说修复思路",
    "Redis 缓存穿透怎么防？一句话总结方案",
]

print("=== 积累主 Agent 对话历史（3 轮简短任务）===\n")
for i, msg in enumerate(history):
    result = isolate_agent.invoke(
        {"messages": [HumanMessage(content=msg)]}, config
    )
    msg_count = len(result["messages"])
    print(f"轮次 {i+1} | 消息数: {msg_count:3d} | {msg[:35]}...")

parent_tokens = sum(count_tokens(str(m.content)) for m in result["messages"])
print(f"\n主 Agent 累积上下文: {parent_tokens} tokens, {len(result['messages'])} 条消息")
print("包含：数据库选型、JWT 修复思路、Redis 缓存方案")

=== 积累主 Agent 对话历史（3 轮简短任务）===

轮次 1 | 消息数:   2 | 帮我选择数据库，PostgreSQL 还是 MySQL？一句话给结论...
轮次 2 | 消息数:   4 | JWT 刷新 token 过期后没有自动续签，一句话说修复思路...
轮次 3 | 消息数:   6 | Redis 缓存穿透怎么防？一句话总结方案...

主 Agent 累积上下文: 89 tokens, 6 条消息
包含：数据库选型、JWT 修复思路、Redis 缓存方案


In [40]:
# === Isolate 策略实战（3/3）：通过 task 工具委派，观察上下文隔离 ===
task_desc = "生成 users 表添加 email_verified 布尔字段的 Alembic 迁移脚本"
task_tokens = count_tokens(task_desc)

# 子 Agent 的完整上下文 = system_prompt + task_description
sub_system_prompt = "你是数据库迁移专家，精通 PostgreSQL、SQLAlchemy 2.0、Alembic。只处理数据库相关任务，输出完整可用的迁移脚本。"
sub_context_tokens = count_tokens(sub_system_prompt) + task_tokens

print("=== 发送数据库迁移任务 ===\n")
print("主 Agent 收到任务后，应通过中间件注入的 task 工具委派给 db_expert...\n")

result = isolate_agent.invoke(
    {"messages": [HumanMessage(content=(
        f"这是数据库任务，请使用 task 工具委派给 db_expert：{task_desc}"
    ))]},
    config
)

# 展示隔离效果
print(f"主 Agent 上下文: {parent_tokens} tokens（含数据库选型、JWT 修复、Redis 方案等全部历史）")
print(f"子 Agent 收到的: {task_tokens} tokens（仅 task description，零历史）")
print(f"上下文隔离率:    {(1 - task_tokens/parent_tokens)*100:.0f}%\n")

print("=== 子 Agent 返回的迁移脚本 ===")
print(result["messages"][-1].content[:500])

print(f"\nSubAgentMiddleware 隔离机制验证：")
print(f"   中间件自动注入 task 工具 → 主 Agent 调用 task(agent_name='db_expert') → ")
print(f"   子 Agent 在隔离上下文中执行（仅 {task_tokens} tokens）→ 返回精炼结果")

=== 发送数据库迁移任务 ===

主 Agent 收到任务后，应通过中间件注入的 task 工具委派给 db_expert...

主 Agent 上下文: 89 tokens（含数据库选型、JWT 修复、Redis 方案等全部历史）
子 Agent 收到的: 18 tokens（仅 task description，零历史）
上下文隔离率:    80%

=== 子 Agent 返回的迁移脚本 ===
已生成完整的 Alembic 迁移脚本，为 users 表添加 email_verified 布尔字段，包含 upgrade 和 downgrade 函数，可直接运行。

SubAgentMiddleware 隔离机制验证：
   中间件自动注入 task 工具 → 主 Agent 调用 task(agent_name='db_expert') → 
   子 Agent 在隔离上下文中执行（仅 18 tokens）→ 返回精炼结果


&emsp;&emsp;对比模拟版和中间件版的实现，可以看到 `SubAgentMiddleware` 的工程价值：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Isolate 策略：SubAgentMiddleware 实现机制</font></p>
<div class="center">

| 概念层 | SubAgentMiddleware 实现 | 说明 |
|--------|------------------------|------|
| 主 Agent | `create_agent` + `checkpointer` | 积累完整对话历史 |
| 子 Agent 注册 | `subagents=[{"name": ..., "system_prompt": ...}]` | 声明式配置，无需手动创建 |
| 隔离边界 | 中间件自动注入的 `task` 工具 | 主 Agent 通过 `task()` 委派 |
| 信息传递 | `task(agent_name, description)` | 只传任务描述，不传历史 |
| 结果回传 | `task` 工具返回值 | 子 Agent 返回精炼结果给主 Agent |

</div>

&emsp;&emsp;<font color=red>核心洞察：`SubAgentMiddleware` 将「创建子 Agent → 包装为工具 → 注入主 Agent」这三步封装为一行声明式配置。工程师只需定义子 Agent 的名称、职责和工具，中间件自动完成上下文隔离。这是从「手动隔离」到「框架级隔离」的关键进化。</font>

#### 3.1.5 Cache 策略：让静态前缀只计算一次

&emsp;&emsp;Cache 策略利用了一个关键事实：**Agent 每轮调用 LLM 时，请求的前缀（系统提示 + 工具描述）几乎完全相同**。Prompt Caching 让 API 提供商缓存这些不变的前缀，后续请求直接命中缓存，跳过重复计算。Anthropic 的缓存命中价格仅为标准价的 10%，意味着系统提示 + 工具描述这部分的输入成本直降 90%。

&emsp;&emsp;我们来做一个可调参数的成本计算器——输入你的场景参数，算出 Prompt Caching 的年度节省：

In [41]:
# === Cache 策略：Prompt Caching 成本计算器 ===

# 可调参数（修改这些值试试你自己的场景）
system_prompt_tokens = 800    # 系统提示 token 数
tool_desc_tokens = 600        # 工具描述 token 数
daily_calls = 500             # 每天 LLM 调用次数
input_price_per_m = 3.00      # 输入价格（$/M tokens），Claude Sonnet 4.6
cache_hit_rate = 0.95         # 缓存命中率（通常 > 90%）

# 计算
static_prefix = system_prompt_tokens + tool_desc_tokens  # 每次请求的固定前缀
monthly_calls = daily_calls * 30
yearly_calls = daily_calls * 365

# 无缓存：每次请求都按标准价计算前缀
cost_no_cache = yearly_calls * static_prefix / 1_000_000 * input_price_per_m

# 有缓存：命中部分按 10% 计价，未命中按标准价 + 25% 缓存写入费
cache_hit_cost = yearly_calls * cache_hit_rate * static_prefix / 1_000_000 * input_price_per_m * 0.1
cache_miss_cost = yearly_calls * (1 - cache_hit_rate) * static_prefix / 1_000_000 * input_price_per_m * 1.25
cost_with_cache = cache_hit_cost + cache_miss_cost

saving = cost_no_cache - cost_with_cache

import pandas as pd

# 按月展示
monthly_no_cache = monthly_calls * static_prefix / 1_000_000 * input_price_per_m
monthly_with_cache = cost_with_cache / 12

df = pd.DataFrame([
    {"场景": "无缓存", "月度成本": f"${monthly_no_cache:.2f}", "年度成本": f"${cost_no_cache:.2f}"},
    {"场景": "Prompt Caching", "月度成本": f"${monthly_with_cache:.2f}", "年度成本": f"${cost_with_cache:.2f}"},
    {"场景": "年度节省", "月度成本": f"${(monthly_no_cache-monthly_with_cache):.2f}/月", "年度成本": f"${saving:.2f}（{saving/cost_no_cache*100:.0f}%）"},
])

print(f"场景参数: 系统提示 {system_prompt_tokens} + 工具描述 {tool_desc_tokens} = {static_prefix} tokens 静态前缀")
print(f"         {daily_calls} 次/天, 命中率 {cache_hit_rate*100:.0f}%, Claude Sonnet ${input_price_per_m}/M\n")
df


场景参数: 系统提示 800 + 工具描述 600 = 1400 tokens 静态前缀
         500 次/天, 命中率 95%, Claude Sonnet $3.0/M



,场景,月度成本,年度成本
0,无缓存,$63.00,$766.50
1,Prompt Caching,$10.06,$120.72
2,年度节省,$52.94/月,$645.78（84%）


&emsp;&emsp;从计算结果可以看到：Prompt Caching 的年度节省通常在 **85-90%** 之间（取决于命中率）。而且这是**零代码改动**的优化——只需要确保请求的前缀保持稳定（系统提示在最前面，动态内容在后面），API 提供商自动处理缓存逻辑。这就是为什么在 3.3 节的决策优先级中，Cache 排在第一位。

&emsp;&emsp;上面的计算器给出了理论上的成本节省。但在真实的 Agent 环境中，Prompt Caching 到底是怎么工作的？接下来，我们用 `ChatDeepSeek` 构造一个接近生产环境的系统提示（角色定义 + 7 个工具描述 + 12 条约束规则），连续发送多次请求，观察 API 返回的缓存命中数据。

&emsp;&emsp;<font color=red>关键机制：DeepSeek 的前缀缓存（Prefix Caching）是服务端自动完成的——只要多次请求的前缀（系统提示 + 工具描述）完全相同，服务端会自动缓存并在后续请求中复用。缓存命中的 token 按标准价格的 10% 计费。需要注意的是，服务端缓存需要多次请求后才建立（通常第 4-6 次开始命中），这在生产环境中意味着冷启动阶段会有短暂的全价期。</font>

In [ ]:
# === Cache 策略实战：观察 DeepSeek Prefix Caching 的 API 响应 ===
import time
from langchain_core.messages import SystemMessage, HumanMessage

# 构造一个真实 Agent 的系统提示（角色 + 工具描述 + 约束规则）
# 生产环境中，这个前缀每次 LLM 调用都要发送——正是缓存的最佳目标
cache_system_prompt = """你是 DevAssist 后端工程助手，精通以下完整技术栈，必须严格遵守所有约束规则。

【角色定义】
你是一名具有 10 年后端开发经验的高级工程师，专注于 Python 生态的 Web 服务架构。核心能力包括：
1. API 设计与实现：精通 RESTful API 规范（Richardson 成熟度模型 L3）、GraphQL Schema 设计（包括 Subscription 和 Federation）、gRPC 服务定义与 Protobuf 编写、WebSocket 长连接管理
2. 数据库设计与优化：PostgreSQL 高级特性（JSONB 索引优化、递归 CTE、窗口函数、分区表）、Redis 数据结构选型（String/Hash/Set/ZSet/Stream）、MongoDB 聚合管道与分片策略
3. 微服务架构：Docker 多阶段构建与镜像瘦身、Kubernetes 资源编排（Deployment/StatefulSet/DaemonSet/Job/CronJob/Service/Ingress/HPA/VPA/PDB）、Istio Service Mesh 流量管理与故障注入
4. 性能调优：多级缓存策略设计（L1 进程内 LRU + L2 Redis Cluster + L3 CDN 边缘缓存）、数据库连接池配置与监控、asyncio 异步编程模式（Task/Gather/Semaphore/Queue）、批处理与流式处理的权衡
5. 安全实践：OAuth 2.0/OIDC 认证授权完整流程（Authorization Code/PKCE/Client Credentials/Device Flow）、数据传输加密（TLS 1.3 配置与证书管理）、OWASP Top 10 漏洞防护策略

【输出风格约束】
- 回答必须简洁精准，优先给出代码示例而非冗长解释
- 涉及架构决策时必须说明 trade-off（至少列出 2 个替代方案的优劣）
- 涉及性能优化时必须给出可量化的指标（QPS、延迟 P99、内存占用）
- 不确定的信息必须明确标注「需要进一步确认」

【可用工具描述】
Tool 1: search_codebase
  功能：搜索项目代码库中的文件、函数、类定义和注释内容
  参数：query(str) 搜索关键词，支持正则表达式和模糊匹配；file_type(str) 文件类型过滤（py/yaml/sql/md/dockerfile/proto）；scope(str) 搜索范围（all/src/tests/docs/configs）；context_lines(int) 上下文行数，默认3
  返回：匹配的代码片段列表，每条包含文件绝对路径、起始行号、匹配行高亮、前后 context_lines 行上下文
  使用场景：了解现有代码结构和模块依赖、查找函数或类的定义和所有调用点、定位 bug 的根因代码、追踪配置项的引用链路

Tool 2: run_tests
  功能：执行项目测试套件，支持单元测试、集成测试和端到端测试
  参数：test_path(str) 测试文件或目录路径；verbose(bool) 是否输出详细的每条用例结果；markers(str) pytest marker 表达式过滤（如 slow/integration/smoke）；parallel(bool) 是否使用 pytest-xdist 并行执行；coverage(bool) 是否生成覆盖率报告
  返回：测试结果摘要（通过/失败/跳过/错误数量）、失败用例的完整 traceback 和局部变量快照、覆盖率报告（行覆盖率/分支覆盖率/未覆盖行号）
  使用场景：验证代码修改未引入回归问题、确认新功能的正向和反向测试用例全部通过、检查关键模块的代码覆盖率是否达标

Tool 3: query_database
  功能：执行只读 SQL 查询，自动添加 LIMIT 保护和超时控制
  参数：sql(str) SQL 查询语句，仅允许 SELECT/EXPLAIN/SHOW 语句；database(str) 目标数据库名称；timeout(int) 查询超时秒数，默认30；format(str) 输出格式（table/json/csv）
  返回：查询结果集（最多 100 行）、EXPLAIN ANALYZE 执行计划摘要、查询实际耗时和扫描行数
  使用场景：诊断数据一致性和完整性问题、验证数据迁移前后的数据正确性、分析慢查询的执行计划和优化方向、统计业务指标和数据分布

Tool 4: deploy_service
  功能：部署服务到指定环境，自动执行健康检查、金丝雀验证和自动回滚
  参数：service(str) 服务名称（必须在服务注册表中）；env(str) 目标环境（dev/staging/production）；version(str) 部署版本号（git tag 或 commit hash）；canary_percent(int) 金丝雀流量百分比（0-100）；timeout(int) 部署超时分钟数
  返回：部署状态（pending/rolling_update/canary_verifying/completed/rolled_back/failed）、健康检查详情（HTTP/TCP/gRPC 探针结果）、金丝雀指标对比（错误率/延迟/CPU 使用率的 baseline vs canary）
  使用场景：发布新版本服务、灰度验证新功能对性能和稳定性的影响、故障版本快速回滚

Tool 5: monitor_metrics
  功能：查询服务实时监控指标、历史趋势和异常检测结果
  参数：service(str) 服务名称；metric(str) 指标名（latency_p50/latency_p95/latency_p99/error_rate/qps/cpu_usage/memory_usage/gc_pause/thread_count/connection_pool_usage）；duration(str) 时间范围（5m/15m/1h/6h/24h/7d/30d）；aggregation(str) 聚合方式（avg/max/min/sum/count）
  返回：时间序列数据点列表（timestamp + value）、统计摘要（min/max/avg/median/p50/p95/p99/stddev）、异常检测结果（基于 3-sigma 和趋势分析）
  使用场景：建立服务性能基线、诊断告警事件的根因、评估优化措施的效果、容量规划和扩缩容决策

Tool 6: manage_secrets
  功能：管理服务密钥、证书和敏感配置，支持多环境隔离和版本审计
  参数：action(str) 操作类型（get/set/rotate/delete/list/history）；key(str) 密钥名称（支持路径格式如 service/db/password）；env(str) 目标环境；version(int) 指定版本号（用于回滚）
  返回：操作结果、密钥元信息（创建时间/过期时间/最近访问时间/访问计数/创建者）
  使用场景：定期密钥轮转（数据库密码/API Token/TLS 证书）、多环境配置同步、安全审计和合规检查

Tool 7: analyze_logs
  功能：搜索和分析服务日志，支持结构化查询、聚合统计和关联追踪
  参数：service(str) 服务名称；query(str) 日志搜索表达式（支持 Lucene 语法）；level(str) 日志级别过滤（DEBUG/INFO/WARN/ERROR/FATAL）；time_range(str) 时间范围；limit(int) 返回条目上限
  返回：匹配的日志条目（含完整结构化字段）、按时间/级别/来源的频率统计直方图、关联的 trace_id 和 span_id 列表（用于分布式追踪）
  使用场景：生产环境错误排查和根因定位、请求全链路追踪（跨服务关联）、异常模式识别（错误频率突增/周期性异常）、SLA 违规事件取证

【约束规则】
1. 所有数据库操作必须使用参数化查询，禁止任何形式的字符串拼接 SQL，防止 SQL 注入攻击
2. API 响应必须遵循统一格式：{"code": int, "message": str, "data": any, "request_id": str, "timestamp": str}
3. 所有敏感操作（部署、密钥变更、数据删除、权限修改）必须记录审计日志，包含操作者身份、时间戳、变更内容和影响范围
4. 生产环境部署必须先通过 staging 环境的完整验证，且自动化测试覆盖率不低于 80%
5. 每次代码修改必须附带对应的单元测试，新增代码的行覆盖率不低于 90%，分支覆盖率不低于 80%
6. 第三方依赖必须锁定精确版本号（使用 == 约束），禁止使用 >=、~=、^= 等模糊版本约束
7. 所有异步操作必须设置超时时间（默认 30 秒），避免资源泄漏和无限等待导致的线程/协程饥饿
8. 数据库连接必须使用连接池管理（最小连接数 5、最大连接数 20、空闲超时 300 秒），禁止每次请求创建新连接
9. 所有缓存必须设置合理的 TTL（默认 300 秒），禁止永久缓存，防止数据不一致和内存泄漏
10. API 限流策略：认证用户 100 req/min，匿名用户 20 req/min，管理员 500 req/min，超限返回 429 状态码
11. 错误处理必须区分可重试错误（5xx、超时、连接中断）和不可重试错误（4xx、业务逻辑错误），可重试错误实现指数退避（初始 1s、最大 60s、抖动 ±20%）
12. 日志输出必须使用结构化 JSON 格式，每条日志必须包含 timestamp、level、service、trace_id、span_id、message 字段"""

# 先测量系统提示的 token 数
prefix_tokens = count_tokens(cache_system_prompt)
print(f"系统提示 token 数: {prefix_tokens}")
print(f"这 {prefix_tokens} tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发\n")

# 模拟 Agent 连续对话：8 轮不同的用户问题，但系统提示完全相同
# DeepSeek 的前缀缓存需要多次请求后才在服务端建立
# 通常第 4-6 次请求开始命中，这正是真实生产环境中的行为模式
questions = [
    "PostgreSQL JSONB 类型有哪些核心优势？",
    "如何用 Alembic 生成添加字段的迁移脚本？",
    "Redis 缓存穿透怎么解决？",
    "FastAPI 中间件的执行顺序是什么？",
    "如何配置 Kubernetes HPA 自动扩缩容？",
    "Docker 多阶段构建的最佳实践？",
    "asyncio 和多线程的选择标准是什么？",
    "如何设计 API 限流的降级策略？",
]

results = []
for i, q in enumerate(questions):
    msgs = [SystemMessage(content=cache_system_prompt), HumanMessage(content=q)]
    response = llm.invoke(msgs)

    # 提取 API 返回的缓存命中数据
    usage = response.response_metadata.get("token_usage", {})
    hit = usage.get("prompt_cache_hit_tokens", 0)
    miss = usage.get("prompt_cache_miss_tokens", 0)
    prompt = usage.get("prompt_tokens", 0)

    results.append({"round": i + 1, "question": q[:15], "prompt": prompt, "hit": hit, "miss": miss})

    status = "缓存命中" if hit > 0 else "缓存未命中"
    print(f"轮次 {i+1}: prompt={prompt} tokens, cache_hit={hit}, cache_miss={miss} [{status}]")
    time.sleep(2)  # 等待服务端缓存传播

print(f"\n说明: prompt_cache_hit_tokens = 缓存命中的 token 数（按 10% 计价）")
print(f"      prompt_cache_miss_tokens = 未命中的 token 数（按标准价 + 25% 写入费）")

系统提示 token 数: 2007
这 2007 tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发

轮次 1: prompt=2021 tokens, cache_hit=0, cache_miss=2021 [缓存未命中]
轮次 2: prompt=2024 tokens, cache_hit=0, cache_miss=2024 [缓存未命中]
轮次 3: prompt=2018 tokens, cache_hit=0, cache_miss=2018 [缓存未命中]
轮次 4: prompt=2020 tokens, cache_hit=0, cache_miss=2020 [缓存未命中]
轮次 5: prompt=2022 tokens, cache_hit=0, cache_miss=2022 [缓存未命中]
轮次 6: prompt=2020 tokens, cache_hit=1984, cache_miss=36 [缓存命中]
轮次 7: prompt=2021 tokens, cache_hit=1984, cache_miss=37 [缓存命中]
轮次 8: prompt=2021 tokens, cache_hit=1984, cache_miss=37 [缓存命中]

说明: prompt_cache_hit_tokens = 缓存命中的 token 数（按 10% 计价）
      prompt_cache_miss_tokens = 未命中的 token 数（按标准价 + 25% 写入费）


In [44]:
msgs = [SystemMessage(content=cache_system_prompt), HumanMessage(content="PostgreSQL JSONB 类型有哪些核心优势？")]
response = llm.invoke(msgs)

# 提取 API 返回的缓存命中数据
print(response.response_metadata)

usage = response.response_metadata.get("token_usage", {})
print(usage)

hit = usage.get("prompt_cache_hit_tokens", 0)
miss = usage.get("prompt_cache_miss_tokens", 0)
prompt = usage.get("prompt_tokens", 0)

results.append({"round": i + 1, "question": q[:15], "prompt": prompt, "hit": hit, "miss": miss})

{'token_usage': {'completion_tokens': 1118, 'prompt_tokens': 2021, 'total_tokens': 3139, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1984}, 'prompt_cache_hit_tokens': 1984, 'prompt_cache_miss_tokens': 37}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '2c1ef813-e591-4b3c-98b6-92623f534984', 'finish_reason': 'stop', 'logprobs': None}
{'completion_tokens': 1118, 'prompt_tokens': 2021, 'total_tokens': 3139, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1984}, 'prompt_cache_hit_tokens': 1984, 'prompt_cache_miss_tokens': 37}


&emsp;&emsp;从 API 返回的 `prompt_cache_hit_tokens` 和 `prompt_cache_miss_tokens` 字段，我们可以直接看到缓存机制在工作。DeepSeek 的前缀缓存是服务端自动管理的——缓存的写入和命中由服务端根据请求前缀的重复频率决定，<font color=red>开发者不需要做任何额外配置</font>，只需保证系统提示放在消息列表的最前面。

&emsp;&emsp;接下来，我们基于实际的 token 数据，计算这个 Agent 在不同调用量下的缓存节省效果：

In [43]:
# === 基于实际 Token 数据计算 Prompt Caching 成本节省 ===
import pandas as pd

# DeepSeek V3 定价（$/M tokens）
deepseek_input_price = 0.28    # 标准输入价格
cache_hit_discount = 0.1       # 缓存命中按 10% 计价
cache_write_surcharge = 1.25   # 首次写入按 125% 计价

# 使用前面测量的实际系统提示 token 数
static_prefix = prefix_tokens
print(f"每次请求的静态前缀: {static_prefix} tokens（角色 + 工具描述 + 约束规则）")
print(f"这部分内容每轮对话都完全相同 → 缓存的理想目标\n")

# 对比三种日调用量场景
scenarios = [100, 500, 2000]

rows = []
for daily in scenarios:
    yearly = daily * 365

    # 无缓存：每次请求都按标准价
    cost_no_cache = yearly * static_prefix / 1_000_000 * deepseek_input_price

    # 有缓存（假设 95% 命中率，保守估计）
    hit_rate = 0.95
    cost_hit = yearly * hit_rate * static_prefix / 1_000_000 * deepseek_input_price * cache_hit_discount
    cost_miss = yearly * (1 - hit_rate) * static_prefix / 1_000_000 * deepseek_input_price * cache_write_surcharge
    cost_cached = cost_hit + cost_miss

    saving_pct = (1 - cost_cached / cost_no_cache) * 100

    rows.append({
        "日调用量": f"{daily} 次/天",
        "年调用量": f"{yearly:,}",
        "无缓存年成本": f"${cost_no_cache:.2f}",
        "有缓存年成本": f"${cost_cached:.2f}",
        "年节省": f"${cost_no_cache - cost_cached:.2f}（{saving_pct:.0f}%）",
    })

df = pd.DataFrame(rows)
print(f"定价基准: DeepSeek V3 ${deepseek_input_price}/M tokens, 命中率假设 95%")
print(f"缓存命中价: ${deepseek_input_price * cache_hit_discount}/M tokens（标准价的 10%）\n")
df

每次请求的静态前缀: 2007 tokens（角色 + 工具描述 + 约束规则）
这部分内容每轮对话都完全相同 → 缓存的理想目标

定价基准: DeepSeek V3 $0.28/M tokens, 命中率假设 95%
缓存命中价: $0.028000000000000004/M tokens（标准价的 10%）



,日调用量,年调用量,无缓存年成本,有缓存年成本,年节省
0,100 次/天,"36,500",$20.51,$3.23,$17.28（84%）
1,500 次/天,"182,500",$102.56,$16.15,$86.40（84%）
2,2000 次/天,"730,000",$410.23,$64.61,$345.62（84%）


&emsp;&emsp;对比成本计算器的理论结果和 API 实测数据，我们可以得出 Prompt Caching 的工程要点：

> &emsp;**Prompt Caching 三条实践规则**：
> &emsp;1. **系统提示放最前面**：缓存匹配从请求头部开始，系统提示 + 工具描述必须在消息列表的最前面，动态内容（用户消息、对话历史）放后面。顺序颠倒会导致缓存完全失效。
> &emsp;2. **前缀保持稳定**：每次请求的系统提示必须逐字节一致。如果系统提示中包含动态时间戳、随机 ID 等变量，会破坏缓存前缀的一致性。
> &emsp;3. **零代码改动**：DeepSeek 的前缀缓存是服务端自动完成的，开发者不需要调用任何缓存 API。只要遵守前两条规则，缓存自动生效。

&emsp;&emsp;这里有一个学员经常问的问题：<font color=red>如果我不用 DeepSeek，换成其他模型，缓存还能用吗？</font>答案是：Prompt Caching 本质上是**模型提供商的服务端能力**，不是客户端代码能控制的——因为缓存的是 Transformer 推理过程中的 KV Cache（每个 token 位置的 Key-Value 向量），这个计算发生在模型推理引擎内部，客户端根本碰不到。

&emsp;&emsp;不同提供商的实现方式差异很大：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>主流模型提供商 Prompt Caching 实现对比</font></p>
<div class="center">

| 提供商 | 缓存方式 | 客户端需要做什么 | 最小前缀要求 |
|--------|----------|-----------------|-------------|
| DeepSeek | 自动前缀缓存 | 无需任何配置，自动生效 | ~1024 tokens |
| OpenAI | 自动前缀缓存 | 无需任何配置，自动生效 | 1024 tokens |
| Anthropic (Claude) | **显式标记**缓存 | 在消息中添加 `cache_control` 标记 | 1024~4096 tokens（按模型） |
| Google (Gemini) | 显式缓存 API | 调用 `CachedContent.create()` 创建缓存对象 | 由 API 定义 |

</div>

&emsp;&emsp;可以看到，自动缓存（DeepSeek/OpenAI）和显式缓存（Anthropic/Google）是两种不同的设计哲学：前者对开发者完全透明，后者给予更精细的控制权。但无论哪种，核心原则不变——**保持前缀稳定、静态内容放最前面**。

&emsp;&emsp;此外，还需要区分**两个不同层次的缓存**：

> &emsp;**前缀缓存 vs 响应缓存——不要混淆**：
> &emsp;**前缀缓存（服务端）**：同样的系统提示 + 不同的用户问题 → 服务端跳过前缀的 KV 计算，节省推理成本，但仍然调用 API、仍然生成新回复。这是本节讲的 Cache 策略。
> &emsp;**响应缓存（客户端）**：完全相同的输入（系统提示 + 用户问题一字不差）→ 直接返回上次的结果，连 API 都不调用。LangChain 提供了 `InMemoryCache`、`SQLiteCache` 等客户端缓存实现。这适用于重复查询场景（如 FAQ 机器人），但不适用于对话型 Agent（每轮输入都不同）。

&emsp;&emsp;<font color=red>核心洞察：Prompt Caching 是所有五大策略中 ROI 最高的——零代码改动、零架构变更，仅通过保持请求前缀稳定就能节省 85%+ 的静态前缀成本。这就是为什么在 3.3 节的决策优先级中，Cache 排在第一位。</font>

### 3.2 策略-模块交叉矩阵解读

&emsp;&emsp;现在我们把五大策略（操作维度）和六大模块（空间维度）交叉起来，形成一个双坐标分析框架。这张矩阵是本课最核心的工具之一——通过它，我们可以系统性地分析任何 Agent 系统的上下文管理策略。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>五策略 × 六模块交叉矩阵</font></p>
<div class="center">

| 策略 | ①系统提示 | ②对话历史 | ③记忆注入 | ④工具上下文 | ⑤任务状态 | ⑥外部知识 |
|------|----------|----------|----------|-----------|----------|----------|
| **Write** | workspace 文件持久化 | session JSON | mem0 向量库 | — | Scratchpad | 知识库索引 |
| **Select** | 条件注入 / Skills | — | 向量检索 + 阈值过滤 | 动态工具加载 | — | RAG 检索 |
| **Compress** | — | 摘要 / 遮蔽 / trim | — | 工具结果清除 | — | — |
| **Isolate** | — | — | 四类型分组 | MCP 动态发现 | 子 Agent 各自窗口 | — |
| **Cache** | Prompt Caching | — | — | 工具描述缓存 | — | — |

</div>

&emsp;&emsp;这张矩阵揭示了几个关键洞察：

&emsp;&emsp;**热区集中**：对话历史（②）和工具上下文（④）是策略覆盖最密集的两列——它们既可以被压缩、又可以被隔离、还可以被缓存。这再次印证了我们在 2.1 节的判断：这两个模块是上下文工程的首要优化目标。

&emsp;&emsp;**空格有原因**：矩阵中的空格（"—"）不是遗漏，而是"该策略在该模块上没有自然的应用场景"。例如，Compress 不适用于系统提示层（因为系统提示需要完整保留），Cache 不适用于对话历史（因为对话历史每轮都在变化）。

&emsp;&emsp;**Write 是基础设施**：Write 策略几乎横跨所有模块——因为持久化是所有其他策略的基础。没有持久化，压缩后的摘要无处存放，检索也无数据源。

### 3.3 决策优先级：先 Cache 后 Isolate 的工程理由

&emsp;&emsp;五种策略不是同等优先级的。从工程实践角度，有一个清晰的递进顺序：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>策略实施优先级递进路线</font></p>
<div class="center">

| 优先级 | 策略 | 投入 | 收益 | 实施时机 |
|-------|------|------|------|---------|
| 1 | Cache（Prompt Caching） | 极低（结构化提示即可） | 输入成本节省 45-90% | 首日 |
| 2 | Compress（工具结果清除） | 零 | 额外节省 20-40% | 首日 |
| 3 | Compress（观察遮蔽 / trim） | 配置成本 | 累计节省 50%+ | 首周 |
| 4 | Isolate（子 Agent 架构） | 中（架构改造） | 推理质量提升 | 按需 |
| 5 | Write + Select | 中-高 | 支持无限扩展 | 按需 |

</div>

&emsp;&emsp;这个优先级的核心逻辑是**零成本策略先用，复杂策略按需叠加**。其中排在第一位的 Prompt Caching 值得展开讲清机制——它不是简单的"缓存"，而是一套精确的工程优化。

&emsp;&emsp;**Prompt Cache 的工作原理**：当 Agent 多次调用 LLM 时，每次请求的前缀往往大量重复——系统指令（~800 tokens）和工具定义（~2000 tokens）在每轮对话中完全相同。Prompt Caching 的机制是：首次请求时，API 侧缓存这些**静态前缀**作为 cache checkpoint；后续请求如果前缀匹配，直接复用缓存，跳过重新计算。关键参数：

- **最小缓存单元**：1024 tokens（Claude Sonnet 4.5/4、Opus 4.1/4）、2048 tokens（Claude Sonnet 4.6）、4096 tokens（Claude Opus 4.5/4.6、Haiku 4.5）

- **TTL（生存时间）**：默认 5 分钟，命中后重置；部分模型支持 1 小时 TTL

- **可缓存字段**：`system`、`messages`、`tools`——恰好覆盖 Agent 最大的三块固定开销

- **成本折扣**：Anthropic 缓存读取价格比标准输入低 **90%**（即缓存读取价格为标准输入的 10%）

&emsp;&emsp;做个简单的算术：假设 Agent 有 3000 tokens 的固定前缀（系统指令 + 工具定义），每天调用 1000 次。无缓存 = 每天 300 万 input tokens；有缓存 = 首次 3000 + 后续 999×300（缓存价）= 30.3 万等效 tokens。<font color=red>仅这一项优化，月节省可达数百美元，且零代码修改</font>——只需要确保提示结构的静态前缀足够长且稳定。这就是为什么 Cache 排在五策略优先级的第一位。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528834.png" width=50%></div>

&emsp;&emsp;工具结果清除同样只需要一行配置，零成本零延迟。这两个策略应该在 Agent 上线的第一天就启用。

&emsp;&emsp;相比之下，子 Agent 架构需要较大的架构改造投入，RAG 管道需要索引维护——这些策略的收益确实更大，但实施成本也高得多，应该根据实际需求按需引入。

### 3.4 上下文质量诊断：四种失败模式

&emsp;&emsp;前面我们讨论了 Context Rot——上下文变"长"带来的量变退化。但在生产环境中，还有一类更隐蔽的问题：上下文变"脏"带来的质变故障。Drew Breunig 在其博客文章[《How Long Contexts Fail》](https://dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)（2025 年 6 月）中总结的四种上下文失败模式，被 LangChain 引用并在 GitHub 仓库 `langchain-ai/how_to_fix_your_context` 中实现，是工程师排查 Agent 表现异常的核心诊断框架：

> **来源**：Drew Breunig 博客文章，发表于 2025 年 6 月 22 日，[原文链接](https://dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四种上下文质量失败模式</font></p>
<div class="center">

| 失败模式 | 定义 | 典型症状 | 修复方向 |
|---------|------|---------|---------|
| **Context Poisoning（上下文中毒）** | 幻觉或错误信息进入上下文，污染后续推理 | Agent 基于错误前提做出看似合理的推理 | 添加验证步骤、工具结果交叉检查 |
| **Context Distraction（上下文干扰）** | 信息量过大淹没模型训练时学到的知识 | Agent 开始"忘记"基本能力，简单问题也答错 | 压缩、裁剪、控制 token 总量 |
| **Context Confusion（上下文混淆）** | 无关上下文影响模型输出 | 回答偏离主题，受不相关信息影响 | 相关性过滤、作用域限制 |
| **Context Clash（上下文冲突）** | 上下文中的信息互相矛盾 | Agent 给出自相矛盾或反复摇摆的回答 | 矛盾检测、时间感知更新、去重 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528818.png" width=60%></div>

&emsp;&emsp;这四种模式和 Context Rot 的区别在于：Rot 是"量"的问题（token 太多导致注意力衰减），而这四种是"质"的问题（信息本身有毒/有噪/有冲突）。一个常见的排查流程是：Agent 表现下降 → 先检查 token 总量（排除 Rot）→ 再逐一排查四种质量问题。例如，ChatGPT 曾出现过将用户地理位置意外注入图片生成的案例（AEO Agency 2026 年实验：即使在匿名模式下，ChatGPT 也会根据 IP 地址在某些查询中注入地理位置信息）——这就是典型的 Context Confusion：记忆检索返回了与当前任务无关的上下文。

> **踩坑预警**：Context Poisoning 是最危险的失败模式，因为它的症状看起来像"模型在正常推理"——只是前提就是错的。最常见的来源是：（1）前序工具调用返回了幻觉数据；（2）LLM 摘要时引入了不存在的细节；（3）多 Agent 架构中子 Agent 的错误结论被主 Agent 当作事实。防范方法：关键决策前增加验证步骤，不盲信上下文中的"事实"。

&emsp;&emsp;在模块二和三中，我们建立了完整的理论框架：六模块坐标系（空间维度）+ 五策略操作框架（操作维度）+ 交叉矩阵（分析工具）。接下来，我们要把这个框架落地到一个真实的 Agent 系统中——回到我们熟悉的 mini-OpenClaw 项目，做全链路实战标注。

---

## 4. mini-OpenClaw 全链路实战分析

&emsp;&emsp;理论框架再精致，也必须经过实战验证才能内化为工程师的直觉。这一模块我们回到学员们已经熟悉的 mini-OpenClaw 项目，用模块二的六模块坐标系和模块三的五策略框架做全链路标注。我们的目标不是学新代码，而是<font color=red>用新框架重新审视旧代码</font>——看看一个真实的 Agent 系统在上下文工程视角下是什么样子。

### 4.1 mini-OpenClaw 后端架构全景

&emsp;&emsp;先来回顾一下 mini-OpenClaw 后端的核心文件结构和模块职责：

```text
backend/
├── api/
│   └── chat.py (394行)          — SSE 流式端点，请求入口
├── graph/
│   ├── agent.py (374行)         — AgentManager 核心，MAX_HISTORY=20
│   ├── prompt_builder.py (266行) — 6层系统提示拼接
│   ├── session_manager.py (246行)— JSON 持久化 + LLM 摘要压缩
│   ├── mem0_manager.py (297行)  — mem0 四类型分组检索
│   ├── smart_extractor.py (174行)— 节流提取器（每3轮提取一次）
│   └── memory_indexer.py (184行) — LlamaIndex RAG 索引
└── tools/
    └── memory_tools.py (180行)  — 8个分类型记忆工具（4 save + 4 search）
```

&emsp;&emsp;在这个架构中，`AgentManager`（`agent.py`）是所有上下文管理的汇聚点——它负责构建系统提示、加载对话历史、注入记忆检索结果、绑定工具列表，最终将所有信息组装成一次完整的 LLM 调用。可以说，`AgentManager` 就是上下文窗口的"总调度员"。

### 4.2 六模块标注：从源码到框架

&emsp;&emsp;现在我们用六模块坐标系来标注 mini-OpenClaw 的关键代码。授课时我们会直接打开项目源码逐文件讲解，这里先用一张表建立从"框架概念"到"源码位置"的映射关系，方便大家边听边对照：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mini-OpenClaw 六模块源码标注</font></p>
<div class="center">

| 模块 | 核心文件 | 关键函数/类 | 设计要点 |
|------|---------|------------|---------|
| ①系统提示层 | `prompt_builder.py` | `build_system_prompt()` | 6 层固定顺序拼接（Skills→Soul→Identity→User→Agents→记忆），每层 `MAX_COMPONENT_LENGTH=20000` 截断保护；第 6 层按 `memory_backend`（mem0/markdown）+ `rag_mode` 条件注入不同内容 |
| ②对话历史管理层 | `session_manager.py` + `agent.py` | `SessionManager` + `_build_messages()` | JSON 持久化 + `MAX_HISTORY=20` 硬截断；截断时若首条为压缩摘要则保留，确保不完全"失忆" |
| ③记忆检索注入层 | `mem0_manager.py` + `agent.py` | `get_typed_context()` + `_format_mem0_context()` | 按 Claude Code 四类型（user/feedback/project/reference）分组检索，格式化后注入系统提示第 6 层 |
| ④工具上下文管理层 | `memory_tools.py` | `create_memory_tools()` | 工厂函数生成 8 个分类型工具（4 save + 4 search），docstring 即上下文开销（~500-800 token），全量静态加载 |
| ⑤任务状态层 | `session_manager.py` | `compressed_context` 字段 | 用压缩摘要部分承担跨轮状态追踪，但无显式 Scratchpad——Agent 无法主动记录任务进度，<font color=red>这是主要缺口</font> |
| ⑥外部知识层 | `memory_indexer.py` | `MemoryIndexer` | LlamaIndex RAG 索引：MD5 变更检测（避免重复建索引）→ `SentenceSplitter`（chunk_size=256）→ 向量化 → 持久化 |

</div>

&emsp;&emsp;从这张表可以快速定位每个模块的实现位置。授课时我们会重点讲解三处设计决策：系统提示层的"多态注入"（同一个 `build_system_prompt()` 函数根据配置生成完全不同的提示内容）、对话历史层的"摘要优先保留"截断策略、以及记忆注入层的四类型分组格式化。这些设计背后的共同原则是：<font color=red>每一层都在做 Select——只注入当前任务需要的信息子集</font>。

### 4.3 五策略标注：mini-OpenClaw 的策略组合

&emsp;&emsp;接下来我们用五策略框架标注 mini-OpenClaw 中已有的和缺失的策略：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mini-OpenClaw 五策略实现标注</font></p>
<div class="center">

| 策略 | 实现状态 | 实现位置 | 说明 |
|------|---------|---------|------|
| **Write** | 已实现 | `session_manager.py`（JSON 持久化）+ `mem0_manager.py`（向量库写入）+ `memory_tools.py`（Agent 主动保存） | 三路写入：会话历史、长期记忆、Agent 主动记忆 |
| **Select** | 已实现 | `mem0_manager.py`（向量检索）+ `memory_indexer.py`（RAG 检索）+ `prompt_builder.py`（条件注入） | Just-in-time 检索 + 按 memory_backend 条件注入 |
| **Compress** | 部分实现 | `session_manager.py`（LLM 摘要压缩）+ `agent.py`（历史截断 MAX_HISTORY=20） | 有摘要和截断，缺工具结果清除和观察遮蔽 |
| **Isolate** | 未实现 | — | 无子 Agent 架构，所有任务在单一窗口中执行 |
| **Cache** | 部分实现 | `agent.py`（markdown 模式下 Agent 实例缓存） | 应用层缓存，未使用 API 级 Prompt Caching |

</div>

&emsp;&emsp;从这张标注表可以看出，mini-OpenClaw 在 Write 和 Select 两个策略上实现较为完善，Compress 有基础实现但未充分利用（缺少工具结果清除和观察遮蔽），而 Isolate 和 Cache（API 级）完全缺失。

&emsp;&emsp;这些缺失的工程影响是什么？**Isolate 缺失**意味着当 Agent 同时处理多个复杂任务时，不同任务的上下文会互相污染，推理质量下降。**API 级 Cache 缺失**意味着每次请求都要重新处理完整的系统提示 + 工具描述，浪费了大量可复用的输入 token。如果 mini-OpenClaw 要服务更多用户和更复杂的任务，这两个策略是首先需要补齐的。

### 4.4 请求链路全景与体检总结

&emsp;&emsp;当用户发送一条消息到 `/api/chat` 端点时，六模块和五策略协同完成上下文组装。下图展示了完整的请求链路：

&emsp;&emsp;这条链路中有一个关键的设计决策值得注意：mini-OpenClaw 将长期记忆注入到**系统提示层**而非对话历史层。这确保了记忆信息不会被历史截断或压缩时误伤，与 Anthropic 推荐的"固定前缀 + 动态后缀"模式一致。从前课视角看，`mem0.add()` / `mem0.search()` 是供给侧操作，本课分析的注入链路是消费侧操作——<font color=red>供需闭环在这里合拢</font>。

&emsp;&emsp;综合以上分析，我们用一张体检表总结 mini-OpenClaw 的上下文工程健康状况：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>mini-OpenClaw 上下文工程体检表</font></p>
<div class="center">

| 模块 | 健康度 | 已实现 | 待改进 |
|------|-------|-------|-------|
| ①系统提示层 | 良好 | 6层拼接+条件注入+截断保护 | 可启用 API 级 Prompt Caching |
| ②对话历史管理层 | 良好 | JSON 持久化+LLM 摘要+截断 | 可增加观察遮蔽（工具结果占比高时） |
| ③记忆检索注入层 | 良好 | mem0 四类型分组+阈值过滤 | 可优化检索相关性（调整 score_threshold） |
| ④工具上下文管理层 | 一般 | 全量静态加载（~10 工具） | 工具数量增长时需转向动态加载 |
| ⑤任务状态层 | 缺失 | 无显式 Scratchpad | 复杂任务场景需增加任务状态持久化 |
| ⑥外部知识层 | 良好 | LlamaIndex RAG 索引 | 可优化分块策略和混合检索 |

</div>

&emsp;&emsp;通过这次全链路标注，我们把模块二和模块三建立的理论框架完整地落地到了一个真实的 Agent 系统中。这不仅验证了框架的实用性，也让我们看到了一个"足够好但还有优化空间"的真实系统——这正是大多数生产系统的真实状态。

---

## 5. 场景决策与成本意识

&emsp;&emsp;有了六模块坐标系、五策略操作框架和实战标注经验，最后一个关键问题是：<font color=red>面对具体的业务场景，应该选择什么策略组合？</font>这不是一个有标准答案的问题——不同场景的最优策略组合差异极大。本模块帮助我们建立场景选型的直觉和成本优化的工程师视角。

### 5.1 场景-策略匹配决策矩阵

&emsp;&emsp;根据调研材料中的行业实践，我们整理出七类典型 Agent 场景的推荐策略组合：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>七类场景策略匹配决策矩阵</font></p>
<div class="center">

| 场景 | 对话特征 | 推荐策略组合 | 推荐框架 |
|------|---------|------------|---------|
| 单轮 QA / 分类 | 无历史 | 系统提示优化 + Cache | 任意 |
| 多轮短对话（<20 轮） | 短交互 | trim_messages + 用户画像 Write + Cache | OpenAI SDK / LangGraph |
| 长程对话（100+ 轮） | 持续增长 | Compress（摘要+遮蔽）+ Select（RAG）+ Write + Cache | LangGraph + Mem0 |
| 编程 Agent（长任务） | 工具密集 | Compress（工具结果清除+auto-compact）+ Write（文件系统）+ Isolate + Cache | Claude Code / Deep Agents |
| 调研 Agent | 多信源聚合 | Isolate（多子 Agent）+ Select（多源 RAG）+ Write（结构化笔记）| Claude Code / LangGraph |
| 多 Agent 协作 | 上下文共享/隔离 | Isolate（原生隔离）+ Write（共享记忆块）+ Select（选择性广播） | LangGraph / Letta |
| 企业级（合规审计） | 审计需求 | Write（完整日志持久化）+ Compress（摘要工作副本）+ Isolate（租户隔离） | LangGraph(PG) + SK |

</div>

&emsp;&emsp;一个快速选型的思路是从对话长度出发：短对话（<20 轮）只需基础的 trim + Cache；中长对话（20-100 轮）需要加入 Compress；超长对话（100+ 轮）需要全套策略组合。对话越长，所需的策略越多，工程复杂度也越高。

> **踩坑预警**：ROI 陷阱——小规模场景（<1 万对话/月）引入复杂上下文管理（如 RAG 管道、子 Agent 架构）的工程成本可能超过节省的 API 成本。正确做法：先启用零成本策略（Cache + 工具结果清除），观察实际 token 消耗数据后再决定是否引入更复杂的策略。排查方向：算一笔帐——引入 RAG 管道的开发成本 + 向量 DB 运维成本（$50-500/月）是否大于节省的 API 成本。

### 5.2 成本优化路线：递进式六阶段

&emsp;&emsp;基于上述分析，我们给出一个递进式的成本优化路线图：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>递进式成本优化六阶段路线</font></p>
<div class="center">

| 阶段 | 策略 | 投入 | Token 节省 | 额外延迟 | 实施时机 |
|------|------|------|----------|---------|---------|
| Stage 0 | Prompt Caching | 结构化提示设计 | 输入成本 45-90% | TTFT 改善 13-31% | 首日 |
| Stage 1 | 工具结果清除 | 零 | 额外 20-40% | 零 | 首日 |
| Stage 2 | 观察遮蔽 / trim_messages | 配置成本 | 累计 50%+ | 零 | 首周 |
| Stage 3 | LLM 摘要 | 摘要模型成本 | 累计 60%+ | 1-5 秒 | 按需 |
| Stage 4 | RAG 检索 | 向量 DB（$50-500/月） | 累计 90%+ | 50-200ms | 按需 |
| Stage 5 | 模型路由 | 路由逻辑 | 累计 95%+ | 变化 | 进阶 |

</div>

&emsp;&emsp;Stage 5 的模型路由是一个高级策略：将简单任务路由到低成本模型（如 DeepSeek V3.2，$0.28/M），复杂任务路由到旗舰模型（如 Claude Opus 4.6，$5.00/M）。这不属于传统的上下文管理范畴，但在实际的成本优化中效果显著。

&emsp;&emsp;**立即行动清单**：

1. 检查你的 Agent 是否已启用 Prompt Caching（结构化提示设计，目标 80%+ 命中率）

2. 实施工具结果清除 + 观察遮蔽 / trim_messages（零成本策略）

3. 建立 Token 消耗监控：P99 token 消耗、每对话成本、缓存命中率

&emsp;&emsp;这三件事投入极低，但能让你在正式投入复杂策略之前，先获得 50-70% 的成本节省，同时积累真实的 token 消耗数据——这些数据是后续决策的基础。

---

&emsp;&emsp;到这里，我们已经建立了从"理论框架"到"工程决策"的完整链路：有了六模块坐标系和五策略工具箱（模块二、三），能在真实代码中识别策略组合（模块四），也能根据场景和成本做出选型判断（模块五）。接下来，让我们把这些知识凝练为一张可带走的知识图谱，并为下节课的 LangGraph 实战做好铺垫。

## 6. 总结

&emsp;&emsp;我们用三个小时完成了一次从"会存取记忆"到"懂窗口策展"的认知升维。最后，让我们用一张完整的知识图谱回顾本课的核心框架，并为下节课做好铺垫。

### 6.1 六模块 + 五策略知识图谱总结

&emsp;&emsp;本课建立的认知框架可以用两个维度概括：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260409140528858.png" width=60%></div>

&emsp;&emsp;**空间维度（六模块）**：上下文窗口由六类信息组成——系统提示（操作系统）、对话历史（短期记忆）、记忆检索（供给侧接口）、工具上下文（能力声明）、任务状态（工作笔记本）、外部知识（图书馆）。这六个模块的 token 消耗分布和增长速度各不相同，管理优先级也不同。

&emsp;&emsp;**操作维度（五策略）**：工程师通过 Write（持久化）、Select（按需检索）、Compress（压缩）、Isolate（隔离）、Cache（缓存）五种策略来管理这些信息。五种策略不是互斥选项，而是组合使用的层次，生产级系统通常同时使用 3-5 种。

&emsp;&emsp;**贯穿主线（供需模型）**：记忆管理是供给侧——负责生产和存储信息；上下文工程是消费侧——负责在有限窗口中策展最有价值的信息组合。前课的 `mem0` 是供给侧的核心工具，本课的六模块 + 五策略是消费侧的完整框架。供需闭环在记忆检索注入层合拢。

&emsp;&emsp;**能力自检清单**：

1. 我能解释 Context Rot 现象，并说出有效窗口约为宣传窗口的 50-65%

2. 我能列出上下文窗口的六大功能模块，并说明每个模块的核心管理问题

3. 我能运用 Anthropic 五策略框架分析一个 Agent 系统的上下文管理策略

4. 我能对 mini-OpenClaw 做六模块 + 五策略的全链路标注

5. 我能根据场景选择合适的策略组合，并估算成本优化效果